# UI measure

In [ ]:
%gui qt
from dual_gate_qt_v15 import launch_in_notebook
mw = launch_in_notebook()


In [ ]:
from daq_xy_qt_readback import launch, run

# Notebook-friendly (non-blocking)
app, win = launch("Dev1", "ao0", "ao1")



# Python measure

In [ ]:
import time, os
# os.environ["QT_API"] = "pyqt6"   # tell IPython/matplotlib to use PySide6
# os.environ["MPLBACKEND"] = "QtAgg" # Qt backend (will bind to PySide6)
# %gui qt6
%matplotlib qt
import pyvisa, iv_automation, lf6_automation, spectral_experiments
import numpy as np
import matplotlib.pyplot as plt
import winsound

rm = pyvisa.ResourceManager()
print(rm.list_resources())
plt.rcParams['figure.raise_window'] = False
# ('ASRL1::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL5::INSTR', 'ASRL7::INSTR', 'GPIB0::23::INSTR', 'GPIB0::24::INSTR
# 'ASRL4::INSTR' MPS

In [4]:
import pyvisa

In [ ]:
# Save data to user folder
DEV_NAME = 'YZD303'
spectral_experiments.USER_FOLDER = rf"C:\Users\commo\Desktop\Instrument control v3\{DEV_NAME}\initial data"


In [ ]:
# change address
keithley1 = iv_automation.KeithControl('GPIB1::3::INSTR', 'GPIB26', 'Vbg', rm)
keithley1.set_volt_step(curr_compliance=1E-7, delay=0.1, volt_compliance=40)
# change address
keithley2 = iv_automation.KeithControl('GPIB1::23::INSTR', 'GPIB23', 'Vtg', rm)
keithley2.set_volt_step(curr_compliance=1E-7, delay=0.1, volt_compliance=40)

keithley3 = iv_automation.KeithControl('GPIB1::7::INSTR', 'GPIB7', 'Vcg', rm)
keithley3.set_volt_step(curr_compliance=1E-7, delay=0.1, volt_compliance=10)

instrument_list = [keithley1, keithley2, keithley3]
iv = iv_automation.IVSetup(instrument_list)

#x_goto(self, x_name, target, delta, delay)
iv.x_goto('Vbg', 0, 0.1, 0.1)
iv.x_goto('Vtg', 0, 0.1, 0.1)
iv.x_goto('Vcg', 0, 0.1, 0.1)
iv.report_status()


In [ ]:
iv.x_goto('Vbg',target= 0, delta=0.1,delay= 0.1, print_steps=True)

In [ ]:
lf6 = lf6_automation.LF6Setup()
lf6.print_saved_experiments()

In [ ]:
# load Newport ratation mount
from backend.instruments import EP300
ep300 = EP300(address='ASRL3::INSTR', axes=[3])
ep300.connect()
# ep300.set_position(3,0) 
ep300.set_speed(2, 20)
ep300.set_speed(1, 3)
# axis 2 : 4deg/s 
# axis 1 : 40deg/s
# ep300.set_position(2,11)

In [ ]:
ep300.connect()
print(ep300.get_positions(2))
# ep300.set_position(3, 10)
# print(ep300.get_positions(3))
# (98.7 parallel, 98.7+45)

In [ ]:
# load power meter
from datetime import datetime
from ctypes import cdll,c_long, c_ulong, c_uint32,byref,create_string_buffer,c_bool,c_char_p,c_int,c_int16,c_double, sizeof, c_voidp
from TLPMX import TLPMX
import time

from TLPMX import TLPM_DEFAULT_CHANNEL

# Find connected power meter devices.
tlPM = TLPMX()
deviceCount = c_uint32()
tlPM.findRsrc(byref(deviceCount))

print("Number of found devices: " + str(deviceCount.value))
print("")

resourceName = create_string_buffer(1024)

for i in range(0, deviceCount.value):
    tlPM.getRsrcName(c_int(i), resourceName)
    print("Resource name of device", i, ":", c_char_p(resourceName.raw).value)
print("")
tlPM.close()

# Connect to last device.
tlPM = TLPMX()
tlPM.open(resourceName, c_bool(True), c_bool(True))

message = create_string_buffer(1024)
tlPM.getCalibrationMsg(message,TLPM_DEFAULT_CHANNEL)
print("Connected to device", i)
print("Last calibration date: ",c_char_p(message.raw).value)
print("")

time.sleep(2)
wavelength = c_double(730)
tlPM.setWavelength(wavelength,TLPM_DEFAULT_CHANNEL)
# Enable auto-range mode.
# 0 -> auto-range disabled
# 1 -> auto-range enabled
tlPM.setPowerAutoRange(c_int16(1),TLPM_DEFAULT_CHANNEL)

# Set power unit to Watt.
# 0 -> Watt
# 1 -> dBm
tlPM.setPowerUnit(c_int16(0),TLPM_DEFAULT_CHANNEL)

In [ ]:
# load linear stage
import pylablib as pll
from pylablib.devices import Thorlabs
pump_stage = Thorlabs.ElliptecMotor("COM5")

In [ ]:
# load scanner
import instrControl as ic, pyvisa, numpy as np, matplotlib, time
%matplotlib qt

daq = ic.DaqControl('Dev1')
daq.add_ao_channel('ao0', 'x_axis')
daq.add_ao_channel('ao1', 'y_axis')
daq.check_status()

instrument_list = [daq]

sample_name = 'dev67_4K'
ex = ic.Experiment(instrument_list, sample_name)


In [ ]:
# load monochromator
mono = ic.MonoControl(address='ASRL7::INSTR', name='SP2300', rm=rm, initial_wl=0)

In [ ]:
daq.add_ai_channel('ai0','ai0')
daq.add_ai_channel('ai1','ai1')
daq.add_ai_channel('ai2','ai2')


In [ ]:
ex.x_goto('x_axis', 5, 0.1, 0.1)
ex.x_goto('y_axis', 5, 0.1, 0.1)
# ex.check_status()

In [ ]:
ex.x_goto('x_axis', 0, 0.1, 0.1)
ex.x_goto('y_axis', 0, 0.1, 0.1)
ex.get_status()

## Control Magnet in Driven Mode

In [5]:
MSP = iv_automation.MagnetPowerSupplyControl('ASRL5::INSTR','MSP','\r\n',pyvisa.ResourceManager())

*IDN?
Attocube,APS100,2301029,1.67,323
REMOTE
UNITS G
x
UNITS?
kG
IMAG?
0.000kG
RATE? 0
0.0343
RATE? 1
0.0171
RATE? 2
0.0100
ULIM 0.0000
LLIM 0.0000


In [ ]:
MSP.set_high_sweeplimit(90)

In [ ]:
MSP.set_low_sweeplimit(-90)

In [ ]:
# MSP.get_sweep_mode()
MSP.turnon_heater()

In [ ]:
MSP.set_high_sweeplimit(0)
MSP.start_sweep('UP')
# in case not stable
# time.sleep(60)

In [ ]:
MSP.unsync_start_sweep('UP')

In [ ]:
A = MSP.get_magnetfield()
print(A)

In [ ]:
MSP.turnoff_heater()

In [ ]:
MSP.get_heaterstatus()
MSP.get_sweep_mode()
MSP.turnoff_heater()
MSP.turnon_heater()
MSP.set_high_sweeplimit(0)
MSP.start_sweep('UP')
MSP.start_sweep('PAUSE')

In [ ]:
MSP.start_sweep('ZERO')


In [ ]:
MSP.turnoff_heater()

In [ ]:
#go to high set field
# MSP.set_high_sweeplimit(60)
MSP.start_sweep('ZERO')

In [ ]:
MSP.set_high_sweeplimit(80)
MSP.start_sweep('UP')

In [ ]:
def RMCD_meas(mode, mcd_sens = 500, laser_sens = 1000,
              file_name = 'Atest_MCD_1', high_target_field = 0, low_target_field = 0,eplison = 0.005):
    MSP.set_low_sweeplimit(low_target_field)
    MSP.set_high_sweeplimit(high_target_field)
    MSP.unsync_start_sweep(mode)
    if mode == 'UP':
        target_field = MSP.get_high_sweeplimit()
    elif mode == 'DOWN':
        target_field = MSP.get_low_sweeplimit()
    target_field = np.array(target_field.split('kG',1)[0]).astype(float)

    file_name = f'{file_name}_mcdsens={mcd_sens}mV_lasersens={laser_sens}mV'
    def check_file_name(file_name: str):
        file_number = 0
        while True:
            file_number += 1
            new_filename = f'{file_name}_{file_number:03d}'
            excel_name = new_filename + '.csv'
            if not os.path.exists(excel_name):
                return new_filename

    new_file_name = check_file_name(file_name)
    output_file_name = new_file_name + '.csv'

    with open(output_file_name, 'a') as f1:
        cols = np.array(['Field','MCD_R','MCD_X','MCD_Y','Laser_R','Theta','Norm_MCD_R','Norm_MCD_X'], ndmin=1, dtype='U').reshape([1, -1])
        np.savetxt(f1, cols, fmt='%s', delimiter=',')
        while True:
            time.sleep(0.5)
            epsilon = eplison
            daq.read_y()
            mcd_X = mcd_sens*daq.update_y(2)/10000
            mcd_Y = mcd_sens*daq.update_y(3)/10000
            laser_R = laser_sens*daq.update_y(4)/10000
            mcd_R = np.sqrt(mcd_X**2+mcd_Y**2)
            field = MSP.get_magnetfield()
            data = np.array([field/10, mcd_R, mcd_X, mcd_Y, laser_R, np.arctan(mcd_Y/mcd_X),mcd_R/laser_R, mcd_X/laser_R]).reshape([1, -1])
            np.savetxt(f1, data, fmt='%s', delimiter=',')
            if abs(target_field - field) <= epsilon:
                time.sleep(10)
                break
        MSP.write('SWEEP PAUSE',print_command=True)
        print('Pause at target field:',MSP.get_magnetfield(),'kG')

In [ ]:
def RMCD_wl_dep (start_wl, end_wl,step, file_name,mcd_sens = 500, laser_sens = 1000):
    file_name = f'{file_name}_mcdsens={mcd_sens}mV_lasersens={laser_sens}mV'
    def check_file_name(file_name: str):
        file_number = 0
        while True:
            file_number += 1
            new_filename = f'{file_name}_{file_number:03d}'
            excel_name = new_filename + '.csv'
            if not os.path.exists(excel_name):
                return new_filename

    new_file_name = check_file_name(file_name)
    output_file_name = new_file_name + '.csv'
    with open(output_file_name, 'a') as f1:
        num_steps = int(round((end_wl - start_wl) / step)) + 1
        wls = np.linspace(start_wl, end_wl, num_steps)
        cols = np.array(['wls','MCD_R','MCD_X','MCD_Y','Laser_R','Norm_MCD_R','Norm_MCD_X'], ndmin=1, dtype='U').reshape([1, -1])
        np.savetxt(f1, cols, fmt='%s', delimiter=',')
        for wl in wls:
            mono.wl_goto(wl)
            time.sleep(1)
            daq.read_y()
            mcd_X = mcd_sens*(daq.update_y(2))/10000
            mcd_Y = mcd_sens*(daq.update_y(3))/10000
            laser_R = laser_sens*(daq.update_y(4))/10000
            mcd_R = np.sqrt(mcd_X**2+mcd_Y**2)
            data = np.array([wl, mcd_R, mcd_X, mcd_Y, laser_R, mcd_R/laser_R, mcd_X/laser_R]).reshape([1, -1])
            np.savetxt(f1, data, fmt='%s', delimiter=',')

def RMCD_wl_dep_manual ( start_wl, end_wl,step,file_name,mcd_sens = 500, laser_sens = 1000):
    with open('{}.csv'.format(file_name), 'a') as f1:
        num_steps = int(round((end_wl - start_wl) / step)) + 1
        wls = np.linspace(start_wl, end_wl, num_steps)
        cols = np.array(['wls','Norm_MCD_R','Norm_MCD_X','MCD_X','MCD_Y','Laser_R'], ndmin=1, dtype='U').reshape([1, -1])
        np.savetxt(f1, cols, fmt='%s', delimiter=',')
        for wl in wls:
            mono.wl_goto(wl)
            Power_key = input(f'current WL: {wl}, change PEM, and input')
            time.sleep(1)
            daq.read_y()
            mcd_X = daq.update_y(2)
            mcd_Y = daq.update_y(3)
            laser_R = daq.update_y(4)
            data = np.array([wl,mcd_sens*np.sqrt(mcd_X**2+mcd_Y**2)/(laser_sens*laser_R),mcd_sens*mcd_X/(laser_sens*laser_R),mcd_X,mcd_Y,laser_R]).reshape([1, -1])
            np.savetxt(f1, data, fmt='%s', delimiter=',')



In [ ]:
def b_field_spectra(mode, file_name = 'Atest_bfield_spectra_1', high_target_field = 0, low_target_field = 0,eplison = 0.005):
    MSP.set_low_sweeplimit(low_target_field)
    MSP.set_high_sweeplimit(high_target_field)
    MSP.unsync_start_sweep(mode)
    if mode == 'UP':
        target_field = MSP.get_high_sweeplimit()
    elif mode == 'DOWN':
        target_field = MSP.get_low_sweeplimit()
    target_field = np.array(target_field.split('kG',1)[0]).astype(float)
    def check_file_name(file_name: str):
        file_number = 0
        while True:
            file_number += 1
            new_filename = f'{file_name}_{file_number:03d}'
            excel_name = new_filename + '.csv'
            if not os.path.exists(excel_name):
                return new_filename

    new_file_name = check_file_name(file_name)
    output_file_name = new_file_name + '.csv'
    
    with open(output_file_name, 'a') as f1:
        wls = lf6.get_wavelength_calibration()
        cols = np.concatenate((np.array(['Field'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
        np.savetxt(f1, cols, fmt='%s', delimiter=',')
        while True:
            time.sleep(0.5)
            epsilon = eplison
            field_before = MSP.get_magnetfield()
            spectra = lf6.acquire()
            field_after = MSP.get_magnetfield()
            field = (field_before+field_after)/2
            data = np.concatenate((np.array([field], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
            np.savetxt(f1, data, fmt='%.5e', delimiter=',') 
            if abs(target_field - field) <= epsilon:
                time.sleep(3)
                break
        MSP.write('SWEEP PAUSE',print_command=True)
        print('Pause at target field:',MSP.get_magnetfield(),'kG')

In [ ]:
bg = -4.332
tg = -1.05
print(f'{-bg-bg}V')
iv.x_goto('Vbg', bg, 0.1, 0.1)
iv.x_goto('Vtg', tg, 0.1, 0.1)
iv.report_status()

In [ ]:
MSP.get_magnetfield()

In [ ]:
stage.move_to(2435)
# stage.move_to(610)
meas_power =  c_double()
tlPM.setWavelength(730)
tlPM.measPower(byref(meas_power))
print(meas_power.value*1e6-0.005)#unit uW

In [ ]:
ep300.set_position(1,147.8+45)
# in halfwaveplate 109.3 64.3
ep300.set_position(2,1.7)
# out halfwaveplate 32.7 laser max KK  77.7 laser min KKp

In [ ]:
b_field_spectra(mode ='UP',file_name=f'$DXD127$~$REF600g720nmCWL$~$pEn2$~$1s$~Bfield0to6_spectra_Vtg={tg} Vbg={bg}',
                high_target_field=60, low_target_field= 0)

In [ ]:
bg = 0
tg = 0
# print(f'{-bg-bg}V')
iv.x_goto('Vbg', bg, 0.1, 0.1)
iv.x_goto('Vtg', tg, 0.1, 0.1)
iv.report_status()

In [ ]:
mono.wl_goto(633)

In [ ]:
field = MSP.get_magnetfield()
start_wl = 700
end_wl = 780
# RMCD_wl_dep(start_wl=start_wl,end_wl=end_wl,step=1,
#             file_name=f'Dev127_RMCD_pE_{np.around(field/10,3)}T_tg={tg}_bg={bg}_wls{start_wl}-{end_wl}',
#             mcd_sens=100,laser_sens=500)
RMCD_wl_dep(start_wl=start_wl,end_wl=end_wl,step=1,
            file_name=f'Dev127_RMCD_PE_{np.around(field/10,3)}T_tg={tg}_bg={bg}_wls{start_wl}-{end_wl}',
            mcd_sens=100,laser_sens=500)

In [ ]:
bg = 2.6
tg = 0
# print(f'{-bg-bg}V')
iv.x_goto('Vbg', bg, 0.1, 0.1)
iv.x_goto('Vtg', tg, 0.1, 0.1)
iv.report_status()

In [ ]:
field = MSP.get_magnetfield()
start_wl = 700
end_wl = 780
# RMCD_wl_dep(start_wl=start_wl,end_wl=end_wl,step=1,
#             file_name=f'Dev127_RMCD_pE_{np.around(field/10,3)}T_tg={tg}_bg={bg}_wls{start_wl}-{end_wl}',
#             mcd_sens=100,laser_sens=500)
RMCD_wl_dep(start_wl=start_wl,end_wl=end_wl,step=1,
            file_name=f'Dev127_Gatemodulation_PC_{np.around(field/10,3)}T_tg={tg}_bg={bg}+80mV_wls{start_wl}-{end_wl}',
            mcd_sens=20,laser_sens=50)

In [ ]:
field = MSP.get_magnetfield()
start_wl = 720
end_wl = 760
RMCD_wl_dep_manual(start_wl=start_wl,end_wl=end_wl,step=1,
            file_name=f'Dev127_pE_{field/10}T_RMCD_Sample_tg={tg}_bg={bg}_wls{start_wl}-{end_wl}')

In [ ]:
MSP.set_high_sweeplimit(60)

In [ ]:
field = MSP.get_magnetfield()

In [ ]:
bg = -2.81
tg = -1.6
# print(f'{-bg-bg}V')
iv.x_goto('Vbg', bg, 0.1, 0.1)
iv.x_goto('Vtg', tg, 0.1, 0.1)
iv.report_status()

In [ ]:
field = MSP.get_magnetfield()
start_wl = 700
end_wl = 780
RMCD_wl_dep(start_wl=start_wl,end_wl=end_wl,step=1,
            file_name=f'Dev127_RMCD_pZ_{np.around(field/10,3)}T_tg={tg}_bg={bg}_wls{start_wl}-{end_wl}',
            mcd_sens=100,laser_sens=500)

In [ ]:
# field = MSP.get_magnetfield()
# start_wl = 700
# end_wl = 780
# # RMCD_wl_dep(start_wl=start_wl,end_wl=end_wl,step=1,
# #             file_name=f'Dev127_RMCD_pE_{np.around(field/10,3)}T_tg={tg}_bg={bg}_wls{start_wl}-{end_wl}',
# #             mcd_sens=100,laser_sens=500)
# RMCD_wl_dep(start_wl=start_wl,end_wl=end_wl,step=1,
#             file_name=f'Dev127_RMCD_PZ_BG_{np.around(field/10,3)}T_tg={tg}_bg={bg}_wls{start_wl}-{end_wl}',
#             mcd_sens=100,laser_sens=500)

In [ ]:
mono.wl_goto(633)

In [ ]:
wavelength = 740
high_target_field = 80
low_target_field = -80
mono.wl_goto(wavelength)
field = MSP.get_magnetfield()
# 'DOWN' 'UP'
if np.isclose(field, high_target_field, atol=1):
    mode = 'DOWN'
elif np.isclose(field, low_target_field, atol=1):
    mode = 'UP'
# mode = 'UP'
RMCD_meas(mode=mode,mcd_sens = 100, laser_sens = 500,
          file_name=f'DXD127_PZ_BG_{wavelength}nm_tg={tg}_bg={bg}_{mode}_{low_target_field/10}Tto{high_target_field/10}T',
          high_target_field=high_target_field,low_target_field=low_target_field,eplison = 0.005)

winsound.Beep(frequency=750, duration=300)

In [ ]:
MSP.write('SWEEP PAUSE',print_command=True)

In [ ]:
MSP.turnoff_heater()


In [ ]:
MSP.turnon_heater()


In [ ]:
def RMCD_megasweep(f_name, wls, vbg_start, vbg_end, vbg_steps, vtg_start, vtg_end, vtg_steps, mcd_sens, laser_sens, sampling):
    def check_file_name(file_name: str):
        file_number = 0
        while True:
            file_number += 1
            new_filename = f'{file_name}_{file_number:03d}'
            excel_name = new_filename + '.csv'
            if not os.path.exists(excel_name):
                return new_filename

    new_file_name = check_file_name(f_name)
    output_file_name = new_file_name + '.csv'

    mono.wl_goto(wls)

    vbgs = np.linspace(vbg_start, vbg_end, vbg_steps)
    vtgs = np.linspace(vtg_start, vtg_end, vtg_steps)

    flag = True
    step = 0.05  # V
    delay = 0.1  # s

    try:
        with open(output_file_name, 'a') as f:
            cols = np.array(['vbg', 'vtg', 'MCD_R', 'MCD_X', 'MCD_Y', 'Laser_R', 'Norm_MCD_R', 'Norm_MCD_X', 'Theta'], ndmin=1, dtype='U').reshape([1, -1])
            np.savetxt(f, cols, fmt='%s', delimiter=',')
            for vbg in vbgs:
                iv.x_goto('Vbg', vbg, step, delay)
                if flag:
                    vtgseq = vtgs
                else:
                    vtgseq = np.flip(vtgs)
                for vtg in vtgseq:
                    iv.x_goto('Vtg', vtg, step, delay)
                    time.sleep(0.5)
                    raw_mcd_X = 0.0
                    raw_mcd_Y = 0.0
                    raw_laser_R = 0.0
                    for j_sample in range(sampling):
                        daq.read_y()
                        raw_mcd_X += daq.update_y(2)
                        raw_mcd_Y += daq.update_y(3)
                        raw_laser_R += daq.update_y(4)
                    mcd_X = mcd_sens * raw_mcd_X / (sampling * 10000)
                    mcd_Y = mcd_sens * raw_mcd_Y / (sampling * 10000)
                    laser_R = laser_sens * raw_laser_R / (sampling * 10000)
                    mcd_R = np.sqrt(mcd_X**2 + mcd_Y**2)
                    data = np.array([vbg, vtg, mcd_R, mcd_X, mcd_Y, laser_R, mcd_R / laser_R, mcd_X / laser_R, np.arctan(mcd_Y / mcd_X)]).reshape([1, -1])
                    np.savetxt(f, data, fmt='%.5e', delimiter=',')
                flag = not flag
    except Exception as e:
        print(f"Error occurred: {e}")
    finally:
        iv.x_goto('Vbg', 0, step, delay)
        iv.x_goto('Vtg', 0, step, delay)
        print('vg:', 0)
        winsound.Beep(frequency=750, duration=300)  # frequency in Hz, duration in milliseconds


In [ ]:
    # f_name =f'DXD127_pE_{wavelength}nm_Megasweep_{np.around(field/10,3)}T_mcdsens={mcd_sens}mV_lasersens={laser_sens}mV'
    # RMCD_megasweep(f_name = f_name, wls = wavelength, vbg_start = -3, vbg_end = 5, vbg_steps = 81, 
    #                                         vtg_start = -3.5, vtg_end= -2, vtg_steps = 16, 
    #                                             mcd_sens = mcd_sens, laser_sens = laser_sens, sampling = 2)

In [ ]:
field = MSP.get_magnetfield()
mcd_sens = 100
laser_sens = 500
wavelengths= [751,729]
# wavelengths= [739]
for wavelength in wavelengths:
    f_name =f'DXD127_pEn3_{wavelength}nm_Megasweep_{np.around(field/10,3)}T_mcdsens={mcd_sens}mV_lasersens={laser_sens}mV'
    RMCD_megasweep(f_name = f_name, wls = wavelength, vbg_start = -8, vbg_end = -2, vbg_steps = 61, 
                                            vtg_start = -2, vtg_end= 0.5, vtg_steps = 26, 
                                                mcd_sens = mcd_sens, laser_sens = laser_sens, sampling = 2)
    f_name =f'DXD127_pEn3_{wavelength}nm_Megasweep_{np.around(field/10,3)}T_mcdsens={mcd_sens}mV_lasersens={laser_sens}mV'
    RMCD_megasweep(f_name = f_name, wls = wavelength, vbg_start = -6, vbg_end = 6, vbg_steps = 121, 
                                            vtg_start = -2.4, vtg_end= 2.4, vtg_steps = 26, 
                                                mcd_sens = mcd_sens, laser_sens = laser_sens, sampling = 2)


## Modulate with AC

In [ ]:
def AC_modulate_wl_dep (start_wl, end_wl,step, file_name,mcd_sens = 500, laser_sens = 1000,delay = 1, sampling = 1):
    file_name = f'{file_name}_mcdsens={mcd_sens}mV_lasersens={laser_sens}mV'
    def check_file_name(file_name: str):
        file_number = 0
        while True:
            file_number += 1
            new_filename = f'{file_name}_{file_number:03d}'
            excel_name = new_filename + '.csv'
            if not os.path.exists(excel_name):
                return new_filename

    new_file_name = check_file_name(file_name)
    output_file_name = new_file_name + '.csv'
    with open(output_file_name, 'a') as f1:
        num_steps = int(round((end_wl - start_wl) / step)) + 1
        wls = np.linspace(start_wl, end_wl, num_steps)
        cols = np.array(['wls','MCD_R','MCD_X','MCD_Y','Laser_R','Norm_MCD_R','Norm_MCD_X'], ndmin=1, dtype='U').reshape([1, -1])
        np.savetxt(f1, cols, fmt='%s', delimiter=',')
        for wl in wls:
            if wl == start_wl:
                time.sleep(5)
            mono.wl_goto(wl)
            time.sleep(delay)
            raw_mcd_X = 0.0
            raw_mcd_Y = 0.0
            raw_laser_R = 0.0
            for j_sample in range(sampling):
                daq.read_y()
                raw_mcd_X += daq.update_y(2)
                raw_mcd_Y += daq.update_y(3)
                raw_laser_R += daq.update_y(4)
            mcd_X = mcd_sens * raw_mcd_X / (sampling * 10000)
            mcd_Y = mcd_sens * raw_mcd_Y / (sampling * 10000)
            laser_R = laser_sens * raw_laser_R / (sampling * 10000)
            mcd_R = np.sqrt(mcd_X**2 + mcd_Y**2)

            data = np.array([wl, mcd_R, mcd_X, mcd_Y, laser_R, mcd_R/laser_R, mcd_X/laser_R]).reshape([1, -1])
            np.savetxt(f1, data, fmt='%s', delimiter=',')

In [ ]:
mono.wl_goto(670.2)

In [ ]:
bg = 0
tg = 0
# print(f'{-bg-bg}V')
iv.x_goto('Vbg', bg, 0.1, 0.1)
iv.x_goto('Vtg', tg, 0.1, 0.1)
iv.report_status()

In [ ]:
mono.wl_goto(633)
meas_power =  c_double()
tlPM.setWavelength(635)
tlPM.measPower(byref(meas_power))
probe_power = np.around(meas_power.value*1e6,2)
print(meas_power.value*1e6)#unit uW
# print(probe_power)

In [ ]:
# field = MSP.get_magnetfield()
field = 90
start_wl = 633
end_wl = 740
power = 0
# RMCD_wl_dep(start_wl=start_wl,end_wl=end_wl,step=1,
#             file_name=f'Dev127_RMCD_pE_{np.around(field/10,3)}T_tg={tg}_bg={bg}_wls{start_wl}-{end_wl}',
#             mcd_sens=100,laser_sens=500)
AC_modulate_wl_dep(start_wl=start_wl,end_wl=end_wl,step=0.25,
            file_name=f'Dev197_PE_{np.around(field/10,3)}T_probe2.21uW@635nm_pump670.5nm{power}uw_tg={tg}_bg={bg}_wls{start_wl}-{end_wl}_Chop661Hz_SiAPD',
            mcd_sens=500,laser_sens=1000, delay = 0.5, sampling = 3)

winsound.Beep(frequency=750, duration=300) 

In [ ]:
from openpyxl import Workbook
from typing import Dict, List

def gate_sweep_AC_modulate_savecol(file_name: str, Vbg_start: float, Vbg_end: float, Vtg_start: float, Vtg_end: float, 
                           frames: int, start_wl: float, end_wl: float, wl_step: float, delay: float, 
                           sampling: int, mcd_sens: float, laser_sens: float) -> None:
    
    file_name = f'{file_name}_mcdsens={mcd_sens}mV_lasersens={laser_sens}mV'

    def check_file_name(file_name: str) -> str:
        """Ensure the file name is unique by appending a number if necessary."""
        file_number = 0
        while True:
            file_number += 1
            new_filename = f'{file_name}_{file_number:03d}'
            excel_name = new_filename + '.xlsx'
            if not os.path.exists(excel_name):
                return new_filename

    new_file_name = check_file_name(file_name)
    output_file_name = new_file_name + '.xlsx'

    # Initialize Excel file
    workbook = Workbook()
    sheet_names = ['MCD_R', 'MCD_X', 'MCD_Y', 'Laser_R', 'Norm_MCD_R', 'Norm_MCD_X']
    sheets = {}

    # Generate wavelength column headers
    num_steps = int(round((end_wl - start_wl) / wl_step)) + 1
    wls = np.linspace(start_wl, end_wl, num_steps)
    headers = ['Vbg', 'Vtg'] + list(wls)

    # Generate Vbg and Vtg arrays
    vbgs = np.linspace(Vbg_start, Vbg_end, frames)
    vtgs = np.linspace(Vtg_start, Vtg_end, frames)

    # Create sheets, set headers, and pre-populate Vbg and Vtg columns
    for name in sheet_names:
        sheet = workbook.create_sheet(title=name)
        sheet.append(headers)  # Add headers to the sheet

        # Pre-populate Vbg and Vtg columns
        for vbg, vtg in zip(vbgs, vtgs):
            row = [vbg, vtg] + [None] * len(wls)  # Initialize with None for wavelength data
            sheet.append(row)

        sheets[name] = sheet

    del workbook['Sheet']  # Remove default sheet

    # Constant sampling factor
    sampling_factor = 10000

    # Iterate through Vbg and Vtg
    for frame_idx, (vbg, vtg) in enumerate(zip(vbgs, vtgs)):
        iv.x_goto('Vbg', vbg, 0.1, 0.05)
        iv.x_goto('Vtg', vtg, 0.1, 0.05)

        for wl_idx, wl in enumerate(wls):
            mono.wl_goto(wl)
            time.sleep(delay)

            raw_mcd_X = 0.0
            raw_mcd_Y = 0.0
            raw_laser_R = 0.0

            for _ in range(sampling):
                daq.read_y()
                raw_mcd_X += daq.update_y(2)
                raw_mcd_Y += daq.update_y(3)
                raw_laser_R += daq.update_y(4)

            mcd_X = mcd_sens * raw_mcd_X / sampling_factor
            mcd_Y = mcd_sens * raw_mcd_Y / sampling_factor
            laser_R = laser_sens * raw_laser_R / sampling_factor
            mcd_R = np.sqrt(mcd_X**2 + mcd_Y**2)
            norm_mcd_R = mcd_R / laser_R if laser_R != 0 else 0
            norm_mcd_X = mcd_X / laser_R if laser_R != 0 else 0

            # Update the pre-populated rows in the sheets
            for name, value in zip(sheet_names, [mcd_R, mcd_X, mcd_Y, laser_R, norm_mcd_R, norm_mcd_X]):
                sheet = sheets[name]
                sheet.cell(row=frame_idx + 2, column=wl_idx + 3, value=value)  # +2 for header and 1-based indexing

        # Save the workbook after each Vbg, Vtg iteration
        workbook.save(output_file_name)
        # print(f"Saved data for Vbg={vbg}, Vtg={vtg} to {output_file_name}")

    print(f"Experiment completed. Final data saved to {output_file_name}")

In [ ]:
from openpyxl import Workbook, load_workbook
from typing import Dict, List

def gate_sweep_AC_modulate(file_name: str, Vbg_start: float, Vbg_end: float, Vtg_start: float, Vtg_end: float, 
                           frames: int, start_wl: float, end_wl: float, wl_step: float, delay: float, 
                           sampling: int, mcd_sens: float, laser_sens: float) -> None:
    file_name = f'{file_name}_mcdsens={mcd_sens}mV_lasersens={laser_sens}mV'

    def check_file_name(file_name: str) -> str:
        """Ensure the file name is unique by appending a number if necessary."""
        file_number = 0
        while True:
            file_number += 1
            new_filename = f'{file_name}_{file_number:03d}'
            excel_name = new_filename + '.xlsx'
            if not os.path.exists(excel_name):
                return new_filename

    new_file_name = check_file_name(file_name)
    output_file_name = new_file_name + '.xlsx'

    # Generate wavelength array
    num_steps = int(round((end_wl - start_wl) / wl_step)) + 1
    wls = np.linspace(start_wl, end_wl, num_steps)

    # Generate Vbg and Vtg arrays
    vbgs = np.linspace(Vbg_start, Vbg_end, frames)
    vtgs = np.linspace(Vtg_start, Vtg_end, frames)

    # Initialize Excel file
    if os.path.exists(output_file_name):
        workbook = load_workbook(output_file_name)
    else:
        workbook = Workbook()
        del workbook['Sheet']  # Remove default sheet if creating a new file

    sheet_names = ['MCD_R', 'MCD_X', 'MCD_Y', 'Laser_R', 'Norm_MCD_R', 'Norm_MCD_X']
    sheets = {}

    # Create or load sheets and set headers
    for name in sheet_names:
        if name in workbook.sheetnames:
            sheet = workbook[name]
        else:
            sheet = workbook.create_sheet(title=name)
            # Add "Wavelength" as the first column header
            sheet.append(["Wavelength"])
            # Pre-populate Vbg row
            sheet.append(["Vbg"] + list(vbgs))
            # Pre-populate Vtg row
            sheet.append(["Vtg"] + list(vtgs))
            # Pre-populate wavelength data
            for wl in wls:
                sheet.append([wl])

        sheets[name] = sheet

    # Constant sampling factor
    sampling_factor = 10000

    # Iterate through Vbg and Vtg
    for sweep_idx, (vbg, vtg) in enumerate(zip(vbgs, vtgs)):
        iv.x_goto('Vbg', vbg, 0.1, 0.1)
        iv.x_goto('Vtg', vtg, 0.1, 0.1)
        time.sleep(1)
        for wl_idx, wl in enumerate(wls):
            # print(wl)
            if wl_idx == 0:
                time.sleep(5)
            mono.wl_goto(wl)
            time.sleep(delay)

            raw_mcd_X = 0.0
            raw_mcd_Y = 0.0
            raw_laser_R = 0.0

            for _ in range(sampling):
                daq.read_y()
                raw_mcd_X += daq.update_y(2)
                raw_mcd_Y += daq.update_y(3)
                raw_laser_R += daq.update_y(4)

            mcd_X = mcd_sens * raw_mcd_X / sampling_factor
            mcd_Y = mcd_sens * raw_mcd_Y / sampling_factor
            laser_R = laser_sens * raw_laser_R / sampling_factor
            mcd_R = np.sqrt(mcd_X**2 + mcd_Y**2)
            norm_mcd_R = mcd_R / laser_R if laser_R != 0 else 0
            norm_mcd_X = mcd_X / laser_R if laser_R != 0 else 0

            # Append data to the corresponding columns
            for name, value in zip(sheet_names, [mcd_R, mcd_X, mcd_Y, laser_R, norm_mcd_R, norm_mcd_X]):
                sheet = sheets[name]
                sheet.cell(row=wl_idx + 4, column=sweep_idx + 2, value=value)  # +4 for header and 1-based indexing
        # Save the workbook after each Vbg, Vtg iteration
        workbook.save(output_file_name)
    iv.x_goto('Vbg', 0, 0.1, 0.05)
    iv.x_goto('Vtg', 0, 0.1, 0.05)
    print(f"Experiment completed. Final data saved to {output_file_name}")

In [ ]:
from openpyxl import Workbook, load_workbook
from typing import Dict, List
import numpy as np
import os
import time

def gate_sweep_AC_modulate_ave(file_name: str, Vbg_start: float, Vbg_end: float, Vtg_start: float, Vtg_end: float, 
                           frames: int, start_wl: float, end_wl: float, wl_step: float, delay: float, 
                           sampling: int, repeat: int, mcd_sens: float, laser_sens: float) -> None:
    file_name = f'{file_name}_mcdsens={mcd_sens}mV_lasersens={laser_sens}mV'

    def check_file_name(file_name: str) -> str:
        """Ensure the file name is unique by appending a number if necessary."""
        file_number = 0
        while True:
            file_number += 1
            new_filename = f'{file_name}_{file_number:03d}'
            excel_name = new_filename + '.xlsx'
            if not os.path.exists(excel_name):
                return new_filename

    new_file_name = check_file_name(file_name)
    output_file_name = new_file_name + '.xlsx'

    # Generate wavelength array
    num_steps = int(round((end_wl - start_wl) / wl_step)) + 1
    wls = np.linspace(start_wl, end_wl, num_steps)

    # Generate Vbg and Vtg arrays
    vbgs = np.linspace(Vbg_start, Vbg_end, frames)
    vtgs = np.linspace(Vtg_start, Vtg_end, frames)

    # Initialize Excel file
    if os.path.exists(output_file_name):
        workbook = load_workbook(output_file_name)
    else:
        workbook = Workbook()
        del workbook['Sheet']  # Remove default sheet if creating a new file

    sheet_names = ['MCD_R', 'MCD_X', 'MCD_Y', 'Laser_R', 'Norm_MCD_R', 'Norm_MCD_X']
    sheets = {}

    # Create or load sheets and set headers
    for name in sheet_names:
        if name in workbook.sheetnames:
            sheet = workbook[name]
        else:
            sheet = workbook.create_sheet(title=name)
            # Add "Wavelength" as the first column header
            sheet.append(["Wavelength"])
            # Pre-populate Vbg row
            sheet.append(["Vbg"] + list(vbgs))
            # Pre-populate Vtg row
            sheet.append(["Vtg"] + list(vtgs))
            # Pre-populate wavelength data
            for wl in wls:
                sheet.append([wl])

        sheets[name] = sheet

    # Constant sampling factor
    sampling_factor = 10000

    # Iterate through Vbg and Vtg
    for sweep_idx, (vbg, vtg) in enumerate(zip(vbgs, vtgs)):
        iv.x_goto('Vbg', vbg, 0.1, 0.1)
        iv.x_goto('Vtg', vtg, 0.1, 0.1)
        time.sleep(1)

        # Initialize accumulators for averaging
        accumulated_values = {name: [] for name in sheet_names}

        for _ in range(repeat):
            for wl_idx, wl in enumerate(wls):
                mono.wl_goto(wl)
                time.sleep(delay)

                raw_mcd_X = 0.0
                raw_mcd_Y = 0.0
                raw_laser_R = 0.0

                for _ in range(sampling):
                    daq.read_y()
                    raw_mcd_X += daq.update_y(2)
                    raw_mcd_Y += daq.update_y(3)
                    raw_laser_R += daq.update_y(4)

                mcd_X = mcd_sens * raw_mcd_X / sampling_factor
                mcd_Y = mcd_sens * raw_mcd_Y / sampling_factor
                laser_R = laser_sens * raw_laser_R / sampling_factor
                mcd_R = np.sqrt(mcd_X**2 + mcd_Y**2)
                norm_mcd_R = mcd_R / laser_R if laser_R != 0 else 0
                norm_mcd_X = mcd_X / laser_R if laser_R != 0 else 0

                # Store data in accumulators for averaging
                accumulated_values['MCD_R'].append(mcd_R)
                accumulated_values['MCD_X'].append(mcd_X)
                accumulated_values['MCD_Y'].append(mcd_Y)
                accumulated_values['Laser_R'].append(laser_R)
                accumulated_values['Norm_MCD_R'].append(norm_mcd_R)
                accumulated_values['Norm_MCD_X'].append(norm_mcd_X)

        # After accumulating all the repeats, average the values
        averaged_values = {name: np.mean(values) for name, values in accumulated_values.items()}

        # Write the averaged values to the correct row for each wavelength and sweep
        for name, avg_value in averaged_values.items():
            sheet = sheets[name]
            # The row for wavelength data is wl_idx + 4 (because we have the headers in the first few rows)
            # Column for Vbg and Vtg is sweep_idx + 2 (to leave space for the row titles)
            sheet.cell(row=wl_idx + 4, column=sweep_idx + 2, value=avg_value)

        # Save the workbook after processing each Vbg, Vtg iteration
        workbook.save(output_file_name)

    iv.x_goto('Vbg', 0, 0.1, 0.05)
    iv.x_goto('Vtg', 0, 0.1, 0.05)
    print(f"Experiment completed. Final data saved to {output_file_name}")


In [ ]:
field = 90
pump_power = 0
mono.wl_goto(635)
meas_power =  c_double()
tlPM.setWavelength(635)
tlPM.measPower(byref(meas_power))
probe_power = np.around(meas_power.value*1e6,2)
print(meas_power.value*1e6)#unit uW
# print(probe_power)
start_wl = 640
end_wl = 680
repeat = 3
for _ in range(repeat):
	gate_sweep_AC_modulate(file_name=f'Dev197_PE_{np.around(field/10,3)}T_probe{probe_power}uW@635nm_pump670.5nm{pump_power}uw_BGonly_wls{start_wl}-{end_wl}_chopper497HZ', 
							Vbg_start=-1, Vbg_end=0, 
							Vtg_start=0, Vtg_end=0, frames=2, 
							start_wl=start_wl, end_wl=end_wl, wl_step=0.2, 
							delay=0.5, sampling=3, 
							mcd_sens=500, laser_sens=1000)
winsound.Beep(frequency=750, duration=300) 
# gate_sweep_AC_modulate(file_name=f'Dev127_Gatemodulation_PC_{np.around(field/10,3)}T+AC=160mV__wls{start_wl}-{end_wl}_3stepgain_whitelighton_70Hz', 
#                     Vbg_start=-7.5, Vbg_end=7.5, 
#                     Vtg_start=-3, Vtg_end=3, frames=31, 
#                     start_wl=680, end_wl=880, wl_step=0.2, 
#                     delay=0.5, sampling=3, 
#                     mcd_sens=500, laser_sens=1000)

## general PL/REF

In [ ]:
def check_file_name(file_name: str):
    file_number = 0
    while True:
        file_number += 1
        new_filename = f'{file_name}_{file_number:03d}'
        excel_name = new_filename + '.csv'
        if not os.path.exists(excel_name):
            return new_filename


In [ ]:
ep300.set_position(1,72)
# 45deg 1/2  27 
# 0deg 1/2 72
ep300.set_position(2,41)
#41 max
#86 min
#in K72 outK 41
#in K'27 out K'86

In [ ]:
ep300.get_positions(2)


In [ ]:
iv.x_goto('Vbg', 0, 0.1, 0.05)
iv.x_goto('Vtg', 0, 0.1, 0.05)
iv.report_status()

In [ ]:
step = 0.1
x = ex.x_channels['x_axis'].collect_x()
y = ex.x_channels['y_axis'].collect_x()
number_of_w = 0
number_of_a = 0
while True:
    key = input()
    print(key)
    if key == "q":
        break
    elif key == "a":
        if  y+step<10:
            y += step
            ex.x_goto('y_axis', y, 0, 0)
            number_of_a += 1 
    elif key == "d":
        if y-step>0:
            y -= step 
            ex.x_goto('y_axis', y, 0, 0)
            number_of_a -= 1 
    elif key == "w":
        if x+step<10:
            x += step
            ex.x_goto('x_axis', x, 0, 0)
            number_of_w += 1
    elif key == "s":
        if x-step>0:
            x -= step
            ex.x_goto('x_axis', x, 0, 0)
            number_of_w -= 1
print('a:',number_of_a,'w:',number_of_w)

# a: -11 w:dd 0s


In [ ]:
pump_stage.move_to(600)
meas_power =  c_double()
tlPM.measPower(byref(meas_power),1)
print(meas_power.value*1e6)#unit uW
power = np.round(meas_power.value*1e6,3)

In [ ]:
# Doping dependent REF under different E-fields
frame_to_combine = '6'
repeat_n = 1
exp_time = '500'
center ='640'   
lf6.change_expose_time(exp_time)
lf6.change_spectra_center(center)
lf6.change_frame_to_combine(frame_to_combine)   
sample_name=f'$yzD289$~$4Kref{center}nm$~$p5$~$0T$~${exp_time}sx{frame_to_combine}x{repeat_n}_HG$~'
exps = spectral_experiments.SpectralExperimentsCollections()
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG-1.2BG=0$', iv, lf6, vbg_start=-8, vbg_stop=13,
#                                 vtg_start=-9.6, vtg_stop=15.6, frames=211, repeat= repeat_n, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG+1.2BG=0$', iv, lf6, vbg_start=-12, vbg_stop=12,
#                                 vtg_start=14.4, vtg_stop=-14.4, frames=241, repeat= repeat_n, plot=False)

exps.add_dual_gate_spectra_sweep(sample_name, '$bgTG-1.2BG=0$', iv, lf6, vbg_start=10, vbg_stop=13,
                                vtg_start=12, vtg_stop=15.6, frames=31, repeat= repeat_n, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG-1.2BG=25$', iv, lf6, vbg_start=-12, vbg_stop=-8,
#                                 vtg_start=10.6, vtg_stop=16.4, frames=91, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-1.2BG=22$', iv, lf6, vbg_start=-12, vbg_stop=-5,
                                vtg_start=7.6, vtg_stop=16, frames=71, repeat= repeat_n, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG-1.2BG=20$', iv, lf6, vbg_start=-12, vbg_stop=-3,
#                                 vtg_start=5.6, vtg_stop=16.4, frames=91, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-1.2BG=18$', iv, lf6, vbg_start=-12, vbg_stop=-2,
                                vtg_start=3.6, vtg_stop=15.6, frames=101, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-1.2BG=16$', iv, lf6, vbg_start=-12, vbg_stop=0,
                                vtg_start=1.6, vtg_stop=16, frames=121, repeat= repeat_n, plot=False)

exps.add_dual_gate_spectra_sweep(sample_name, '$bgTG-1.2BG=0$', iv, lf6, vbg_start=10, vbg_stop=13,
                                vtg_start=12, vtg_stop=15.6, frames=31, repeat= repeat_n, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG-1.2BG=15$', iv, lf6, vbg_start=-12, vbg_stop=1,
#                                 vtg_start=0.6, vtg_stop=16.2, frames=131, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-1.2BG=14$', iv, lf6, vbg_start=-12, vbg_stop=2,
                                vtg_start=-0.4, vtg_stop=16.4, frames=141, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-1.2BG=12$', iv, lf6, vbg_start=-12, vbg_stop=3,
                                vtg_start=-2.4, vtg_stop=15.6, frames=151, repeat= repeat_n, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG-1.2BG=10$', iv, lf6, vbg_start=-12, vbg_stop=5,
#                                 vtg_start=-4.4, vtg_stop=16, frames=171, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-1.2BG=8$', iv, lf6, vbg_start=-10, vbg_stop=7,
                                vtg_start=-4, vtg_stop=16.4, frames=171, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-1.2BG=6$', iv, lf6, vbg_start=-10, vbg_stop=8,
                                vtg_start=-6, vtg_stop=15.6, frames=181, repeat= repeat_n, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG-1.2BG=5$', iv, lf6, vbg_start=-9, vbg_stop=9,
#                                 vtg_start=-5.8, vtg_stop=15.8, frames=181, repeat= repeat_n, plot=False)

exps.add_dual_gate_spectra_sweep(sample_name, '$bgTG-1.2BG=0$', iv, lf6, vbg_start=10, vbg_stop=13,
                                vtg_start=12, vtg_stop=15.6, frames=31, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-1.2BG=4$', iv, lf6, vbg_start=-9, vbg_stop=10,
                                vtg_start=-6.8, vtg_stop=16, frames=191, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-1.2BG=2$', iv, lf6, vbg_start=-8, vbg_stop=11,
                                vtg_start=-7.6, vtg_stop=15.2, frames=191, repeat= repeat_n, plot=False)
exps.execute(repeat=1)

In [ ]:
# E-fields dependent REF under different dopings
frame_to_combine = '6'
repeat_n = 1
exp_time = '500'
center ='640'   
lf6.change_expose_time(exp_time)
lf6.change_spectra_center(center)
lf6.change_frame_to_combine(frame_to_combine)   
sample_name=f'$yzD289$~$4Kref{center}nm$~$p5$~$0T$~${exp_time}sx{frame_to_combine}x{repeat_n}_HG$~'
exps = spectral_experiments.SpectralExperimentsCollections()
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG-1.2BG=0$', iv, lf6, vbg_start=-8, vbg_stop=13,
#                                 vtg_start=-9.6, vtg_stop=15.6, frames=211, repeat= repeat_n, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG+1.2BG=0$', iv, lf6, vbg_start=-12, vbg_stop=12,
#                                 vtg_start=14.4, vtg_stop=-14.4, frames=241, repeat= repeat_n, plot=False)

exps.add_dual_gate_spectra_sweep(sample_name, '$bgTG-1.2BG=0$', iv, lf6, vbg_start=10, vbg_stop=13,
                                vtg_start=12, vtg_stop=15.6, frames=31, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+1.2BG=2$', iv, lf6, vbg_start=-10, vbg_stop=12,
                                vtg_start=14, vtg_stop=-12.4, frames=221, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+1.2BG=3.5$', iv, lf6, vbg_start=-9, vbg_stop=12,
                                vtg_start=14.3, vtg_stop=-10.9, frames=211, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+1.2BG=5$', iv, lf6, vbg_start=-8, vbg_stop=12,
                                vtg_start=14.6, vtg_stop=-9.4, frames=201, repeat= repeat_n, plot=False)

exps.add_dual_gate_spectra_sweep(sample_name, '$bgTG-1.2BG=0$', iv, lf6, vbg_start=10, vbg_stop=13,
                                vtg_start=12, vtg_stop=15.6, frames=31, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+1.2BG=7$', iv, lf6, vbg_start=-7, vbg_stop=12,
                                vtg_start=15.4, vtg_stop=-7.4, frames=191, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+1.2BG=10$', iv, lf6, vbg_start=-5, vbg_stop=13,
                                vtg_start=16, vtg_stop=-5.6, frames=181, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+1.2BG=13$', iv, lf6, vbg_start=-3, vbg_stop=13,
                                vtg_start=16.6, vtg_stop=-2.6, frames=161, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG+1.2BG=16$', iv, lf6, vbg_start=0, vbg_stop=13,
                                vtg_start=16, vtg_stop=0.4, frames=131, repeat= repeat_n, plot=False)
exps.execute(repeat=1)

In [ ]:
import sympy as sp

tg, bg = sp.symbols('tg bg')

# two equations
## -10, 18.8
eq1 = sp.Eq(0.9*tg - bg, 10)
eq2 = sp.Eq(0.9*tg + bg, -15)

# solve for tg, bg
sol = sp.solve((eq1, eq2), (tg, bg))
print(sol)


In [ ]:
ep300.set_position(1,98.7+45)

In [ ]:
# ep300.set_position(1,190)
# # inhalf 190 190-45
frame_to_combine = '10'
repeat_n = 6
exp_time = '300'
centers =['780']

lf6.change_expose_time(exp_time)
lf6.change_frame_to_combine(frame_to_combine)
in_halfs = [98.7, 98.7+45]
for  center in centers:    
	lf6.change_spectra_center(center)
	for in_half in in_halfs:
		ep300.set_position(1,in_half)
		# sample_name=f'$YZ315$~$4KPL{center}nm1uW$~$p2$~$0T$~${exp_time}sx{frame_to_combine}x{repeat_n}$'
		sample_name=f'$YZ315$~$4KREF{center}nm$~$p2$~$8Thf={in_half}$~${exp_time}msx{frame_to_combine}x{repeat_n}$'
		exps = spectral_experiments.SpectralExperimentsCollections()

		# exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly$', iv, lf6, vbg_start=0, vbg_stop=0,
		#                             vtg_start=-20, vtg_stop=20, frames=201, repeat= repeat_n, plot=False)
		# exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly$', iv, lf6, vbg_start=-20, vbg_stop=20,
		#                             vtg_start=0, vtg_stop=0, frames=201, repeat= repeat_n, plot=False)


		exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=0$', iv, lf6, vbg_start=-9, vbg_stop=19.8,
		                            vtg_start=-10, vtg_stop=22, frames=321, repeat= repeat_n, plot=False)
		# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=0$', iv, lf6, vbg_start=9.9, vbg_stop=-22.5,
		#                             vtg_start=-11, vtg_stop=25, frames=361, repeat=repeat_n, plot=False)


		# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=10$', iv, lf6, vbg_start=-14.5, vbg_stop=5.3,
		#                             vtg_start=-5, vtg_stop=17, frames=221, repeat= repeat_n, plot=False)
		# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=20$', iv, lf6, vbg_start=-20, vbg_stop=-2,
		#                             vtg_start=0, vtg_stop=20, frames=201, repeat= repeat_n, plot=False)
		# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=30$', iv, lf6, vbg_start=-21, vbg_stop=-7.5,
		#                             vtg_start=10, vtg_stop=25, frames=151, repeat= repeat_n, plot=False)


		
		# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=35$', iv, lf6, vbg_start=-21.5, vbg_stop=-8,
		# 							vtg_start=15, vtg_stop=30, frames=151, repeat= repeat_n, plot=False)
		
		# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=37.5$', iv, lf6, vbg_start=-21.75, vbg_stop=-8.25,
		# 							vtg_start=17.5, vtg_stop=32.5, frames=151, repeat= repeat_n, plot=False)
		
		# exps.add_dual_gate_spectra_sweep(sample_name, '$BACK0.9TG-BG=0$', iv, lf6, vbg_start=19.7, vbg_stop=19.8,
		# 							vtg_start=21.9, vtg_stop=22, frames=2, repeat= repeat_n, plot=False)
		# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=42.5$', iv, lf6, vbg_start=-22.25, vbg_stop=-10.1,
		# 							vtg_start=22.5, vtg_stop=36, frames=136, repeat= repeat_n, plot=False)

		
		# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=40$', iv, lf6, vbg_start=-22, vbg_stop=-10.3,
		#                             vtg_start=20, vtg_stop=33, frames=131, repeat= repeat_n, plot=False)
		
		# exps.add_dual_gate_spectra_sweep(sample_name, '$BACK0.9TG-BG=0$', iv, lf6, vbg_start=19.7, vbg_stop=19.8,
		#                             vtg_start=21.9, vtg_stop=22, frames=2, repeat= repeat_n, plot=False)
		
		# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=45$', iv, lf6, vbg_start=-22.5, vbg_stop=-12.6,
		# 							vtg_start=25, vtg_stop=36, frames=111, repeat= repeat_n, plot=False)


		
		exps.execute(repeat=1)

winsound.Beep(frequency=750, duration=300)   

In [ ]:
# reflection megasweep with background 
# sample_name0 = '$Y311$~$REFMegaseep$~$4Kp5$~$630nmC$~$0.8sx5$~$REF_forBackground$'
# exps = spectral_experiments.SpectralExperimentsCollections()
# exps.add_dual_gate_spectra_sweep(sample_name0, '$TG-BG=0$', iv, lf6, vbg_start=-10, vbg_stop=10,
#                                 vtg_start=-10, vtg_stop=10, frames=51, repeat= repeat_n, plot=False)
# exps.execute(repeat=1)



f_name =f'$Y311$~$REFMegaseep$~$4Kp5$~$630nmC$~$0.8sx5$'
sample_name = f_name
vbgs = np.linspace(-15, 4, 96)
vtgs = np.linspace(-4, 15, 96)

flag = True
step = 0.1 #V
delay = 0.05 #s
with open('{}.csv'.format(f_name), 'a') as f:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['Vbg','Vtg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f, cols, fmt='%s', delimiter=',')
            for vbg in vbgs:
                iv.x_goto('Vbg', vbg, step, delay)
                if flag == True:
                    vtgseq = vtgs
                else:
                    vtgseq = np.flip(vtgs)
                for vtg in vtgseq:
                    iv.x_goto('Vtg', vtg, step, delay)
                    spectra = lf6.acquire()
                    data = np.concatenate((np.array([vbg, vtg], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                    np.savetxt(f, data, fmt='%.5e', delimiter=',') 
                flag = not flag
            iv.x_goto('Vbg', 0, step, delay)
            iv.x_goto('Vtg', 0, step, delay)
            print('vg:', 0)
            

sample_name0 = '$Y311$~$REFMegaseep$~$4Kp5$~$630nmC$~$0.8sx5$~$REF_forBackgroundend$'
exps = spectral_experiments.SpectralExperimentsCollections()
exps.add_dual_gate_spectra_sweep(sample_name0, '$TG-BG=0$', iv, lf6, vbg_start=-10, vbg_stop=10,
                                vtg_start=-10, vtg_stop=10, frames=51, repeat= repeat_n, plot=False)
exps.execute(repeat=1)

In [ ]:
iv.x_goto('Vbg', 0, 0.2, 0.05)
iv.x_goto('Vtg', 0, 0.2, 0.05)
iv.report_status()

In [ ]:
iv.x_goto('Vbg', 4, 0.1, 0.05)
iv.x_goto('Vtg', 15, 0.1, 0.05)
iv.report_status()

In [ ]:
frame_to_combine = '5'
repeat_n = 1
exp_time = '800'
# centers =['750', '860']
centers =['630']

xscan = 6.5
yscan = 6.4
# ex.x_goto('x_axis', xscan, 0.1, 0.1)
# ex.x_goto('y_axis', yscan, 0.1, 0.1)

for  center in centers:    
# for vds in vdss:
    lf6.change_expose_time(exp_time)
    lf6.change_spectra_center(center)
    lf6.change_frame_to_combine(frame_to_combine)
    sample_name=f'$YZ311$~$4KREF{center}nm$~$p1x{xscan}y{yscan}$~$0T$~${exp_time}sx{frame_to_combine}x{repeat_n}$'
    exps = spectral_experiments.SpectralExperimentsCollections()
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-10, vbg_stop=16,
                                vtg_start=-10, vtg_stop=16, frames=261, repeat= repeat_n, plot=False)
    exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly$', iv, lf6, vbg_start=0, vbg_stop=0,
                                vtg_start=-15, vtg_stop=15, frames=151, repeat= repeat_n, plot=False)
    exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly$', iv, lf6, vbg_start=-10, vbg_stop=10,
                                vtg_start=0, vtg_stop=0, frames=101, repeat= repeat_n, plot=False)
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0$', iv, lf6, vbg_start=-9, vbg_stop=6.5,
                                vtg_start=9, vtg_stop=-6.5, frames=146, repeat=repeat_n, plot=False)
    exps.execute(repeat=1)
winsound.Beep(frequency=750, duration=300)    
 


In [ ]:
frame_to_combine = '5'
repeat_n = 1
# exp_times = ['500']
exp_time = '800'
# centers =['750', '860']
centers =['630', '720']
# vdss=[1.6, 1.2, -1.2, -1.6, 0]
# stage_poss = [765]
# exp_times = ['5000', '3000','2000','1000','1000','1000','1000']
# stage_poss = [1200, 1600, 2000, 2300, 2650, 2900,3300]
# for  exp_time in exp_times:
for  center in centers:    
# for vds in vdss:
    lf6.change_expose_time(exp_time)
    lf6.change_spectra_center(center)
    lf6.change_frame_to_combine(frame_to_combine)
    # iv.x_goto('Vcg', vds, 0.1, 0.1)
    # pump_stage.move_to(stage_pos)
    # time.sleep(2)
    # meas_power =  c_double()
    # tlPM.measPower(byref(meas_power),1)
    # print(meas_power.value*1e6)#unit uW
    # power = np.round(meas_power.value*1e6,3)
    
    # sample_name=f'$YZ311$~$4KPL{center}nm$~$p1$~$0.02uW@633$~$0T$~${exp_time}sx{frame_to_combine}x{repeat_n}$'
    sample_name=f'$YZ311$~$4KREF{center}nm$~$p1$~$0T$~${exp_time}sx{frame_to_combine}x{repeat_n}$'

    exps = spectral_experiments.SpectralExperimentsCollections()



    # exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly$', iv, lf6, vbg_start=0, vbg_stop=0,
    #                             vtg_start=-1.5, vtg_stop=1.5, frames=31, repeat= repeat_n, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly$', iv, lf6, vbg_start=0, vbg_stop=0,
    #                             vtg_start=-6, vtg_stop=10, frames=161, repeat= repeat_n, plot=False)
    # # exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly$', iv, lf6, vbg_start=-2, vbg_stop=-2,
    #                             vtg_start=-6, vtg_stop=8, frames=141, repeat= repeat_n, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly$', iv, lf6, vbg_start=-1.6, vbg_stop=-1.6,
    #                             vtg_start=-6, vtg_stop=8, frames=141, repeat= repeat_n, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly$', iv, lf6, vbg_start=0, vbg_stop=0,
    #                             vtg_start=-15, vtg_stop=15, frames=151, repeat= repeat_n, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly$', iv, lf6, vbg_start=-10, vbg_stop=10,
    #                             vtg_start=0, vtg_stop=0, frames=101, repeat= repeat_n, plot=False)
    exps.add_dual_gate_spectra_sweep(sample_name, '$2TG-BG=0$', iv, lf6, vbg_start=-10, vbg_stop=10,
                                vtg_start=-5, vtg_stop=5, frames=101, repeat= repeat_n, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly(TG1)$', iv, lf6, vbg_start=0, vbg_stop=0,
    #                             vtg_start=8, vtg_stop=-8, frames=161, repeat= repeat_n, plot=False)
    exps.add_dual_gate_spectra_sweep(sample_name, '$2TG+BG=0$', iv, lf6, vbg_start=-9, vbg_stop=6.4,
                                vtg_start=4.5, vtg_stop=-3.2, frames=78, repeat=repeat_n, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly(TG2)$', iv, lf6, vbg_start=-5, vbg_stop=5,
    #                             vtg_start=0, vtg_stop=0, frames=101, repeat=1, plot=False)

    # exps.add_dual_gate_spectra_sweep(sample_name, '$TGonlyrev$', iv, lf6, vbg_start=0, vbg_stop=0,
    #                             vtg_start=9, vtg_stop=-9, frames=181, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$0.7TG-BG=0$', iv, lf6, vbg_start=6.3, vbg_stop=7.7,
    #                             vtg_start=9, vtg_stop=11, frames=21, repeat= repeat_n, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$0.7TG+BG=0$', iv, lf6, vbg_start=-7.7, vbg_stop=7.7,
    #                             vtg_start=11, vtg_stop=-11, frames=111, repeat= repeat_n, plot=False)

    # exps.add_dual_gate_spectra_sweep(sample_name, '$T-0.56BG=-0.765$', iv, lf6, vbg_start=0.3, vbg_stop=1.84,
    #                             vtg_start=-6, vtg_stop=2.6, frames=87, repeat= repeat_n, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+2.22BG=6.73$', iv, lf6, vbg_start=1.8, vbg_stop=-0.6,
    #                             vtg_start=2.7, vtg_stop=8, frames=54, repeat= repeat_n, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-2$', iv, lf6, vbg_start=-6, vbg_stop=4,
    #                             vtg_start=4, vtg_stop=-6, frames=101, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-0.5$', iv, lf6, vbg_start=-6, vbg_stop=5.5,
    #                             vtg_start=5.5, vtg_stop=-6, frames=116, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=2.7$', iv, lf6, vbg_start=-3.3, vbg_stop=6,
    #                             vtg_start=6, vtg_stop=-3.3, frames=94, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=1.6$', iv, lf6, vbg_start=-4.4, vbg_stop=6,
    #                             vtg_start=6, vtg_stop=-4.4, frames=105, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=4.8$', iv, lf6, 
    #                                  vbg_start=6, vbg_stop=-1.2,vtg_start=-1.2, vtg_stop=6, frames=144, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=3.7$', iv, lf6, 
    #                                  vbg_start=-3.4, vbg_stop=7.1,vtg_start=7.1, vtg_stop=-3.4, frames=106, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=4.8$', iv, lf6, 
    #                                  vbg_start=-5.2, vbg_stop=8,vtg_start=10, vtg_stop=-3.2, frames=133, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=5.0$', iv, lf6, 
    #                                  vbg_start=-5, vbg_stop=8,vtg_start=10, vtg_stop=-3, frames=131, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=5.2$', iv, lf6, 
    #                                  vbg_start=-4.8, vbg_stop=8,vtg_start=10, vtg_stop=-2.8, frames=129, repeat=1, plot=False)

    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=5.2rev$', iv, lf6, 
    #                                  vbg_start=8, vbg_stop=-4.8,vtg_start=-2.8, vtg_stop=10, frames=129, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=5.4$', iv, lf6, 
    #                                  vbg_start=-4.6, vbg_stop=8,vtg_start=10, vtg_stop=-2.6, frames=127, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=5.6$', iv, lf6, 
    #                                  vbg_start=-4.4, vbg_stop=8,vtg_start=10, vtg_stop=-2.4, frames=125, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=5.8$', iv, lf6, 
    #                                  vbg_start=-4.2, vbg_stop=8,vtg_start=10, vtg_stop=-2.2, frames=123, repeat=1, plot=False)

    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=4.4$', iv, lf6, 
    #                                  vbg_start=-4.8, vbg_stop=6.6,vtg_start=9.2, vtg_stop=-2.2, frames=115, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=4.0$', iv, lf6, 
    #                                  vbg_start=-4.8, vbg_stop=6.2,vtg_start=8.8, vtg_stop=-2.2, frames=111, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=6.0$', iv, lf6, 
    #                                  vbg_start=-4, vbg_stop=8,vtg_start=10, vtg_stop=-2, frames=121, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=6.2$', iv, lf6, 
    #                                  vbg_start=-3.8, vbg_stop=8,vtg_start=10, vtg_stop=-1.8, frames=119, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=6.4$', iv, lf6, 
    #                                  vbg_start=-3.6, vbg_stop=8,vtg_start=10, vtg_stop=-1.6, frames=117, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=8.0$', iv, lf6, 
    #                                  vbg_start=-2, vbg_stop=8,vtg_start=10, vtg_stop=0, frames=101, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=7.0$', iv, lf6, 
    #                                  vbg_start=-3, vbg_stop=8,vtg_start=10, vtg_stop=-1, frames=111, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-3.6$', iv, lf6, 
    #                                  vbg_start=-7.1, vbg_stop=3.5,vtg_start=3.5, vtg_stop=-7.1, frames=107, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-4.5$', iv, lf6, 
    #                                  vbg_start=-7.1, vbg_stop=2.6,vtg_start=2.6, vtg_stop=-7.1, frames=98, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-6.0$', iv, lf6, 
    #                                  vbg_start=-7.0, vbg_stop=1,vtg_start=1, vtg_stop=-7.0, frames=81, repeat=1, plot=False)
    exps.execute(repeat=1)
winsound.Beep(frequency=750, duration=300)





In [ ]:
ep300.set_position(1,98.7)
# inhalf 40.9+90 40.9+90+45

In [ ]:
# For 9T measurements
repeat_n = 1
frame_to_combine = '1'
exp_time = '500'
lf6.change_expose_time(exp_time)
lf6.change_frame_to_combine(frame_to_combine)
in_halfs = [40.9+90, 40.9+90+45] 
bfield = 9
for in_half in in_halfs:
	ep300.set_position(1,in_half)
	# For 9T measurements

	# 720nm center
	lf6.change_spectra_center('725')
	sample_name=f'$YZD241$~$4KREF720nm$~$p4n0$~${bfield}T{in_half}deg$~${exp_time}msx{frame_to_combine}x{repeat_n}HG$~'   
	exps = spectral_experiments.SpectralExperimentsCollections()
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=0$', iv, lf6, vbg_start= -9, vbg_stop= 11.7,
											vtg_start= -10, vtg_stop= 13, frames= 231, repeat= repeat_n, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=0$', iv, lf6, vbg_start= -11.7, vbg_stop= 11.7,
											vtg_start= 13, vtg_stop= -13, frames= 131, repeat= repeat_n, plot=False)	
	exps.execute(repeat=1)


	# 630nm center
	lf6.change_spectra_center('630')
	sample_name=f'$YZD241$~$4KREF630nm$~$p4n0$~${bfield}T{in_half}deg$~${exp_time}msx{frame_to_combine}x{repeat_n}_HG$~'   
	exps = spectral_experiments.SpectralExperimentsCollections()
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=0$', iv, lf6, vbg_start= -11.7, vbg_stop= 11.7,
											vtg_start= 13, vtg_stop= -13, frames= 261, repeat= repeat_n, plot=False)	
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=0$', iv, lf6, vbg_start= -9, vbg_stop= 11.7,
											vtg_start= -10, vtg_stop= 13, frames= 231, repeat= repeat_n, plot=False)									 
	exps.execute(repeat=1)


	#n-doping E-dependent REF
	lf6.change_spectra_center('630')
	repeat_n = 4
	sample_name=f'$YZD241$~$4KREF630nm$~$p4n0$~${bfield}T{in_half}deg$~${exp_time}msx{frame_to_combine}x{repeat_n}_HG$~'   
	exps = spectral_experiments.SpectralExperimentsCollections()
	exps.add_dual_gate_spectra_sweep(sample_name, '$bg0.9TG-BG=0$', iv, lf6, vbg_start= 9, vbg_stop= 11.7,
											vtg_start= 10, vtg_stop= 13, frames= 31, repeat= repeat_n, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=2$', iv, lf6, vbg_start= -9.7, vbg_stop= 11.9,
											vtg_start= 13, vtg_stop= -11, frames= 241, repeat= repeat_n, plot=False)	
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=2.25$', iv, lf6, vbg_start= -9.45, vbg_stop= 12.15,
											vtg_start= 13, vtg_stop= -11, frames= 241, repeat= repeat_n, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=2.5$', iv, lf6, vbg_start= -9.2, vbg_stop= 12.4,
											vtg_start= 13, vtg_stop= -11, frames= 241, repeat= repeat_n, plot=False)	
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=2.75$', iv, lf6, vbg_start= -8.95, vbg_stop= 12.65,
											vtg_start= 13, vtg_stop= -11, frames= 241, repeat= repeat_n, plot=False)	

	exps.add_dual_gate_spectra_sweep(sample_name, '$bg0.9TG-BG=0$', iv, lf6, vbg_start= 9, vbg_stop= 11.7,
											vtg_start= 10, vtg_stop= 13, frames= 31, repeat= repeat_n, plot=False)	
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=3$', iv, lf6, vbg_start= -8.7, vbg_stop= 12,
											vtg_start= 13, vtg_stop= -10, frames= 231, repeat= repeat_n, plot=False)		
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=4$', iv, lf6, vbg_start= -7.7, vbg_stop= 12.1,
											vtg_start= 13, vtg_stop= -9, frames= 221, repeat= repeat_n, plot=False)	
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=5$', iv, lf6, vbg_start= -7.6, vbg_stop= 12.2,
											vtg_start= 14, vtg_stop= -8, frames= 221, repeat= repeat_n, plot=False)

	exps.add_dual_gate_spectra_sweep(sample_name, '$bg0.9TG-BG=0$', iv, lf6, vbg_start= 9, vbg_stop= 11.7,
											vtg_start= 10, vtg_stop= 13, frames= 31, repeat= repeat_n, plot=False)	
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=7.5$', iv, lf6, vbg_start= -5.1, vbg_stop= 12.9,
											vtg_start= 14, vtg_stop= -6, frames= 201, repeat= repeat_n, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=10$', iv, lf6, vbg_start= -3.5, vbg_stop= 13.6,
											vtg_start= 15, vtg_stop= -4, frames= 191, repeat= repeat_n, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=12$', iv, lf6, vbg_start= -1.5, vbg_stop= 13.8,
											vtg_start= 15, vtg_stop= -2, frames= 171, repeat= repeat_n, plot=False)

	exps.add_dual_gate_spectra_sweep(sample_name, '$bg0.9TG-BG=0$', iv, lf6, vbg_start= 9, vbg_stop= 11.7,
											vtg_start= 10, vtg_stop= 13, frames= 31, repeat= repeat_n, plot=False)	
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=14$', iv, lf6, vbg_start= 0.5, vbg_stop= 14,
											vtg_start= 15, vtg_stop= 0, frames= 151, repeat= repeat_n, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=16$', iv, lf6, vbg_start= 2.5, vbg_stop= 14.2,
											vtg_start= 15, vtg_stop= 2, frames= 131, repeat= repeat_n, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=18$', iv, lf6, vbg_start= 4.5, vbg_stop= 13.5,
											vtg_start= 15, vtg_stop= 5, frames= 101, repeat= repeat_n, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=20$', iv, lf6, vbg_start= 6.5, vbg_stop= 13.7,
											vtg_start= 15, vtg_stop= 7, frames= 81, repeat= repeat_n, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$bg0.9TG-BG=0$', iv, lf6, vbg_start= 9, vbg_stop= 11.7,
											vtg_start= 10, vtg_stop= 13, frames= 31, repeat= repeat_n, plot=False)	

	#p-doping E-dependent REF
	exps.add_dual_gate_spectra_sweep(sample_name, '$bg0.9TG-BG=0$', iv, lf6, vbg_start= 9, vbg_stop= 11.7,
											vtg_start= 10, vtg_stop= 13, frames= 31, repeat= repeat_n, plot=False)	
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=-2.5$', iv, lf6, vbg_start= -12.4, vbg_stop= 9.2,
											vtg_start= 11, vtg_stop= -13, frames= 241, repeat= repeat_n, plot=False)	
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=-5$', iv, lf6, vbg_start= -13.1, vbg_stop= 6.7,
											vtg_start= 9, vtg_stop= -13, frames= 221, repeat= repeat_n, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=-7.5$', iv, lf6, vbg_start= -12.9, vbg_stop= 4.2,
											vtg_start= 6, vtg_stop= -13, frames= 191, repeat= repeat_n, plot=False)

	exps.add_dual_gate_spectra_sweep(sample_name, '$bg0.9TG-BG=0$', iv, lf6, vbg_start= 9, vbg_stop= 11.7,
											vtg_start= 10, vtg_stop= 13, frames= 31, repeat= repeat_n, plot=False)	
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=-10$', iv, lf6, vbg_start= -13.6, vbg_stop= 1.7,
											vtg_start= 4, vtg_stop= -13, frames= 171, repeat= repeat_n, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=-12.5$', iv, lf6, vbg_start= -13.4, vbg_stop= -0.8,
											vtg_start= 1, vtg_stop= -13, frames= 141, repeat= repeat_n, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=-15$', iv, lf6, vbg_start= -13.2, vbg_stop= -3.3,
											vtg_start= -2, vtg_stop= -13, frames= 111, repeat= repeat_n, plot=False)

	exps.add_dual_gate_spectra_sweep(sample_name, '$bg0.9TG-BG=0$', iv, lf6, vbg_start= 9, vbg_stop= 11.7,
											vtg_start= 10, vtg_stop= 13, frames= 31, repeat= repeat_n, plot=False)	
	exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=-16$', iv, lf6, vbg_start= -13.3, vbg_stop= -4.3,
											vtg_start= -3, vtg_stop= -13, frames= 101, repeat= repeat_n, plot=False)

	exps.execute(repeat=1)



In [ ]:
# Refactored version of your 9T measurement code
# Each sweep is measured at two angles, alternating the order between sweeps

from collections import namedtuple

# Define sweep configuration
Sweep = namedtuple('Sweep', ['label', 'vbg_start', 'vbg_stop', 'vtg_start', 'vtg_stop', 'frames','repeat_n'])

# Sweep definitions
sweeps_all = [


		('780',  [
		# Sweep('$bg0.9TG-BG=0$', 9, 11.7, 10, 13, 31),
		Sweep('$0.9TG-BG=0$',     -9,    19.8,  -10,   22,   321, 5),
        Sweep('$bg0.9TG-BG=0$',   19.35,   19.8,  21.5,  22,     6, 5),

		Sweep('$0.9TG+BG=0$',      9.9,  -26.1,  -11,   29,   401, 5),
        Sweep('$bg0.9TG-BG=0$',   19.35,   19.8,  21.5,  22,     6, 5),
		Sweep('$0.9TG-BG=10$',   -14.5,   5.3,   -5,    17,   221, 5),
        # Sweep('$bg0.9TG-BG=0$',   19.35,   19.8,  21.5,  22,     6),
        Sweep('$bg0.9TG-BG=0$',   19.35,   19.8,  21.5,  22,     6, 5),

		Sweep('$0.9TG-BG=20$',   -20,    -2,     0,     20,   201, 5),
        Sweep('$bg0.9TG-BG=0$',   19.35,   19.8,  21.5,  22,     6, 5),
		Sweep('$0.9TG-BG=30$',   -21,    -7.5,   10,    25,   151,5 ),
        # Sweep('$bg0.9TG-BG=0$',   19.7,   19.8,  21.9,  22,     2),
		# Sweep('$0.9TG-BG=35$',   -21.5,  -8,     15,    30,   151),
        # Sweep('$bg0.9TG-BG=0$',   19.35,   19.8,  21.5,  22,     6),
        # Sweep('$0.9TG-BG=36$',    -21.6, -6.3,  16,  33, 171),
        Sweep('$bg0.9TG-BG=0$',   19.35,   19.8,  21.5,  22,     6, 5),
		Sweep('$0.9TG-BG=37.5$', -21.75, -8.25,  17.5,  32.5, 151,5),
        Sweep('$bg0.9TG-BG=0$',   19.35,   19.8,  21.5,  22,     6,5),

        # Sweep('$0.9TG-BG=38$',    -21.8, -8.3,  18,  33, 151),
		Sweep('$0.9TG-BG=40$',   -22,    -10.3,  20,    33,   131, 5),
        Sweep('$bg0.9TG-BG=0$',   19.35,   19.8,  21.5,  22,     6, 5),
		Sweep('$0.9TG-BG=42.5$', -22.25, -10.1,  22.5,  36,   136, 5),
        # Sweep('$bg0.9TG-BG=0$',   19.7,   19.8,  21.9,  22,     2),
        Sweep('$bg0.9TG-BG=0$',   19.35,   19.8,  21.5,  22,     6, 5),
		Sweep('$0.9TG-BG=45$',   -22.5,  -12.6,  25,    36,   111, 5 ),
        Sweep('$bg0.9TG-BG=0$',   19.35,   19.8,  21.5,  22,     6, 5),

        # Sweep('$0.9TG-BG=-10$',  -0.8,      15.4,  -12,   6,    181),


		# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=10$', iv, lf6, vbg_start=-14.5, vbg_stop=5.3,
		#                             vtg_start=-5, vtg_stop=17, frames=221, repeat= repeat_n, plot=False)
		# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=20$', iv, lf6, vbg_start=-20, vbg_stop=-2,
		#                             vtg_start=0, vtg_stop=20, frames=201, repeat= repeat_n, plot=False)
		# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=30$', iv, lf6, vbg_start=-21, vbg_stop=-7.5,
		#                             vtg_start=10, vtg_stop=25, frames=151, repeat= repeat_n, plot=False)


		
		# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=35$', iv, lf6, vbg_start=-21.5, vbg_stop=-8,
		# 							vtg_start=15, vtg_stop=30, frames=151, repeat= repeat_n, plot=False)
		
		# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=37.5$', iv, lf6, vbg_start=-21.75, vbg_stop=-8.25,
		# 							vtg_start=17.5, vtg_stop=32.5, frames=151, repeat= repeat_n, plot=False)
		
		# exps.add_dual_gate_spectra_sweep(sample_name, '$BACK0.9TG-BG=0$', iv, lf6, vbg_start=19.7, vbg_stop=19.8,
		# 							vtg_start=21.9, vtg_stop=22, frames=2, repeat= repeat_n, plot=False)
		# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=42.5$', iv, lf6, vbg_start=-22.25, vbg_stop=-10.1,
		# 							vtg_start=22.5, vtg_stop=36, frames=136, repeat= repeat_n, plot=False)

		
		# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=40$', iv, lf6, vbg_start=-22, vbg_stop=-10.3,
		#                             vtg_start=20, vtg_stop=33, frames=131, repeat= repeat_n, plot=False)
		
		# exps.add_dual_gate_spectra_sweep(sample_name, '$BACK0.9TG-BG=0$', iv, lf6, vbg_start=19.7, vbg_stop=19.8,
		#                             vtg_start=21.9, vtg_stop=22, frames=2, repeat= repeat_n, plot=False)
		
		# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=45$', iv, lf6, vbg_start=-22.5, vbg_stop=-12.6,
		# 							vtg_start=25, vtg_stop=36, frames=111, repeat= repeat_n, plot=False)
	]),

]

# === Fixed parameters ===
exp_time = '200'
frame_to_combine = '10'
bfield = 0
angles = [98.7]  # 
lf6.change_expose_time(exp_time)
lf6.change_frame_to_combine(frame_to_combine)

# === Run all measurements ===
for center_wavelength, sweep_group in sweeps_all:
    lf6.change_spectra_center(center_wavelength)
    for i, sweep in enumerate(sweep_group):
        angle_order = angles if i % 2 == 0 else angles[::-1]  # alternate order
        for angle in angle_order:
            ep300.set_position(1, angle)
            sample_name = f'$YZD315$~$4KREF{center_wavelength}nm$~$p2$~${bfield}T{angle}deg$~${exp_time}msx{frame_to_combine}HG$~'
            exps = spectral_experiments.SpectralExperimentsCollections()
            exps.add_dual_gate_spectra_sweep(
                sample_name,
                sweep.label,
                iv,
                lf6,
                vbg_start=sweep.vbg_start,
                vbg_stop=sweep.vbg_stop,
                vtg_start=sweep.vtg_start,
                vtg_stop=sweep.vtg_stop,
                frames=sweep.frames,
                repeat=sweep.repeat_n,
                plot=False
            )
            exps.execute(repeat=1)


In [ ]:
# 720nm center
repeat_n = 1
frame_to_combine = '10'
exp_time = '500'

lf6.change_expose_time(exp_time)
lf6.change_frame_to_combine(frame_to_combine)

lf6.change_spectra_center('720')
sample_name=f'$DXD40$~$4KREF720nm$~$p3n1$~$0T$~${exp_time}msx{frame_to_combine}x{repeat_n}HG$~'   
exps = spectral_experiments.SpectralExperimentsCollections()

# exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly$', iv, lf6, vbg_start= 0, vbg_stop= 0,
#                                          vtg_start= -5, vtg_stop= 2, frames= 71, repeat= repeat_n, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly$', iv, lf6, vbg_start= -5, vbg_stop= 2,
#                                          vtg_start= 0, vtg_stop= 0, frames= 71, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0rev$', iv, lf6, vbg_start= 5, vbg_stop= -5,
                                         vtg_start= 5, vtg_stop= -5, frames= 101, repeat= repeat_n, plot=False)	

exps.execute(repeat=1)

In [ ]:
# Assume TG/BG has same efficency 
tg_voltages = np.linspace(-5, 5, 101)
bg_voltages = np.linspace(-5, 5, 101)
delay = 5
frame_to_combine = '10'
exp_time = '500'
lf6.change_expose_time(exp_time)
lf6.change_frame_to_combine(frame_to_combine)
lf6.change_spectra_center('720')

f_name = f'$DXD40$~$4KREF720nm$~$p3$~$0T$~${exp_time}msx{frame_to_combine}HGdelay={delay}s$~$TG-BG=0$'  
new_file_name = check_file_name(f_name)
output_file_name = new_file_name + '.csv'

with open(output_file_name, 'a') as f:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['Vbg','Vtg','Vtg+Vbg','Vtg-Vbg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f, cols, fmt='%s', delimiter=',')
            for vtg, vbg in zip(tg_voltages,bg_voltages):
                Doping = vtg+vbg
                Efield = vtg-vbg
                iv.x_goto('Vbg', vbg, 0.03, 0.1)
                iv.x_goto('Vtg', vtg, 0.03, 0.1)       
                time.sleep(delay)
                spectra = lf6.acquire()
                data = np.concatenate((np.array([vbg,vtg,Doping,Efield], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                np.savetxt(f, data, fmt='%.5e', delimiter=',') 
            iv.x_goto('Vbg', 0, 0.03, 0.1)
            iv.x_goto('Vtg', 0, 0.03, 0.1) 

In [ ]:
# Doping-dependent REF under a series of E-field
repeat_n = 4
frame_to_combine = '4'
exp_time = '500'

lf6.change_expose_time(exp_time)
lf6.change_frame_to_combine(frame_to_combine)


lf6.change_spectra_center('630')
sample_name=f'$YZD241$~$4KREF630nm$~$p4n0$~$0T$~${exp_time}msx{frame_to_combine}x{repeat_n}_HG$~'   
exps = spectral_experiments.SpectralExperimentsCollections()

exps.add_dual_gate_spectra_sweep(sample_name, '$bg0.9TG-BG=0$', iv, lf6, vbg_start= 9, vbg_stop= 11.7,
                                         vtg_start= 10, vtg_stop= 13, frames= 31, repeat= repeat_n, plot=False)	

# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=1$', iv, lf6, vbg_start= -9.1, vbg_stop= 8.9,
#                                          vtg_start= -9, vtg_stop= 11, frames= 201, repeat= repeat_n, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=-1$', iv, lf6, vbg_start= -8, vbg_stop= 10,
#                                          vtg_start= -10, vtg_stop= 10, frames= 201, repeat= repeat_n, plot=False)	
exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=2$', iv, lf6, vbg_start= -9.2, vbg_stop= 7.9,
                                         vtg_start= -8, vtg_stop= 11, frames= 191, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=-2$', iv, lf6, vbg_start= -7, vbg_stop= 10.1,
                                         vtg_start= -10, vtg_stop= 9, frames= 191, repeat= repeat_n, plot=False)				

exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=4$', iv, lf6, vbg_start= -10.3, vbg_stop= 6.8,
                                         vtg_start= -7, vtg_stop= 12, frames= 191, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=-4$', iv, lf6, vbg_start= -5.9, vbg_stop= 11.2,
                                         vtg_start= -11, vtg_stop= 8, frames= 191, repeat= repeat_n, plot=False)

exps.add_dual_gate_spectra_sweep(sample_name, '$bg0.9TG-BG=0$', iv, lf6, vbg_start= 9, vbg_stop= 11.7,
                                         vtg_start= 10, vtg_stop= 13, frames= 31, repeat= repeat_n, plot=False)	
exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=6$', iv, lf6, vbg_start= -11.4, vbg_stop= 5.7,
                                         vtg_start= -6, vtg_stop= 13, frames= 191, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=-6$', iv, lf6, vbg_start= -4.8, vbg_stop= 12.3,
                                         vtg_start= -12, vtg_stop= 7, frames= 191, repeat= repeat_n, plot=False)

exps.add_dual_gate_spectra_sweep(sample_name, '$bg0.9TG-BG=0$', iv, lf6, vbg_start= 9, vbg_stop= 11.7,
                                         vtg_start= 10, vtg_stop= 13, frames= 31, repeat= repeat_n, plot=False)	
exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=8$', iv, lf6, vbg_start= -12.5, vbg_stop= 4.6,
                                         vtg_start= -5, vtg_stop= 14, frames= 191, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=-8$', iv, lf6, vbg_start= -3.7, vbg_stop= 13.4,
                                         vtg_start= -13, vtg_stop= 6, frames= 191, repeat= repeat_n, plot=False)

exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=10$', iv, lf6, vbg_start= -13.6, vbg_stop= 2.6,
                                         vtg_start= -4, vtg_stop= 14, frames= 181, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=-10$', iv, lf6, vbg_start= -1.7, vbg_stop= 13.6,
                                         vtg_start= -13, vtg_stop= 4, frames= 171, repeat= repeat_n, plot=False)

exps.add_dual_gate_spectra_sweep(sample_name, '$bg0.9TG-BG=0$', iv, lf6, vbg_start= 9, vbg_stop= 11.7,
                                         vtg_start= 10, vtg_stop= 13, frames= 31, repeat= repeat_n, plot=False)	
exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=12$', iv, lf6, vbg_start= -12.9, vbg_stop= 0.6,
                                         vtg_start= -1, vtg_stop= 14, frames= 151, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=-12$', iv, lf6, vbg_start= 0.7, vbg_stop= 13.8,
                                         vtg_start= -13, vtg_stop= 2, frames= 151, repeat= repeat_n, plot=False)

exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=14$', iv, lf6, vbg_start= -13.1, vbg_stop= -1.4,
                                         vtg_start= 1, vtg_stop= 14, frames= 131, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=-14$', iv, lf6, vbg_start= 2.3, vbg_stop= 14,
                                         vtg_start= -13, vtg_stop= 0, frames= 131, repeat= repeat_n, plot=False)

exps.add_dual_gate_spectra_sweep(sample_name, '$bg0.9TG-BG=0$', iv, lf6, vbg_start= 9, vbg_stop= 11.7,
                                         vtg_start= 10, vtg_stop= 13, frames= 31, repeat= repeat_n, plot=False)	
exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=16$', iv, lf6, vbg_start= -12.4, vbg_stop= -3.4,
                                         vtg_start= 4, vtg_stop= 14, frames= 101, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=-16$', iv, lf6, vbg_start= 4.3, vbg_stop= 13.3,
                                         vtg_start= -13, vtg_stop= -3, frames= 101, repeat= repeat_n, plot=False)

exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=18$', iv, lf6, vbg_start= -13.5, vbg_stop= -5.4,
                                         vtg_start= 5, vtg_stop= 14, frames= 91, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=-18$', iv, lf6, vbg_start= 6.3, vbg_stop= 13.5,
                                         vtg_start= -13, vtg_stop= -5, frames= 81, repeat= repeat_n, plot=False)

exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=20$', iv, lf6, vbg_start= -12.8, vbg_stop= -7.4,
                                         vtg_start= 8, vtg_stop= 14, frames= 61, repeat= repeat_n, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=-20$', iv, lf6, vbg_start= 8.3, vbg_stop= 13.7,
                                         vtg_start= -13, vtg_stop= -7, frames= 61, repeat= repeat_n, plot=False)

exps.add_dual_gate_spectra_sweep(sample_name, '$bg0.9TG-BG=0$', iv, lf6, vbg_start= 9, vbg_stop= 11.7,
                                         vtg_start= 10, vtg_stop= 13, frames= 31, repeat= repeat_n, plot=False)	
	 


exps.execute(repeat=1)


In [ ]:
frame_to_combine = '4'
exp_time = '750'
for center in ['650']:
    lf6.change_expose_time(exp_time)
    lf6.change_spectra_center(center)
    lf6.change_frame_to_combine(frame_to_combine)
    # meas_power =  c_double()
    # tlPM.measPower(byref(meas_power),1)
    # print(meas_power.value*1e6)#unit uW
    # power = np.round(meas_power.value*1e6,3)
    
    # sample_name=f'$YZD241$~$4KREF720nm$~$p1n1$~$0T$~${power}uw$~${exp_time}sx{frame_to_combine}_HG$~'
    sample_name=f'$YZD241$~$4KREF650nm$~$p1n1$~$0T$~${exp_time}sx{frame_to_combine}_HG$~'
    
    exps = spectral_experiments.SpectralExperimentsCollections()
    exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=0$', iv, lf6, vbg_start= -8.1, vbg_stop= 8.1,
                                         vtg_start= -9, vtg_stop= 9, frames= 91, repeat=1, plot=False)

    exps.execute(repeat=1)


In [ ]:
frame_to_combine = '1'
exp_time = '10000'
for center in ['890']:
    lf6.change_expose_time(exp_time)
    lf6.change_spectra_center(center)
    lf6.change_frame_to_combine(frame_to_combine)
    meas_power =  c_double()
    tlPM.measPower(byref(meas_power),1)
    print(meas_power.value*1e6)#unit uW
    power = np.round(meas_power.value*1e6,3)
    
    sample_name=f'$YZD72$~$4KPL730nm$~$p11nX2$~$0T$~${power}uw$~${exp_time}sx{frame_to_combine}_HG$~'
    
    exps = spectral_experiments.SpectralExperimentsCollections()
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-6.0$', iv, lf6, vbg_start=-6, vbg_stop=0.0,
                                         vtg_start=0.0, vtg_stop=-6, frames=121, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-5.5$', iv, lf6, vbg_start=-6, vbg_stop=0.5,
											vtg_start=0.5, vtg_stop=-6, frames=131, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-5.0$', iv, lf6, vbg_start=-6, vbg_stop=1.0,
											vtg_start=1.0, vtg_stop=-6, frames=141, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-4.5$', iv, lf6, vbg_start=-6, vbg_stop=1.5,
											vtg_start=1.5, vtg_stop=-6, frames=151, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-4.0$', iv, lf6, vbg_start=-6, vbg_stop=2.0,
											vtg_start=2.0, vtg_stop=-6, frames=161, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-3.5$', iv, lf6, vbg_start=-6, vbg_stop=2.5,
											vtg_start=2.5, vtg_stop=-6, frames=171, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-3.0$', iv, lf6, vbg_start=-6, vbg_stop=3.0,
											vtg_start=3.0, vtg_stop=-6, frames=181, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-2.5$', iv, lf6, vbg_start=-6, vbg_stop=3.5,
											vtg_start=3.5, vtg_stop=-6, frames=191, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-2.0$', iv, lf6, vbg_start=-6, vbg_stop=4.0,
											vtg_start=4.0, vtg_stop=-6, frames=201, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-1.5$', iv, lf6, vbg_start=-6, vbg_stop=4.5,
											vtg_start=4.5, vtg_stop=-6, frames=211, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-1.0$', iv, lf6, vbg_start=-6, vbg_stop=5.0,
											vtg_start=5.0, vtg_stop=-6, frames=221, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-0.5$', iv, lf6, vbg_start=-6, vbg_stop=5.5,
											vtg_start=5.5, vtg_stop=-6, frames=231, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0.0$', iv, lf6, vbg_start=-6, vbg_stop=6.0,
											vtg_start=6.0, vtg_stop=-6, frames=241, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0.5$', iv, lf6, vbg_start=6, vbg_stop=-5.5,
											vtg_start=-5.5, vtg_stop=6, frames=231, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=1.0$', iv, lf6, vbg_start=6, vbg_stop=-5.0,
											vtg_start=-5.0, vtg_stop=6, frames=221, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=1.5$', iv, lf6, vbg_start=6, vbg_stop=-4.5,
											vtg_start=-4.5, vtg_stop=6, frames=211, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=2.0$', iv, lf6, vbg_start=6, vbg_stop=-4.0,
											vtg_start=-4.0, vtg_stop=6, frames=201, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=2.5$', iv, lf6, vbg_start=6, vbg_stop=-3.5,
											vtg_start=-3.5, vtg_stop=6, frames=191, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=3.0$', iv, lf6, vbg_start=6, vbg_stop=-3.0,
											vtg_start=-3.0, vtg_stop=6, frames=181, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=3.5$', iv, lf6, vbg_start=6, vbg_stop=-2.5,
											vtg_start=-2.5, vtg_stop=6, frames=171, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=4.0$', iv, lf6, vbg_start=6, vbg_stop=-2.0,
											vtg_start=-2.0, vtg_stop=6, frames=161, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=4.5$', iv, lf6, vbg_start=6, vbg_stop=-1.5,
											vtg_start=-1.5, vtg_stop=6, frames=151, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=5.0$', iv, lf6, vbg_start=6, vbg_stop=-1.0,
											vtg_start=-1.0, vtg_stop=6, frames=141, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=5.5$', iv, lf6, vbg_start=6, vbg_stop=-0.5,
											vtg_start=-0.5, vtg_stop=6, frames=131, repeat=1, plot=False)
	exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=6.0$', iv, lf6, vbg_start=6, vbg_stop=0.0,
											vtg_start=0.0, vtg_stop=6, frames=121, repeat=1, plot=False)

    exps.execute(repeat=1)

In [ ]:
##PL spectra power dep
power_angles = np.linspace(100,3600,36)
# power_angles = [500., 3600.]
vbg=-0.7
vtg=0.7
iv.x_goto('Vbg', vbg, 0.1, 0.05)
iv.x_goto('Vtg', vtg, 0.1, 0.05)
iv.report_status()

power_efficient = 1
# power_angles = [72.0, 71.55, 71.1]
file_name = f'$YZD72-4K$~$PLcw730nm5cmL$~$p11X2$~$Power5s$~$TG={vtg}Bg={vbg}V_001$'
lf6.change_expose_time('10000')

with open('{}.csv'.format(file_name), 'a') as f:
    wls = lf6.get_wavelength_calibration()
    cols = np.concatenate((np.array(['Vbg','Vtg','power'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
    np.savetxt(f, cols, fmt='%s', delimiter=',')

    for power_angle in power_angles:    
        pump_stage.move_to(power_angle)
        meas_power =  c_double()
        tlPM.measPower(byref(meas_power),1)
        input_power = round(meas_power.value*1e6,5)*power_efficient
        spectra = lf6.acquire()
        data = np.concatenate((np.array([vbg, vtg, input_power], ndmin=1, dtype=np.float64), *-spectra)).reshape([1, -1])
        np.savetxt(f, data, fmt='%.5e', delimiter=',')
        
winsound.Beep(frequency=750, duration=300)  # frequency in Hz, duration in milliseconds
        

In [ ]:
# K K' 29. 119 quarter angle
quarter_angles = [29]
frame_to_combine = '4'
bfield = 0

# center = '650'
# lf6.change_spectra_center(center)
# lf6.change_expose_time('500')
# lf6.change_frame_to_combine(frame_to_combine)
# for _ in range(25):
# 	for quarter_angle in quarter_angles:
# 		ep300.set_position(1, quarter_angle)

# 		sample_name=f'$YZD197$~$4KREFqt={quarter_angle}deg{center}nm$~$pE$~${bfield}T$~$0.5sx{frame_to_combine}_HG$~'
# 			# sample_name=f'$YZD267$~$4KPL{center}nm$~$p1$~$9T_100uW$~$1s_HG$~'
# 		exps = spectral_experiments.SpectralExperimentsCollections()
# 		exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly$', iv, lf6, vbg_start=-1, vbg_stop=1,
# 											vtg_start=0, vtg_stop=0, frames=21, repeat=1, plot=False)
# 		# exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly$', iv, lf6, vbg_start=-2.5, vbg_stop=2.5,
# 		# 									vtg_start=0, vtg_stop=0, frames=51, repeat=2, plot=False)
# 		exps.execute(repeat=1)

center = '730'
lf6.change_spectra_center(center)
lf6.change_expose_time('500')
lf6.change_frame_to_combine(frame_to_combine)
for _ in range(100):

	for quarter_angle in quarter_angles:
		ep300.set_position(1, quarter_angle)

		sample_name=f'$YZD197$~$4KREFqt={quarter_angle}deg{center}nm$~$pE$~${bfield}T$~$0.5sx{frame_to_combine}_HG$~'
			# sample_name=f'$YZD267$~$4KPL{center}nm$~$p1$~$9T_100uW$~$1s_HG$~'
		exps = spectral_experiments.SpectralExperimentsCollections()
		exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-5, vbg_stop=5,
									vtg_start=-5, vtg_stop=5, frames=201, repeat=1, plot=False)
		exps.execute(repeat=1)

winsound.Beep(frequency=750, duration=300)

In [ ]:
# for center in ['695','580']:
for center in ['730']:
# for center in ['680']:
    lf6.change_spectra_center(center)
    sample_name=f'$YZD261$~$4KREF{center}nm$~$pb_vbias={vbias}$~$9T$~$0.5sx4_HG$~'
    exps = spectral_experiments.SpectralExperimentsCollections()
    # exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly$', iv, lf6, vbg_start=-7, vbg_stop=7,
    #                             vtg_start=0, vtg_stop=0, frames=141, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly_TG=2V$', iv, lf6, vbg_start=-7, vbg_stop=7,
    #                             vtg_start=2, vtg_stop=2, frames=141, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly$', iv, lf6, vbg_start=0, vbg_stop=0,
    #                             vtg_start=-7, vtg_stop=7, frames=141, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly$', iv, lf6, vbg_start=0, vbg_stop=0,
    #                             vtg_start=-7, vtg_stop=7, frames=141, repeat=1, plot=False)
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-6, vbg_stop=6,
                                vtg_start=-6, vtg_stop=6, frames=241, repeat=1, plot=False)

    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=1$', iv, lf6, vbg_start=-7, vbg_stop=6,
    #                             vtg_start=-6, vtg_stop=7, frames=261, repeat=1, plot=False)
	# exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=2$', iv, lf6, vbg_start=-7, vbg_stop=5,
	# 							vtg_start=-5, vtg_stop=7, frames=241, repeat=1, plot=False)

    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=3$', iv, lf6, vbg_start=-7, vbg_stop=4,
    #                             vtg_start=-4, vtg_stop=7, frames=221, repeat=1, plot=False)
	# exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=4$', iv, lf6, vbg_start=-7, vbg_stop=3,
	# 	                        vtg_start=-3, vtg_stop=7, frames=201, repeat=1, plot=False)
	# exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-1$', iv, lf6, vbg_start=-6, vbg_stop=7,
    #                             vtg_start=-7, vtg_stop=6, frames=261, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-2$', iv, lf6, vbg_start=-5, vbg_stop=7,
    #                             vtg_start=-7, vtg_stop=5, frames=241, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-3$', iv, lf6, vbg_start=-4, vbg_stop=7,
    #                             vtg_start=-7, vtg_stop=4, frames=221, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-4$', iv, lf6, vbg_start=-3, vbg_stop=7,
    #                             vtg_start=-7, vtg_stop=3, frames=201, repeat=1, plot=False)


    exps.execute(repeat=1)
winsound.Beep(frequency=750, duration=300)

In [ ]:
# for center in ['710','580']:
for center in ['730', '685']:
# for center in ['680']:
    lf6.change_spectra_center(center)
    sample_name=f'$YZD261$~$4KREFnm{center}nm$~$p7$~$9T$~$0.5sx4_HG$~'
    exps = spectral_experiments.SpectralExperimentsCollections()
    # exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly$', iv, lf6, vbg_start=-7, vbg_stop=7,
    #                             vtg_start=0, vtg_stop=0, frames=141, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly_TG=2V$', iv, lf6, vbg_start=-7, vbg_stop=7,
    #                             vtg_start=2, vtg_stop=2, frames=141, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly$', iv, lf6, vbg_start=0, vbg_stop=0,
    #                             vtg_start=-7, vtg_stop=7, frames=141, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly$', iv, lf6, vbg_start=0, vbg_stop=0,
    #                             vtg_start=-7, vtg_stop=7, frames=141, repeat=1, plot=False)
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-5, vbg_stop=7,
                                vtg_start=-5, vtg_stop=7, frames=241, repeat=1, plot=False)

    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=1$', iv, lf6, vbg_start=-7, vbg_stop=6,
    #                             vtg_start=-6, vtg_stop=7, frames=261, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=2$', iv, lf6, vbg_start=-7, vbg_stop=5,
    #                             vtg_start=-5, vtg_stop=7, frames=241, repeat=1, plot=False)


    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=3$', iv, lf6, vbg_start=-7, vbg_stop=4,
    #                             vtg_start=-4, vtg_stop=7, frames=221, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=4$', iv, lf6, vbg_start=-7, vbg_stop=3,
	# 	                        vtg_start=-3, vtg_stop=7, frames=201, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-1$', iv, lf6, vbg_start=-6, vbg_stop=7,
    #                             vtg_start=-7, vtg_stop=6, frames=261, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-2$', iv, lf6, vbg_start=-5, vbg_stop=7,
    #                             vtg_start=-7, vtg_stop=5, frames=241, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-3$', iv, lf6, vbg_start=-4, vbg_stop=7,
    #                             vtg_start=-7, vtg_stop=4, frames=221, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-4$', iv, lf6, vbg_start=-3, vbg_stop=7,
    #                             vtg_start=-7, vtg_stop=3, frames=201, repeat=1, plot=False)


    exps.execute(repeat=1)
winsound.Beep(frequency=750, duration=300)

In [ ]:
sample_name=f'$YZD261$~$4KPL633nm$~$pb$~$9T$~$1s_HG$~'

# sample_name='$YBoD3$~$4KREF_lamp$~$9T$~$670C1sx4x4p4$~'
exps = spectral_experiments.SpectralExperimentsCollections()
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-7, vbg_stop=7,
                                vtg_start=-7, vtg_stop=7, frames=281, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0$', iv, lf6, vbg_start=-5, vbg_stop=5,
#                                 vtg_start=5, vtg_stop=-5, frames=101, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly$', iv, lf6, vbg_start=-7, vbg_stop=7,
                                vtg_start=0, vtg_stop=0, frames=141, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly$', iv, lf6, vbg_start=0, vbg_stop=0,
                                vtg_start=-7, vtg_stop=7, frames=141, repeat=1, plot=False)


# exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-2.5$', iv, lf6, vbg_start=-5, vbg_stop=5,
#                                 vtg_start=-7.5, vtg_stop=2.5, frames=101, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-5$', iv, lf6, vbg_start=-2.5, vbg_stop=5,
#                                 vtg_start=-7.5, vtg_stop=0, frames=76, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=2.5$', iv, lf6, vbg_start=-7.5, vbg_stop=2.5,
#                                 vtg_start=-5, vtg_stop=5, frames=101, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=5$', iv, lf6, vbg_start=-7.5, vbg_stop=0,
#                                 vtg_start=-2.5, vtg_stop=5, frames=76, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=7.5$', iv, lf6, vbg_start=-7.5, vbg_stop=-2.5,
#                                 vtg_start=0, vtg_stop=5, frames=51, repeat=1, plot=False)



# exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-5.1$', iv, lf6, vbg_start=-7.5, vbg_stop=2.4,
#                                 vtg_start=2.4, vtg_stop=-7.5, frames=100, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=2.8V$', iv, lf6, vbg_start=-3.2, vbg_stop=6,
#                                 vtg_start=6, vtg_stop=-3.2, frames=93, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=1.4V$', iv, lf6, vbg_start=-4.6, vbg_stop=6,
#                                 vtg_start=6, vtg_stop=-4.6, frames=107, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=4.8V$', iv, lf6, vbg_start=-5.2, vbg_stop=8,
#                             vtg_start=10, vtg_stop=-3.2, frames=133, repeat=1, plot=False)

# exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=6.0V$', iv, lf6, vbg_start=-4, vbg_stop=8,
#                             vtg_start=10, vtg_stop=-2, frames=121, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=5.5V$', iv, lf6, vbg_start=-0.5, vbg_stop=6,
#                             vtg_start=6, vtg_stop=0.5, frames=61, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=6.5V$', iv, lf6, vbg_start=0.5, vbg_stop=6,
#                             vtg_start=6, vtg_stop=0.5, frames=56, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-3.0V$', iv, lf6, vbg_start=5, vbg_stop=-7,
#                             vtg_start=-8, vtg_stop=4, frames=121, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-3.5V$', iv, lf6, vbg_start=2.5, vbg_stop=-6,
#                             vtg_start=-6, vtg_stop=2.5, frames=121, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-6.0V$', iv, lf6, vbg_start=2, vbg_stop=-7,
#                             vtg_start=-8, vtg_stop=1, frames=91, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-7.0V$', iv, lf6, vbg_start=2, vbg_stop=-7,
#                             vtg_start=-9, vtg_stop=0, frames=91, repeat=1, plot=False)


exps.execute(repeat=1)
winsound.Beep(frequency=750, duration=300)

In [ ]:
##PL spectra Vbg dep
vbgs = [0,-2.5]
vtg = 0
file_name = '$YZD197-4K9T$~$REF615nmc$~$pE$~$657nmlaser80.5uw$~$0.4sx500$'

with open('{}.csv'.format(file_name), 'a') as f:
    wls = lf6.get_wavelength_calibration()
    cols = np.concatenate((np.array(['Vbg','Vtg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
    np.savetxt(f, cols, fmt='%s', delimiter=',')
    for vbg in vbgs:
        iv.x_goto('Vbg', vbg, 0.1, 0.05)
        spectra = lf6.acquire()
        data = np.concatenate((np.array([vbg, vtg], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
        np.savetxt(f, data, fmt='%.5e', delimiter=',')
iv.x_goto('Vbg', 0, 0.1, 0.05)  
winsound.Beep(frequency=750, duration=300)  # frequency in Hz, duration in milliseconds

In [ ]:

tg = 0
bg = 0
print(f'{tg-bg}V')
iv.x_goto('Vbg', bg, 0.1, 0.1)
iv.x_goto('Vtg', tg, 0.1, 0.1)
iv.report_status()


In [ ]:
# inhalf: 46.3 laser max; laser min 1.3

In [ ]:
ep300.set_position(2,170.8-45) #
print(ep300.get_positions('2'))
# meas_power =  c_double()
# tlPM.setWavelength(730)
# tlPM.measPower(byref(meas_power))
# print(meas_power.value*1e6-0.005)#unit uW
# print(round(meas_power.value*1e6*0.75,3))#unit uW
# # 10/7

In [ ]:
stage.move_to(134.5)
# stage.move_to(610)
meas_power =  c_double()
tlPM.setWavelength(730)
tlPM.measPower(byref(meas_power))
print(meas_power.value*1e6)#unit uW

## PL Efield and Doping without valley


In [ ]:
lf6.change_spectra_center('620')

In [ ]:
# sample_name = '$YZD241$~$REF600g650nmCWL$~$p4n12$~$0.75sx4x6$~$0T$~'
sample_name=f'$YZD267$~$4KREF705nm$~$p1$~$9T$~$0.75sx4x5_HG$~'
# number = 15
# vbg = -7
# vtg =7
# for dv in np.linspace(vbg, vtg, number):
#     if dv <= 0:
#         vbgs = vbg
#         vtge = vbg
#         vbge = dv-vbgs
#         vtgs = dv-vbgs

#     else:
#         vbgs = vtg
#         vtge = vtg
#         vbge = dv-vbgs
#         vtgs = dv-vbgs
#     frs = abs(int((vtge - vtgs)/0.05 ))+1 
#     if frs == 1:
#         pass
#     else:
#         exps = spectral_experiments.SpectralExperimentsCollections()
#         exptitle = '$TG+BG={}$'.format(dv)
#         exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
#                                         vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
#         print(vbgs, vbge, vtgs, vtge, exptitle, frs)
#         # exps.execute(repeat=1)

##different doping level            
vgs = -7
vge = 7
number1 = 15
total_frs = 0
for dv in np.linspace(vgs,vge,number1):
    if dv <= 0:
        vbge = vge
        vtgs = vgs
        vbgs = vtgs - dv
        vtge = vbge + dv
    else:
        vtge = vge
        vbgs = vgs
        vbge = vtge - dv
        vtgs = vbgs + dv
    frs = int((vtge - vtgs)/0.05 +1)
    total_frs = total_frs + frs
    if frs == 1:
        pass
    else:
        exps = spectral_experiments.SpectralExperimentsCollections()
        exptitle = '$TG-BG={}$'.format(dv)
        exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                        vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=4, plot=True)
            
        # print(vbgs, vbge, vtgs, vtge, exptitle, frs)

        exps.execute(repeat=1)
# print(total_frs)

In [ ]:
import numpy as np
center_wavelengths = ['630']
total_frames = 0
for center_wavelength in center_wavelengths:
    lf6.change_spectra_center(center_wavelength)
    sample_name = '$YZD144$~$REF600g{}nmC$~$pBn5$~$2s$~$0T$~'.format(center_wavelength)
    ##different doping level            
    vgs = -5.5
    vge = 5.5
    number1 = 12
    for dv in np.linspace(vgs,vge,number1):
        if dv <= 0:
            vbge = vge
            vtgs = vgs
            vbgs = vtgs - dv
            vtge = vbge + dv
        else:
            vtge = vge
            vbgs = vgs
            vbge = vtge - dv
            vtgs = vbgs + dv
        frs = int((vtge - vtgs)/0.05 +1)
        if frs == 1 or dv<0:
            pass
        else:
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG-BG={}$'.format(dv)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=False)
                
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            total_frames += frs
            exps.execute(repeat=1)

# center_wavelengths = ['775','630']
# total_frames = 0
# for center_wavelength in center_wavelengths:
#     lf6.change_spectra_center(center_wavelength)
#     sample_name = '$YZD144$~$REF600g{}nmC$~$pBn5$~$2s$~$0T$~'.format(center_wavelength)
#     ##different doping level            
#     number = 12
#     vbg = -5.5
#     vtg =5.5
#     for dv in np.linspace(vbg, vtg, number):
#         if dv <= 0:
#             vbgs = vbg
#             vtge = vbg
#             vbge = dv-vbgs
#             vtgs = dv-vbgs

#         else:
#             vbgs = vtg
#             vtge = vtg
#             vbge = dv-vbgs
#             vtgs = dv-vbgs
#         frs = abs(int((vtge - vtgs)/0.05 ))+1 
#         if frs == 1:
#             pass
#         else:
#             exps = spectral_experiments.SpectralExperimentsCollections()
#             exptitle = '$TG+BG={}$'.format(dv)
#             exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
#                                             vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=False)
                
#             print(vbgs, vbge, vtgs, vtge, exptitle, frs)
#             exps.execute(repeat=1)


In [ ]:
print(total_frames)

In [ ]:
bgvalue = -5
tgvalue = -5

bgend = np.around(np.linspace(0, 5, 51),decimals=2)
tgstart = bgend
bgstart = np.full_like(bgend,bgvalue)
tgend = np.full_like(bgend,tgvalue)
frs = np.linspace(101,201,51)
dopingvalue = np.around(np.linspace(-5, 0, 51),decimals=2)
sample_name = '$DX156_20231224-MEGA$~$PL$~$7Kp4up$~$730nm5uW$~$3s$~'
for bgsv,bgev,tgsv,tgev,frsv,dopingv in zip(bgstart,bgend,tgstart,tgend,frs,dopingvalue):
    if frsv < 0:
        pass
    else:
        exps = spectral_experiments.SpectralExperimentsCollections()
        exptitle = '$TG+BG={}$'.format(dopingv)
        exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=bgsv, vbg_stop=bgev,
                                        vtg_start=tgsv, vtg_stop=tgev, frames=int(frsv), repeat=1, plot=False)
            
        exps.execute(repeat=1)
        print(bgsv,bgev,tgsv,tgev,exptitle,int(frsv))
print('-------------------')
bgvalue = 5
tgvalue = 5
bgend = np.around(np.linspace(0, -5, 51),decimals=2)
tgstart = bgend
bgstart = np.full_like(bgend,bgvalue)
tgend = np.full_like(bgend,tgvalue)
frs = np.linspace(101,201,51)
dopingvalue = np.around(np.linspace(5, 0, 51),decimals=2)
for bgsv,bgev,tgsv,tgev,frsv,dopingv in zip(bgstart,bgend,tgstart,tgend,frs,dopingvalue):
    if frsv < 0:
        pass
    else:
        exps = spectral_experiments.SpectralExperimentsCollections()
        exptitle = '$TG+BG={}$'.format(dopingv)
        exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=bgsv, vbg_stop=bgev,
                                        vtg_start=tgsv, vtg_stop=tgev, frames=int(frsv), repeat=1, plot=False)
            
        exps.execute(repeat=1)
        print(bgsv,bgev,tgsv,tgev,exptitle,int(frsv))

## PL with VP

In [ ]:
step = 0.1
x = ex.x_channels['x_axis'].collect_x()
y = ex.x_channels['y_axis'].collect_x()
number_of_w = 0
number_of_a = 0
while True:
    key = input()
    print(key)
    if key == "q":
        break
    elif key == "a":
        if  y+step<10:
            y += step
            ex.x_goto('y_axis', y, 0, 0)
            number_of_a += 1 
    elif key == "d":
        if y-step>0:
            y -= step 
            ex.x_goto('y_axis', y, 0, 0)
            number_of_a -= 1 
    elif key == "w":
        if x+step<10:
            x += step
            ex.x_goto('x_axis', x, 0, 0)
            number_of_w += 1
    elif key == "s":
        if x-step>0:
            x -= step
            ex.x_goto('x_axis', x, 0, 0)
            number_of_w -= 1
print('a:',number_of_a,'w:',number_of_w)

# a: -11 w:dd 0s


In [ ]:
iv.x_goto('Vbg', 0, 0.1, 0.05)
iv.x_goto('Vtg', 0, 0.1, 0.05)
iv.report_status()

In [ ]:
iv.x_goto('Vbg', 4, 0.1, 0.05)
iv.x_goto('Vtg', -4, 0.1, 0.05)
iv.report_status()

In [ ]:
pump_stage.move_to(2800)
meas_power =  c_double()
tlPM.measPower(byref(meas_power),1)
print(meas_power.value*1e6)#unit uW
power = np.round(meas_power.value*1e6,3)

In [ ]:
#5.5 p-pol, 141 s-pol
# ep300.set_position(1, 141)

# ep300.set_position(2, 42)


In [ ]:
# === Define sweep configurations ===
in_halfs  = [14.1]
# in_halfs  = [14.1]
out_halfs = [42,87]
# out_halfs = [42, 87]
# ep300.set_position(1,in_half)
stage_poss = [2800]
exp_times = ['3000']
bfield = 0
sweep_configs = [
    # {"label": "$TG-BG=1.4$", "vbg_start": -5, "vbg_stop": 3.6, "vtg_start": -3.6, "vtg_stop": 5, "frames": 173},
    # {"label": "$TG+BG=0$", "vbg_start": -6, "vbg_stop": 6, "vtg_start": 6, "vtg_stop": -6, "frames": 121},
    {"label": "$TG+BG=1.6$", "vbg_start": -4.4, "vbg_stop": 6, "vtg_start": 6, "vtg_stop": -4.4, "frames": 105},
    {"label": "$TG+BG=2.7$", "vbg_start": -3.3, "vbg_stop": 6, "vtg_start": 6, "vtg_stop": -3.3, "frames": 94},
    {"label": "$TG+BG=3.7$", "vbg_start": -3.4, "vbg_stop": 7.1, "vtg_start": 7.1, "vtg_stop": -3.4, "frames": 106},
    {"label": "$TG+BG=4.0$", "vbg_start": -4.8, "vbg_stop": 6.2, "vtg_start": 8.8, "vtg_stop": -2.2, "frames": 111},
    {"label": "$TG+BG=4.8$", "vbg_start": -5.2, "vbg_stop": 8, "vtg_start": 10, "vtg_stop": -3.2, "frames": 133},
    {"label": "$TG+BG=5.0$", "vbg_start": -5, "vbg_stop": 8, "vtg_start": 10, "vtg_stop": -3, "frames": 131},
    {"label": "$TG+BG=5.2$", "vbg_start": -4.8, "vbg_stop": 8, "vtg_start": 10, "vtg_stop": -2.8, "frames": 129},
    {"label": "$TG+BG=5.4$", "vbg_start": -4.6, "vbg_stop": 8, "vtg_start": 10, "vtg_stop": -2.6, "frames": 127},
    {"label": "$TG+BG=5.6$", "vbg_start": -4.4, "vbg_stop": 8, "vtg_start": 10, "vtg_stop": -2.4, "frames": 125},
    {"label": "$TG+BG=5.8$", "vbg_start": -4.2, "vbg_stop": 8, "vtg_start": 10, "vtg_stop": -2.2, "frames": 123},
    {"label": "$TG+BG=6.0$", "vbg_start": -4, "vbg_stop": 8, "vtg_start": 10, "vtg_stop": -2, "frames": 121},
    {"label": "$TG+BG=6.2$", "vbg_start": -3.8, "vbg_stop": 8, "vtg_start": 10, "vtg_stop": -1.8, "frames": 119},
    {"label": "$TG+BG=6.4$", "vbg_start": -3.6, "vbg_stop": 8, "vtg_start": 10, "vtg_stop": -1.6, "frames": 117},
    {"label": "$TG+BG=7.0$", "vbg_start": -3, "vbg_stop": 8, "vtg_start": 10, "vtg_stop": -1, "frames": 111},
    {"label": "$TG+BG=8.0$", "vbg_start": -2, "vbg_stop": 8, "vtg_start": 10, "vtg_stop": 0, "frames": 101},
    {"label": "$TG+BG=-3.6$", "vbg_start": -7.1, "vbg_stop": 3.5, "vtg_start": 3.5, "vtg_stop": -7.1, "frames": 107},
    {"label": "$TG+BG=-4.5$", "vbg_start": -7.1, "vbg_stop": 2.6, "vtg_start": 2.6, "vtg_stop": -7.1, "frames": 98},
    {"label": "$TG+BG=-6.0$", "vbg_start": -7.0, "vbg_stop": 1, "vtg_start": 1, "vtg_stop": -7.0, "frames": 81},
]

# === Loop over stage positions ===
for stage_pos, exp_time in zip(stage_poss, exp_times):
    lf6.change_expose_time(exp_time)
    pump_stage.move_to(stage_pos)
    time.sleep(2)

    meas_power = c_double()
    tlPM.measPower(byref(meas_power), 1)
    power = round(meas_power.value * 1e6, 3)
    print(f'current power = {power}')
    # === Loop over all sweep configs ===
    for sweep in sweep_configs:
        # === Loop over each out_half angle ===
        for in_half in in_halfs:
            ep300.set_position(1, in_half)
            for out_half in out_halfs:
                ep300.set_position(2, out_half)

                sample_name = (
					f"$YZD72$~$4KPL730nmVP$~$p11n4$~${bfield}T$~${power}uw$~"
					f"$Inhalf{in_half}degree_Outhalf{out_half}degree$~"
					f"${exp_time}msx_HG$~"
				)

                exps = spectral_experiments.SpectralExperimentsCollections()
                exps.add_dual_gate_spectra_sweep(
					sample_name,
					sweep["label"],
					iv,
					lf6,
					vbg_start=sweep["vbg_start"],
					vbg_stop=sweep["vbg_stop"],
					vtg_start=sweep["vtg_start"],
					vtg_stop=sweep["vtg_stop"],
					frames=sweep["frames"],
					repeat=1,
					plot=False
				)
                exps.execute(repeat=1)

# === Done beep ===
winsound.Beep(frequency=750, duration=300)


In [ ]:
# inhalf 41.6; good cir 181 191 68quater
# inhalf 41.6-45; bad circ 68quater 
# inhalf 41.6+45; bad circ 68quater
# inhalf 41.6+45; bad circ 158quater 82%  using 
# inhalf 41.6; bad circ 158quater 91%	using
# inhalf 41.6-45; bad circ 158quater 


ep300.set_position(1,41.6)

In [ ]:
# outhalf 43-48 using 45 n>1 signal min
# outhalf 90 n>1 signal max
ep300.set_position(2,90)

In [ ]:
iv.x_goto('Vbg', 2.3, 0.1, 0.05)
iv.x_goto('Vtg', -2.3, 0.1, 0.05)
iv.report_status()


In [ ]:
sample_name = '$YZD72-5cmL$~$4KPL730nm$~$p11n6$~$400uw$~$1s$~'
valleys = [45,90]
for valley in valleys:
    ep300.set_position(2,valley)
    exps = spectral_experiments.SpectralExperimentsCollections()
    # exptitle = '$TG-BG=0V$~$in_41.6degree_out_{}degree$'.format(valley)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0V$~$in_41.6degree_out_{}degree$'.format(valley), iv, lf6, vbg_start=-6, vbg_stop=6,
    #                             vtg_start=-6, vtg_stop=6, frames=121, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=1.4V$~$in_41.6degree_out_{}degree$'.format(valley), iv, lf6, vbg_start=-5, vbg_stop=3.6,
    #                             vtg_start=-3.6, vtg_stop=5, frames=87, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0V$~$in_41.6degree_out_{}degree$'.format(valley), iv, lf6, vbg_start=-6, vbg_stop=6,
    #                             vtg_start=6, vtg_stop=-6, frames=121, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=4.8V$~$in_41.6degree_out_{}degree$'.format(valley), iv, lf6, vbg_start=-1.2, vbg_stop=6,
    #                             vtg_start=6, vtg_stop=-1.2, frames=73, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=4.8V$~$in_41.6degree_out_{}degree$'.format(valley), iv, lf6, vbg_start=-5.2, vbg_stop=8,
    #                             vtg_start=10, vtg_stop=-3.2, frames=133, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=6.0V$~$in_41.6degree_out_{}degree$'.format(valley), iv, lf6, vbg_start=-4, vbg_stop=8,
    #                             vtg_start=10, vtg_stop=-2, frames=121, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=5.5V$~$in_41.6degree_out_{}degree$'.format(valley), iv, lf6, vbg_start=-0.5, vbg_stop=6,
    #                             vtg_start=6, vtg_stop=0.5, frames=61, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=6.5V$~$in_41.6degree_out_{}degree$'.format(valley), iv, lf6, vbg_start=0.5, vbg_stop=6,
    #                             vtg_start=6, vtg_stop=0.5, frames=56, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-3.0V$~$in_41.6degree_out_{}degree$'.format(valley), iv, lf6, vbg_start=5, vbg_stop=-7,
    #                             vtg_start=-8, vtg_stop=4, frames=121, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-3.5V$~$in_41.6degree_out_{}degree$'.format(valley), iv, lf6, vbg_start=2.5, vbg_stop=-6,
    #                             vtg_start=-6, vtg_stop=2.5, frames=121, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-6.0V$~$in_41.6degree_out_{}degree$'.format(valley), iv, lf6, vbg_start=2, vbg_stop=-7,
    #                             vtg_start=-8, vtg_stop=1, frames=91, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-7.0V$~$in_41.6degree_out_{}degree$'.format(valley), iv, lf6, vbg_start=2, vbg_stop=-7,
    #                             vtg_start=-9, vtg_stop=0, frames=91, repeat=1, plot=False)
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=2.8V$~$in_41.6degree_out_{}degree$'.format(valley), iv, lf6, vbg_start=-6.2, vbg_stop=7,
                                vtg_start=9, vtg_stop=-4.2, frames=133, repeat=1, plot=False)
            
    # print(vbgs, vbge, vtgs, vtge, exptitle, frs)
    exps.execute(repeat=1)
winsound.Beep(frequency=750, duration=300)  # frequency in Hz, duration in milliseconds

In [ ]:
stage.move_to(100)
# stage.move_to(610)
meas_power =  c_double()
tlPM.setWavelength(670)
tlPM.measPower(byref(meas_power))
print(meas_power.value*1e6)#unit uW

In [ ]:
import numpy as np
# ep300.set_position(1,41.6)
## fix Doping TG+BG
lf6.change_expose_time('1000')
number = 51
sample_name = '$YZD72$~$PL730nm$~$4Kp11n3_a4w3$~$400uW$~$1s$~'
vbg = -5
vtg =5
valleys = [45,90]
total_frs = 0
for dv in np.linspace(vbg, vtg, number):
    if dv <= 0:
        vbgs = vbg
        vtge = vbg
        vbge = dv-vbgs
        vtgs = dv-vbgs

    else:
        vbgs = vtg
        vtge = vtg
        vbge = dv-vbgs
        vtgs = dv-vbgs
    vbge = np.round(vbge,3)
    vtgs = np.round(vtgs,3)
    frs = int(np.ceil(abs((vtge - vtgs)/0.05)))+1
	
    # frs = abs(int(np.ceil((vtge - vtgs)/0.05)))+1 
    if dv >= -1:
        pass
    else:
        for valley in valleys:
            ep300.set_position(2,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG+BG={}$~${}degree$'.format(np.round(dv,3),valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=False)
            total_frs +=frs
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
print(f'total_frs={total_frs}s')
## fix Efield TG-BG
# vgs = -5
# vge =5
# for dv in np.linspace(vgs,vge,number):
#     if dv <= 0:
#         vbge = vge
#         vtgs = vgs
#         vbgs = vtgs - dv
#         vtge = vbge + dv
#     else:
#         vtge = vge
#         vbgs = vgs
#         vbge = vtge - dv
#         vtgs = vbgs + dv
#     frs = int((vtge - vtgs)/0.05 +1)
#     if frs == 1:
#         pass
#     else:
#         for valley in valleys:
#             ep300.set_position(1,valley)
#             exps = spectral_experiments.SpectralExperimentsCollections()
#             exptitle = '$TG-BG={}$~${}degree$'.format(dv,valley)
#             exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
#                                             vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
#             print(vbgs, vbge, vtgs, vtge, exptitle, frs)
#             exps.execute(repeat=1)
# for valley in valleys:
#     ep300.set_position(1,valley)
#     exps = spectral_experiments.SpectralExperimentsCollections()
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=1.6$~${}degree$'.format(valley), iv, lf6, vbg_start=-5, vbg_stop=3.4,
#                                         vtg_start=-3.4, vtg_stop=5, frames=169, repeat=1, plot=False)
#     exps.execute(repeat=1)




In [ ]:
import numpy as np
ep300.set_position(2,99.5)
lf6.change_expose_time(2000)
number = 11
sample_name = '$YZD73$~$PL730nm$~$7Kpb$~$50uW$~$2s$~'
vbg = -5
vtg =5
valleys = [18,63]
for dv in np.linspace(vbg, vtg, number):
    if dv <= 0:
        vbgs = vbg
        vtge = vbg
        vbge = dv-vbgs
        vtgs = dv-vbgs

    else:
        vbgs = vtg
        vtge = vtg
        vbge = dv-vbgs
        vtgs = dv-vbgs
    frs = abs(int((vtge - vtgs)/0.05 ))+1 
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG+BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
for valley in valleys:
    ep300.set_position(1,valley)
    exps = spectral_experiments.SpectralExperimentsCollections()
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=6$~${}degree$'.format(valley), iv, lf6, vbg_start=1, vbg_stop=5,
                                        vtg_start=5, vtg_stop=1, frames=81, repeat=1, plot=False) 
    exps.execute(repeat=1)
vgs = -5
vge =5
for dv in np.linspace(vgs,vge,number):
    if dv <= 0:
        vbge = vge
        vtgs = vgs
        vbgs = vtgs - dv
        vtge = vbge + dv
    else:
        vtge = vge
        vbgs = vgs
        vbge = vtge - dv
        vtgs = vbgs + dv
    frs = int((vtge - vtgs)/0.05 +1)
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG-BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
for valley in valleys:
    ep300.set_position(1,valley)
    exps = spectral_experiments.SpectralExperimentsCollections()
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=1.6$~${}degree$'.format(valley), iv, lf6, vbg_start=-5, vbg_stop=3.4,
                                        vtg_start=-3.4, vtg_stop=5, frames=169, repeat=1, plot=False)
    exps.execute(repeat=1)
# 
ep300.set_position(2,110.9)
lf6.change_expose_time(6000)
number = 11
sample_name = '$YZD73$~$PL730nm$~$7Kpb$~$2.5uW$~$6s$~'
vbg = -5
vtg =5
valleys = [18,63]
for dv in np.linspace(vbg, vtg, number):
    if dv <= 0:
        vbgs = vbg
        vtge = vbg
        vbge = dv-vbgs
        vtgs = dv-vbgs

    else:
        vbgs = vtg
        vtge = vtg
        vbge = dv-vbgs
        vtgs = dv-vbgs
    frs = abs(int((vtge - vtgs)/0.05 ))+1 
    if frs == 1:
        pass
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG+BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
for valley in valleys:
    ep300.set_position(1,valley)
    exps = spectral_experiments.SpectralExperimentsCollections()
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=6$~${}degree$'.format(valley), iv, lf6, vbg_start=1, vbg_stop=5,
                                        vtg_start=5, vtg_stop=1, frames=81, repeat=1, plot=False) 
    exps.execute(repeat=1)
vgs = -5
vge =5
for dv in np.linspace(vgs,vge,number):
    if dv <= 0:
        vbge = vge
        vtgs = vgs
        vbgs = vtgs - dv
        vtge = vbge + dv
    else:
        vtge = vge
        vbgs = vgs
        vbge = vtge - dv
        vtgs = vbgs + dv
    frs = int((vtge - vtgs)/0.05 +1)
    if frs == 1:
        pass;
    else:
        for valley in valleys:
            ep300.set_position(1,valley)
            exps = spectral_experiments.SpectralExperimentsCollections()
            exptitle = '$TG-BG={}$~${}degree$'.format(dv,valley)
            exps.add_dual_gate_spectra_sweep(sample_name, exptitle, iv, lf6, vbg_start=vbgs, vbg_stop=vbge,
                                            vtg_start=vtgs, vtg_stop=vtge, frames=frs, repeat=1, plot=True)
            
            print(vbgs, vbge, vtgs, vtge, exptitle, frs)
            exps.execute(repeat=1)
for valley in valleys:
    ep300.set_position(1,valley)
    exps = spectral_experiments.SpectralExperimentsCollections()
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=1.6$~${}degree$'.format(valley), iv, lf6, vbg_start=-5, vbg_stop=3.4,
                                        vtg_start=-3.4, vtg_stop=5, frames=169, repeat=1, plot=False)
    exps.execute(repeat=1)


## Step and Glue

In [ ]:
centers_leftedge = np.array([810,837,864,891,918,945,972])#wav 815nm to 995nm

centers = centers_leftedge+15
sample_name = '$YZD154$~$PL$~$10Kp51+1+1$~$730nm50uW1200g$~$4s$~'
for center in centers:
            lf6.change_spectra_center(int(center))
            exps = spectral_experiments.SpectralExperimentsCollections()
#             exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0$~${}center$'.format(center), iv, lf6, vbg_start=-5, vbg_stop=5,
#                                    vtg_start=5, vtg_stop=-5, frames=201, repeat=1, plot=False)
#             exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$~${}center$'.format(center), iv, lf6, vbg_start=-5, vbg_stop=5,
#                                    vtg_start=-5, vtg_stop=5, frames=201, repeat=1, plot=False)
            exps.execute(repeat=1)
#             print(center)

## Power dep


In [ ]:
ep300.set_position(2,115.05)

#max 78 344uW;88.5 300uw;98.15 200uw;106.7 100uw;111.8 50uW;115.2 25uw;118.15 10uw;119.6 5uw; 120.6 2.5uW;121.5 1uw;123 0.02UW 
# 0.8/0.5

In [ ]:
ep300.set_position(1,3)
print(ep300.get_positions('1'))
meas_power =  c_double()
tlPM.setWavelength(730)
tlPM.measPower(byref(meas_power))
print(meas_power.value*1e6)#unit uW
print(round(meas_power.value*1e6*0.75,3))#unit uW
# 10/7

In [ ]:
import numpy as np
power_angles = [33.5, 36]
diff_times = ['50000','40000']
powers = [0.1, 0.24]
center = '840'
lf6.change_spectra_center(center)

power_efficient = 1 #WL on

for power_angle, diff_time,power in zip(power_angles,diff_times,powers):
    ep300.set_position(1,power_angle)
    lf6.change_expose_time(diff_time)
    lf6.change_roi_FullSensor()
    meas_power =  c_double()
    tlPM.setWavelength(730)
    tlPM.measPower(byref(meas_power))
    power = round(meas_power.value*1e6*power_efficient,3)
     
    sample_name = '$YZD200-50cmL$~$4KDiff880nmc$~$p6$~${}uw$~${}s$~'.format(power,round(float(diff_time)/1000,1))
    exps = spectral_experiments.SpectralExperimentsCollections()
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=5.5, vbg_stop=-5.5,
                                  vtg_start=5.5, vtg_stop=-5.5, frames=441, repeat=1, plot=False)
    exps.execute(repeat=1)


    # quick PL spectra
    lf6.change_roi_LineSensor()
    lf6.change_expose_time('3000')
    ep300.set_position(1,33.5)
    meas_power =  c_double()
    tlPM.setWavelength(730)
    tlPM.measPower(byref(meas_power))
    power = round(meas_power.value*1e6*power_efficient,3)
    sample_name = '$YZD200-50cmL$~$4KPL730nm$~$p6$~${}uw$~${}s$~'.format(power,3)
    exps = spectral_experiments.SpectralExperimentsCollections()
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-5.5, vbg_stop=5.5,
                                  vtg_start=-5.5, vtg_stop=5.5, frames=111, repeat=1, plot=False)
    
    exps.execute(repeat=1)

# power_angles = [38.3]
# powers = [0.5]

# diff_times = ['35000']
# for power_angle, diff_time,power in zip(power_angles,diff_times,powers):
#     ep300.set_position(1,power_angle)
#     lf6.change_expose_time(diff_time)

#     # meas_power =  c_double()
#     # tlPM.setWavelength(730)
#     # tlPM.measPower(byref(meas_power))
#     # power = round(meas_power.value*1e6*power_efficient,3)
#     # sample_name = '$YZD215-50cmL$~$4KPL730nmWLon$~$p7up$~$0.1uw$~$1s$~$0T$~'

#     sample_name = '$YZD200-50cmL$~$4KDiff880nmc$~$p6$~${}uw$~${}s$~'.format(power,round(float(diff_time)/1000,1))
#     exps = spectral_experiments.SpectralExperimentsCollections()
#     # exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly$', iv, lf6, vbg_start=0, vbg_stop=0,
#     #                                vtg_start=-8, vtg_stop=8, frames=321, repeat=1, plot=False)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-6, vbg_stop=6,
#                                   vtg_start=-6, vtg_stop=6, frames=241, repeat=1, plot=False)
    
#     exps.execute(repeat=1)

In [ ]:
ep300.set_position(1,3) #
print(ep300.get_positions('1'))
# meas_power =  c_double()
# tlPM.setWavelength(730)
# tlPM.measPower(byref(meas_power))
# print(meas_power.value*1e6-0.005)#unit uW
# print(round(meas_power.value*1e6*0.75,3))#unit uW
# # 10/7

In [ ]:
# lf6.change_spectra_center('880')
# lf6.change_expose_time('70000')
# lf6.change_roi_FullSensor()
# ep300.set_position(1,5.5)
# print(ep300.get_positions('1'))
# meas_power =  c_double()
# tlPM.setWavelength(730)
# tlPM.measPower(byref(meas_power))
# print(meas_power.value*1e6)#unit uW
# power = round(meas_power.value*1e6*1-0.005,4)#unit uW
# sample_name = f'$YZD200$~$Diff730nmCW 880nmc$~$4Kp6n1$~${power} WLoff$~$70s$~'
# exps = spectral_experiments.SpectralExperimentsCollections()
# # exps.add_dual_gate_spectra_sweep(sample_name, '$check n=0$', iv, lf6, vbg_start=0, vbg_stop=0,
# #                                 vtg_start=0, vtg_stop=0, frames=1, repeat=1, plot=False)
# # exps.add_dual_gate_spectra_sweep(sample_name, '$check n=1$', iv, lf6, vbg_start=1.6, vbg_stop=1.6,
# #                                 vtg_start=1.6, vtg_stop=1.6, frames=1, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-3, vbg_stop=3,
#                                          vtg_start=-3, vtg_stop=3, frames=481, repeat=1, plot=False)

# exps.execute(repeat=1)



lf6.change_expose_time('40000')
lf6.change_roi_FullSensor()
ep300.set_position(1,10.5)
# print(ep300.get_positions('1'))
# meas_power =  c_double()
# tlPM.setWavelength(730)
# tlPM.measPower(byref(meas_power))
print(meas_power.value*1e6)#unit uW
power = round(meas_power.value*1e6*1-0.005,4)#unit uW
sample_name = f'$YZD200$~$Diff730nmCW 880nmc$~$4Kp6n1$~$100nw (10.5)WLoff$~$30s$~'
exps = spectral_experiments.SpectralExperimentsCollections()
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-1, vbg_stop=1,
                                         vtg_start=-1, vtg_stop=1, frames=41, repeat=1, plot=False)

exps.execute(repeat=1)


# lf6.change_expose_time('60000')
# lf6.change_roi_FullSensor()
# ep300.set_position(1,9.5)
# print(ep300.get_positions('1'))
# meas_power =  c_double()
# tlPM.setWavelength(730)
# tlPM.measPower(byref(meas_power))
# print(meas_power.value*1e6)#unit uW
# power = round(meas_power.value*1e6*1-0.005,4)#unit uW
# sample_name = f'$YZD200$~$Diff730nmCW 880nmc$~$4Kp6n1$~${power} WLoff$~$60s$~'
# exps = spectral_experiments.SpectralExperimentsCollections()
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-3, vbg_stop=3,
#                                          vtg_start=-3, vtg_stop=3, frames=241, repeat=1, plot=False)

# exps.execute(repeat=1)

In [ ]:
lf6.change_spectra_center('860')
lf6.change_expose_time('60000')
lf6.change_roi_FullSensor()
sample_name = '$YZD112$~$Diff730nmCW$~$4KpD5w$~$0.048uw WLoff$~$60s$~'
exps = spectral_experiments.SpectralExperimentsCollections()
# exps.add_dual_gate_spectra_sweep(sample_name, '$check n=0$', iv, lf6, vbg_start=0, vbg_stop=0,
#                                 vtg_start=0, vtg_stop=0, frames=1, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$check n=1$', iv, lf6, vbg_start=1.6, vbg_stop=1.6,
#                                 vtg_start=1.6, vtg_stop=1.6, frames=1, repeat=1, plot=False)
exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-3, vbg_stop=3,
                                         vtg_start=-3, vtg_stop=3, frames=241, repeat=1, plot=False)
exps.execute(repeat=1)

In [ ]:
import numpy as np
#max 56 358uW; 67.6 300uw;76.7 200uw;85 100uw;90 50uW;93.3 25uw;96.25 10uw;97.75 5uw;98.75 2.5uW;100.1 1uw;100.9 0.66UW 
power_angles = [100.1,90]
powers = [1,10,25]
diff_times = [120000, 30000,15000]
gating_times = [120000, 30000,15000]
center = 860
for power_angle, power, diff_time, gating_time in zip(power_angles,powers,diff_times,gating_times):
    print(power_angle, power, diff_time, gating_time)
#     start = 2
#     frame = 121
    if power > 40:
        start = -4
        frame = 201
    else:
        start = -6
        frame =241
    ep300.set_position(2,power_angle)
    lf6.change_spectra_center(center)
    lf6.change_expose_time(gating_time)
    lf6.change_roi_FullSensor()
    sample_name = '$YZD139_20231229$~$Diff860nmc$~$7Kp3$~${}uw$~${}ms$~'.format(power,gating_time)
    exps = spectral_experiments.SpectralExperimentsCollections()
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=start, vbg_stop=3,
#                                   vtg_start=start, vtg_stop=3, frames=frame, repeat=1, plot=False)
    exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly$', iv, lf6, vbg_start=start, vbg_stop=6,
                                  vtg_start=0, vtg_stop=0, frames=frame, repeat=1, plot=False)
    exps.execute(repeat=1)
    
    if power > 40:
        pass
    else:
        lf6.change_spectra_center(0)
        lf6.change_expose_time(diff_time)
        lf6.change_roi_FullSensor()
        sample_name = '$YZD139_20231229$~$Diff0nm$~$7Kp3$~${}uw$~${}ms$~'.format(power,diff_time)
        exps = spectral_experiments.SpectralExperimentsCollections()
        exps.add_dual_gate_spectra_sweep(sample_name, '$check1$', iv, lf6, vbg_start=0, vbg_stop=0,
                                      vtg_start=0, vtg_stop=0, frames=1, repeat=1, plot=False)
    #     exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=start, vbg_stop=3,
    #                                   vtg_start=start, vtg_stop=3, frames=frame, repeat=1, plot=False)
        exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly$', iv, lf6, vbg_start=start, vbg_stop=6,
                                      vtg_start=0, vtg_stop=0, frames=frame, repeat=1, plot=False)
        exps.add_dual_gate_spectra_sweep(sample_name, '$check2$', iv, lf6, vbg_start=0, vbg_stop=0,
                                      vtg_start=0, vtg_stop=0, frames=1, repeat=1, plot=False)
        exps.execute(repeat=1)


In [ ]:
import numpy as np
#max 78 331uW;87 300uw;97.5 200uw;106.35 100uw;111.5 50uW;115.05 25uw;118.02 10uw;119.5 5uw; 120.55 2.5uW;121.48 1uw;123 0.03UW 
power_angles = [121.48,119.5,118.02,111.5]
powers = [1,5,10,50]
diff_times = [120000,60000,30000,10000]
gating_times = [120000,60000,30000,10000]
center = 860
for power_angle, power, diff_time, gating_time in zip(power_angles,powers,diff_times,gating_times):
    print(power_angle, power, diff_time, gating_time)
#     start = 2
#     frame = 121
    ep300.set_position(2,power_angle)
    lf6.change_spectra_center(center)
    lf6.change_expose_time(gating_time)
    lf6.change_roi_FullSensor()
    sample_name = '$YZD95_20240120$~$Diff860nmc$~$7KpH2$~${}uw$~${}ms$~'.format(power,gating_time)
    exps = spectral_experiments.SpectralExperimentsCollections()
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0$', iv, lf6, vbg_start=-3.5, vbg_stop=3.5,
#                                    vtg_start=3.5, vtg_stop=-3.5, frames=141, repeat=1, plot=False)
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-3.5, vbg_stop=3,
                                  vtg_start=-3.5, vtg_stop=3, frames=261, repeat=1, plot=False)
    exps.execute(repeat=1)

In [ ]:
#max 78 331uW;87 300uw;97.5 200uw;106.35 100uw;111.5 50uW;115.05 25uw;118.02 10uw;119.5 5uw; 120.55 2.5uW;121.48 1uw;123 0.03UW 
x0 = np.concatenate((np.arange(0,0.5,0.05),np.arange(0.5,0.85,0.015),np.arange(0.85,1,0.01)))
power_angles = x0*45+72-45
x = x0*np.pi/4
powers = np.cos(2*x)*np.cos(2*x)*41
plt.scatter(power_angles,powers)
# for p,p_a in zip(powers, power_angles):
#     print(p,' ',p_a,'\n')
print(np.flip(power_angles))

In [ ]:
# setup diffusion time:
diff_expose_times = np.zeros_like(x0)
for i,power in enumerate(powers):
    if power < 2:
        diff_expose_times[i] = 90000
    elif  power >= 2 and power < 10:
        diff_expose_times[i] = 60000
    elif  power >= 10 and power < 50:
        diff_expose_times[i] = 40000   
    elif  power>= 50:
        diff_expose_times[i] = 15000
plt.scatter(powers,diff_expose_times)

In [ ]:
# setup diffusion time:
spectra_expose_times = np.zeros_like(x0)
for i,power in enumerate(powers):
    if power < 5:
        spectra_expose_times[i] = 3000
    elif  power >= 5 and power < 50:
        spectra_expose_times[i] = 1000
    elif  power>= 50:
        spectra_expose_times[i] = 1000
plt.scatter(powers,spectra_expose_times)

In [ ]:
print(int(diff_expose_time))
lf6.change_expose_time(int(diff_expose_time))


In [ ]:
import numpy as np
file_name = '$YZD163$~$PL730nm850SP$~$7Kp6$'
# power_angles = []
# powers = []
bg_voltages = [-3.5, -3 ,-2.4,-2.2, -1.925,-1, -0.1, 0.5,0.925, 1.2,1.4, 1.6,1.85,2.2] #low energy
# bg_voltages = [2.85, 3.3, 3.75,4, 4.2, 4.45, 4.7]
# diff_expose_times = []
# spectra_expose_times = []
center = 850
for bg_voltage in bg_voltages:
#     Diffusion image
    lf6.change_roi_FullSensor()
    lf6.change_spectra_center(0)
    iv.x_goto('Vbg', bg_voltage, 0.02, 0.1)
    with open('{}~$PowerDiff$~${}$.csv'.format(file_name,bg_voltage), 'a') as f:
        wls = lf6.get_wavelength_calibration()
        cols = np.concatenate((np.array(['Vbg','power_angle','power','expose_time'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
        np.savetxt(f, cols, fmt='%s', delimiter=',')
        for power_angle, power, diff_expose_time in zip(power_angles, powers, diff_expose_times):
            ep300.set_position(2,power_angle)
            lf6.change_expose_time(int(diff_expose_time))
            spectra = lf6.acquire()
            data = np.concatenate((np.array([bg_voltage, power_angle, power, diff_expose_time], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
            np.savetxt(f, data, fmt='%.5e', delimiter=',')
#   Linespectra   
    lf6.change_roi_LineSensor()
    lf6.change_spectra_center(center)
    with open('{}~$PowerSpectra$~${}$.csv'.format(file_name,bg_voltage), 'a') as f:
        wls = lf6.get_wavelength_calibration()
        cols = np.concatenate((np.array(['Vbg','power_angle','power','expose_time'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
        np.savetxt(f, cols, fmt='%s', delimiter=',')
        for power_angle, power, spectra_expose_time in zip(power_angles, powers, spectra_expose_times):
            ep300.set_position(2,power_angle)
            lf6.change_expose_time(int(spectra_expose_time))
            spectra = lf6.acquire()
            data = np.concatenate((np.array([bg_voltage, power_angle, power, spectra_expose_time], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
            np.savetxt(f, data, fmt='%.5e', delimiter=',')
iv.x_goto('Vbg', 0, 0.02, 0.1)
       

In [ ]:
ep300.set_position(2, 114.0)

In [ ]:
#max 78 331uW;87 300uw;97.5 200uw;106.35 100uw;111.5 50uW;115.05 25uw;118.02 10uw;119.5 5uw; 120.55 2.5uW;121.48 1uw;123 0.03UW 

# x0 = np.concatenate((np.arange(0,0.5,0.075),np.arange(0.5,0.85,0.015),np.arange(0.85,0.99,0.01)))
x0 = np.concatenate((np.arange(0.6,0.86,0.02),np.arange(0.86,0.99,0.01)))

power_angles = x0*45+78
rounded_power_angles = np.around(power_angles,decimals=3)[::-1]
x = x0*np.pi/4
powers = np.cos(2*x)*np.cos(2*x)*331+0.03
rounded_powers = np.around(powers,decimals=3)[::-1]
# plt.scatter(power_angles,powers)

# for p,p_a in zip(powers, power_angles):
#     print(p,' ',p_a,'\n')
# print(len(powers))

diff_expose_times = np.zeros_like(x0)
times = 0
for i,power in enumerate(rounded_powers):
    if power < 1:
        diff_expose_times[i] = 120000
    elif  power >= 1 and power < 3:
        diff_expose_times[i] = 90000
    elif  power >= 3 and power < 5:
        diff_expose_times[i] = 60000
    elif  power >= 5 and power < 10:
        diff_expose_times[i] = 45000
    elif  power >= 10 and power < 20:
        diff_expose_times[i] = 25000 
    elif  power >= 20 and power < 30:
        diff_expose_times[i] = 15000  
    elif  power >= 30 and power < 50:
        diff_expose_times[i] = 10000 
    elif  power >= 50 and power < 200:
        diff_expose_times[i] = 10000 
    elif  power>= 200:
        diff_expose_times[i] = 5000
# plt.scatter(powers,diff_expose_times)
for p,p_a,dt in zip(rounded_powers, rounded_power_angles,diff_expose_times):
    print(p,' ',p_a,' ',int(dt),'\n')
    times = times + dt
print(times)
    


In [ ]:
# power dep 
#max 56 358uW; 67.6 300uw;76.7 200uw;85 100uw;90 50uW;93.3 25uw;96.25 10uw;97.75 5uw;100.1 1uw;
import numpy as np
# power_angles = [  67.6, 85,   90, 93.3,96.25,97.75 ,98.75, 100.1][::-1]
# powers =       [   300, 100,  50, 25,  10   ,5   ,2.5  ,1][::-1] 

lf6.change_roi_FullSensor()
lf6.change_spectra_center(860)
for power_angle, power,diff_expose_time in zip(rounded_power_angles, rounded_powers,diff_expose_times):
    lf6.change_expose_time(int(diff_expose_time))
    ep300.set_position(2,power_angle)
    sample_name = '$YZ139_20231229$~$Diff860nmc~In$~$7Kp3$~${}uw$~${}ms$~'.format(power,int(diff_expose_time))
    exps = spectral_experiments.SpectralExperimentsCollections()

    exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly$', iv, lf6, vbg_start=-3.5, vbg_stop=3.5,
                                   vtg_start=-3.5, vtg_stop=3.5, frames=141, repeat=1, plot=False)


    exps.execute(repeat=1)

In [ ]:
print(diff_expose_time)

In [ ]:
# power dep 
#max 56 358uW; 67.6 300uw;76.7 200uw;85 100uw;90 50uW;93.3 25uw;96.25 10uw;97.75 5uw;100.1 1uw;
import numpy as np
# [94, 86.3, 81.1, 76.5, 71.7, 69.2, 66.2, 62.6, 60.6, 59.5, 57.7, 56.4, 55]
# [300, 250, 200, 150, 100, 75, 50, 25, 15, 10, 5, 2.5, 1.3] 
# power_angles = [110, 100, 92, 88, 84, 79, 75.5, 73.7, 72.6, 71, 69.7,67.7][::-1]
# powers = [370, 300, 200, 150, 100,  50, 25, 15, 10, 5, 2.5,1.5][::-1] 
# power_angles = [56,  67.6, 76.7, 85,   90, 93.3,96.25,97.75 ,100.1][::-1]
# powers =       [358, 300,  200,  100,  50, 25,  10   ,5     ,1][::-1] 
power_angles = [  67.6, 85,   90, 93.3,96.25,97.75 ,98.75, 100.1][::-1]
powers =       [   300, 100,  50, 25,  10   ,5   ,2.5  ,1][::-1] 
expose_time = 4
lf6.change_expose_time(4000)
# -5 0.0 0.0 -5 $TG+BG=-5.0$ 101
# -5 1.0 1.0 -5 $TG+BG=-4.0$ 121
# -5 2.0 2.0 -5 $TG+BG=-3.0$ 141
# -5 3.0 3.0 -5 $TG+BG=-2.0$ 161
# -5 4.0 4.0 -5 $TG+BG=-1.0$ 181
# -5 5.0 5.0 -5 $TG+BG=0.0$ 201
# 5 -4.0 -4.0 5 $TG+BG=1.0$ 181
# 5 -3.0 -3.0 5 $TG+BG=2.0$ 161
# 5 -2.0 -2.0 5 $TG+BG=3.0$ 141
# 5 -1.0 -1.0 5 $TG+BG=4.0$ 121
# 5 0.0 0.0 5 $TG+BG=5.0$ 101

# 0.0 5 -5 0.0 $TG-BG=-5.0$ 101
# -2.5 5 -5 2.5 $TG-BG=-2.5$ 151
# -5.0 5 -5 5.0 $TG-BG=0.0$ 201
# -5 2.5 -2.5 5 $TG-BG=2.5$ 151
# -5 0.0 0.0 5 $TG-BG=5.0$ 101
for power_angle, power in zip(power_angles, powers):
    if power > 20:
        lf6.change_expose_time(2000)
        expose_time = 2
    ep300.set_position(2,power_angle)
    sample_name = '$DX156_20231224$~$PL730nm$~$7Kp4up$~${}uw$~${}s$~'.format(power,expose_time)
    exps = spectral_experiments.SpectralExperimentsCollections()
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-5.0$', iv, lf6, vbg_start=-5, vbg_stop=0,
#                                        vtg_start=0, vtg_stop=-5, frames=101, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-4.0$', iv, lf6, vbg_start=-5, vbg_stop=1,
#                                        vtg_start=1, vtg_stop=-5, frames=121, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-3.0$', iv, lf6, vbg_start=-5, vbg_stop=2,
#                                        vtg_start=2, vtg_stop=-5, frames=141, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-2.0$', iv, lf6, vbg_start=-5, vbg_stop=3,
#                                        vtg_start=3, vtg_stop=-5, frames=161, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-1.0$', iv, lf6, vbg_start=-5, vbg_stop=4,
#                                        vtg_start=4, vtg_stop=-5, frames=181, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0.0$', iv, lf6, vbg_start=-5, vbg_stop=5,
#                                        vtg_start=5, vtg_stop=-5, frames=201, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=1.0$', iv, lf6, vbg_start=5, vbg_stop=-4,
#                                        vtg_start=-4, vtg_stop=5, frames=181, repeat=1, plot=True)   
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=2.0$', iv, lf6, vbg_start=5, vbg_stop=-3,
#                                        vtg_start=-3, vtg_stop=5, frames=161, repeat=1, plot=True)     
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=3.0$', iv, lf6, vbg_start=5, vbg_stop=-2,
#                                        vtg_start=-2, vtg_stop=5, frames=141, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=4.0$', iv, lf6, vbg_start=5, vbg_stop=-1,
#                                        vtg_start=-1, vtg_stop=5, frames=121, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=5.0$', iv, lf6, vbg_start=5, vbg_stop=0,
#                                        vtg_start=0, vtg_stop=5, frames=101, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=5.5$', iv, lf6, vbg_start=5, vbg_stop=0.5,
#                                        vtg_start=0.5, vtg_stop=5, frames=91, repeat=1, plot=True)    
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=6.0$', iv, lf6, vbg_start=5, vbg_stop=1,
#                                        vtg_start=1, vtg_stop=5, frames=81, repeat=1, plot=True)

#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-5.0$', iv, lf6, vbg_start=0, vbg_stop=5,
#                                   vtg_start=-5, vtg_stop=0, frames=101, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-2.5$', iv, lf6, vbg_start=-2.5, vbg_stop=5,
#                                   vtg_start=-5, vtg_stop=2.5, frames=151, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-5, vbg_stop=5,
#                                   vtg_start=-5, vtg_stop=5, frames=201, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=2.5$', iv, lf6, vbg_start=-5, vbg_stop=2.5,
#                                   vtg_start=-2.5, vtg_stop=5, frames=151, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=5.0$', iv, lf6, vbg_start=-5, vbg_stop=0,
#                                   vtg_start=0, vtg_stop=5, frames=101, repeat=1, plot=True)

#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0$', iv, lf6, vbg_start=-5, vbg_stop=5,
#                                   vtg_start=-5, vtg_stop=5, frames=201, repeat=1, plot=True)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-2.5$', iv, lf6, vbg_start=-5, vbg_stop=2.5,
#                                    vtg_start=2.5, vtg_stop=-5, frames=151, repeat=1, plot=False)
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=2.5$', iv, lf6, vbg_start=-2.5, vbg_stop=5,
#                                    vtg_start=5, vtg_stop=-2.5, frames=151, repeat=1, plot=False)    
#     exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-5.0$', iv, lf6, vbg_start=-5, vbg_stop=0,
#                                    vtg_start=0, vtg_stop=-5, frames=101, repeat=1, plot=False)
    exps.execute(repeat=1)

In [ ]:
lf6.change_expose_time("2000")


In [ ]:
ep300.set_position(1,41.25)
print(ep300.get_positions('1'))
meas_power =  c_double()
tlPM.setWavelength(730)
tlPM.measPower(byref(meas_power))
print(meas_power.value*1e6)#unit uW
print(round(meas_power.value*1e6-0.005,4)) #unit uW

In [ ]:
# power dep with measure power
power_angles = [  48.00  ]
for power_angle in power_angles:
    ep300.set_position(1,power_angle)
    print('Angle=',ep300.get_positions(1))
    meas_power =  c_double()
    tlPM.measPower(byref(meas_power))
    power = round(meas_power.value*1e6-0.005,4)#unit uW
    print('Power=',power,'uw')
    if power < 0.2:
        lf6.change_expose_time("2000")
        expose_time = 2
    else:
        lf6.change_expose_time("1000")
        expose_time = 1
    sample_name = '$YZ72$~$PL730nm$~$4Kp11n2$~${}uw$~${}s$~'.format(power,expose_time)
    exps = spectral_experiments.SpectralExperimentsCollections()
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-5.0$', iv, lf6, vbg_start=-5, vbg_stop=0,
    #                                    vtg_start=0, vtg_stop=-5, frames=101, repeat=1, plot=True)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-4.0$', iv, lf6, vbg_start=-5, vbg_stop=1,
    #                                    vtg_start=1, vtg_stop=-5, frames=121, repeat=1, plot=True)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-3.0$', iv, lf6, vbg_start=-5, vbg_stop=2,
    #                                    vtg_start=2, vtg_stop=-5, frames=141, repeat=1, plot=True)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-2.0$', iv, lf6, vbg_start=-5, vbg_stop=3,
    #                                    vtg_start=3, vtg_stop=-5, frames=161, repeat=1, plot=True)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=-1.0$', iv, lf6, vbg_start=-5, vbg_stop=4,
    #                                    vtg_start=4, vtg_stop=-5, frames=181, repeat=1, plot=True)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG+BG=0.0$', iv, lf6, vbg_start=-4.5, vbg_stop=4.5,
    #                                    vtg_start=5, vtg_stop=-5, frames=201, repeat=1, plot=True)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0.0$', iv, lf6, vbg_start=-5, vbg_stop=5,
    #                                    vtg_start=5, vtg_stop=-5, frames=201, repeat=1, plot=True) 
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=1.0$', iv, lf6, vbg_start=5, vbg_stop=-4,
    #                                    vtg_start=-4, vtg_stop=5, frames=181, repeat=1, plot=True)   
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=2.0$', iv, lf6, vbg_start=5, vbg_stop=-3,
    #                                    vtg_start=-3, vtg_stop=5, frames=161, repeat=1, plot=True)     
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=3.0$', iv, lf6, vbg_start=5, vbg_stop=-2,
    #                                    vtg_start=-2, vtg_stop=5, frames=141, repeat=1, plot=True)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=4.0$', iv, lf6, vbg_start=5, vbg_stop=-1,
    #                                    vtg_start=-1, vtg_stop=5, frames=121, repeat=1, plot=True)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=5.0$', iv, lf6, vbg_start=5, vbg_stop=0,
    #                                    vtg_start=0, vtg_stop=5, frames=101, repeat=1, plot=True)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=5.8$', iv, lf6, vbg_start=5, vbg_stop=0.8,
    #                                    vtg_start=0.8, vtg_stop=5, frames=169, repeat=1, plot=True)
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=5.8$', iv, lf6, vbg_start=6, vbg_stop=-0.2,
                                       vtg_start=-0.2, vtg_stop=6, frames=249, repeat=1, plot=True)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-5.0$', iv, lf6, vbg_start=0, vbg_stop=5,
    #                               vtg_start=-5, vtg_stop=0, frames=101, repeat=1, plot=True)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=-2.5$', iv, lf6, vbg_start=-2.5, vbg_stop=5,
    #                               vtg_start=-5, vtg_stop=2.5, frames=151, repeat=1, plot=True)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$0.9TG-BG=0$', iv, lf6, vbg_start=-4.5, vbg_stop=4.5,
    #                               vtg_start=-5, vtg_stop=5, frames=201, repeat=1, plot=True)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0rev$', iv, lf6, vbg_start=5, vbg_stop=-5,
    #                               vtg_start=5, vtg_stop=-5, frames=101, repeat=1, plot=True)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=2.5$', iv, lf6, vbg_start=-5, vbg_stop=2.5,
    #                               vtg_start=-2.5, vtg_stop=5, frames=151, repeat=1, plot=True)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=5.0$', iv, lf6, vbg_start=-5, vbg_stop=0,
    #                               vtg_start=0, vtg_stop=5, frames=101, repeat=1, plot=True)    
   
    exps.execute(repeat=1)
winsound.Beep(frequency=750, duration=300)  # frequency in Hz, duration in milliseconds



In [ ]:
# power dep with measure power, move stage ver
import numpy as np
# power_angles = [500, 600., 720., 840., 960., 1080., 1200., 1320., 
#                 1440., 1560., 1680., 1800., 1920., 2040., 2160., 2280., 2400., 2520., 2640., 
#                 2760., 2880., 3000., 3120., 3240., 3360., 3480., 3600.]
power_angles = [500., 3600.]
tlPM.setWavelength(730)

# power_angles = [87,75]
# powers = [ 0.1, 10.8, 12.4, 14, 17.6] 
# doping = np.linspace()
for power_angle in power_angles:
    stage.move_to(power_angle)
    meas_power =  c_double()
    tlPM.measPower(byref(meas_power))
    power = round(meas_power.value*1e6,5) #unit uW
    print('Power=',power,'uw')
    # if power < 2:
    #     lf6.change_expose_time("2000")
    #     expose_time = 2
    # else:
    #     lf6.change_expose_time("1000")
    #     expose_time = 1
    expose_time = 1
    sample_name = '$YZ248$~$PL730nm$~$pQn1$~${}uw$~${}s$~'.format(power,expose_time)
    exps = spectral_experiments.SpectralExperimentsCollections()
    exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0.0$', iv, lf6, vbg_start=-6, vbg_stop=6,
                                        vtg_start=-6, vtg_stop=6, frames=121, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0.0$', iv, lf6, vbg_start=-7.5, vbg_stop=9,
    #                                vtg_start=7.5, vtg_stop=-9, frames=166, repeat=1, plot=False)
    # exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0.0$', iv, lf6, vbg_start=-6, vbg_stop=6,
    #                                 vtg_start=6, vtg_stop=-6, frames=121, repeat=1, plot=False)
    exps.execute(repeat=1)
winsound.Beep(frequency=750, duration=300)  # frequency in Hz, duration in milliseconds

In [ ]:
ep300.set_position(1,3)
meas_power =  c_double()
tlPM.setWavelength(650)
# tlPM.setWavelength(730)
tlPM.measPower(byref(meas_power))
print(meas_power.value*1e6)#unit uW
print(round(meas_power.value*1e6,3)*0.75) #unit uW

In [ ]:
#max 78 331uW;87 300uw;97.5 200uw;106.35 100uw;111.5 50uW;115.05 25uw;118.02 10uw;119.5 5uw; 120.55 2.5uW;121.48 1uw;123 0.03UW 
x0 = np.concatenate((np.arange(0,0.5,0.05),np.arange(0.5,0.85,0.015),np.arange(0.85,1,0.01)))
power_angles = -x0*45+48
x = x0*np.pi/4
powers = np.cos(2*x)*np.cos(2*x)*41
# plt.scatter(power_angles,powers)
# for p,p_a in zip(powers, power_angles):
#     print(p,' ',p_a,'\n')
max_per_row = 10  # Adjust as needed
formatted_angles = [f"{angle:.2f}" for angle in np.flip(power_angles)]
print("\n".join(", ".join(formatted_angles[i:i + max_per_row]) for i in range(0, len(formatted_angles), max_per_row)))
# print(", ".join(map(str, np.flip(power_angles))))

In [ ]:
# '$YZ72-5cmL$~$4KCW730nmPL$~$p11n2$~$400uW WLoff$~$1s$~$0T$~'
vbg = 0
vtg = 0
print(f'{vtg-vbg}V')
iv.x_goto('Vbg', vbg, 0.1, 0.1)
iv.x_goto('Vtg', vtg, 0.1, 0.1)
iv.report_status()

In [ ]:
# sample_name = '$YZD144-5cmL$~$4KCW633nmPL$~$pX$~$4uW$~$1s$~$0T$~'
# sample_name = '$YZD144-5cmL$~$4KPL$~$p1$~$775nmC$~$1s$~$0T$~'
sample_name = '$YZ247-5cmL$~$4K730nmPL$~$p4n2$~$WLoff$~$5uW0.5s$~$0T$~'

# sample_name = '$YZD95$~$REF600g720nmCWL$~$pR3$~$2s$~$0T$~'

# sample_name = '$YZ182$~$REF670nmc$~$4Kp3$~$1s$~$0T$~'

# sample_name = '$YZD77$~$REF$~$7KpEl$~$620nmC300g$~$2s$~'
# sample_name = '$TEST$~$PL$~$monoWSe2$~$633nm600g50uW0T$~$1s$~'

exps = spectral_experiments.SpectralExperimentsCollections()

exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0.0$', iv, lf6, vbg_start=-8, vbg_stop=8,
                                        vtg_start=-8, vtg_stop=8, frames=161, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG-BG=0.0rev$', iv, lf6, vbg_start=8, vbg_stop=-8,
#                                         vtg_start=8, vtg_stop=-8, frames=161, repeat=1, plot=False)

# exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0.0$', iv, lf6, vbg_start=-8, vbg_stop=8,
#                                    vtg_start=8, vtg_stop=-8, frames=161, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0.0rev$', iv, lf6, vbg_start=8, vbg_stop=-8,
#                                    vtg_start=-8, vtg_stop=8, frames=161, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TG+BG=0.0$', iv, lf6, vbg_start=-11, vbg_stop=9,
#                                    vtg_start=11, vtg_stop=-9, frames=201, repeat=1, plot=False)

# exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly$', iv, lf6, vbg_start=-8, vbg_stop=8,
#                                    vtg_start=0, vtg_stop=0, frames=161, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly_vbg_rev$', iv, lf6, vbg_start=8, vbg_stop=-8,
#                                    vtg_start=0, vtg_stop=0, frames=161, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$BGonly+to-$', iv, lf6, vbg_start=5, vbg_stop=-5,
#                                    vtg_start=0, vtg_stop=0, frames=101, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly$', iv, lf6, vbg_start=0, vbg_stop=0,
#                                    vtg_start=-8, vtg_stop=8, frames=161, repeat=1, plot=False)
# exps.add_dual_gate_spectra_sweep(sample_name, '$TGonly+to-$', iv, lf6, vbg_start=0, vbg_stop=0,
#                                    vtg_start=5, vtg_stop=-5, frames=101, repeat=1, plot=False)

# exps.add_dual_gate_spectra_sweep(sample_name, '$TG=BG=0$', iv, lf6, vbg_start=0, vbg_stop=0,
#                                   vtg_start=0, vtg_stop=0, frames=1, repeat=1, plot=False)


exps.execute(repeat=1)
winsound.Beep(frequency=750, duration=300)  # frequency in Hz, duration in milliseconds


In [ ]:
iv.x_goto('Vbg', 7.5, 0.1, 0.05)
iv.x_goto('Vtg', -7.5, 0.1, 0.05)

In [ ]:
stage.move_to(3600)
meas_power =  c_double()
        
tlPM.measPower(byref(meas_power))
print(meas_power.value*1e6)#unit uW

In [ ]:
##PL spectra power dep
power_angles = [500., 600., 720., 840., 960., 1080., 1200., 1320., 
                1440., 1560., 1680., 1800., 1920., 2040., 2160., 2280., 2400., 2520., 2640., 
                2760., 2880., 3000., 3120., 3240., 3330 , 3480., 3600.]
# power_angles = [500., 3600.]

power_efficient = 1
# power_angles = [72.0, 71.55, 71.1]
file_name = '$YZD248-4K$~$PLcw730nm5cmL$~$pQn1$~$Power1s$~$TG-BG=-15V_002$'
lf6.change_expose_time('1000')
tlPM.setWavelength(730)
# vbg = 0
# vtg = 0
with open('{}.csv'.format(file_name), 'a') as f:
    wls = lf6.get_wavelength_calibration()
    cols = np.concatenate((np.array(['Vbg','Vtg','power'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
    np.savetxt(f, cols, fmt='%s', delimiter=',')
    # while True:
        # input_power = input('power (uW)')
        # if input_power == 'q':
            # break
    for power_angle in power_angles:    
        stage.move_to(power_angle)
        meas_power =  c_double()
        
        tlPM.measPower(byref(meas_power))
        # print(meas_power.value*1e6)#unit uW
        input_power = round(meas_power.value*1e6,5)*power_efficient
        spectra = lf6.acquire()
        data = np.concatenate((np.array([vbg, vtg, input_power], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
        np.savetxt(f, data, fmt='%.5e', delimiter=',')
        
winsound.Beep(frequency=750, duration=300)  # frequency in Hz, duration in milliseconds
        

In [ ]:
ep300.set_position(1,40)
meas_power =  c_double()
tlPM.setWavelength(730)
tlPM.measPower(byref(meas_power))
print(meas_power.value*1e6)#unit uW
print(round(meas_power.value*1e6,3)) #unit uW

In [ ]:
#manual power dep
f_name = '$YZD95$~$PL730nm$~$15KpRE$~$2s$~$0T$~$TG=BG=0$~$Powerdep$_001'
# f_name = 'test'
# wls = lf6.get_wavelength_calibration()
with open('{}.csv'.format(f_name), 'a') as f:
    wls = lf6.get_wavelength_calibration()
    cols = np.concatenate((np.array(['Vbg','Vtg','Power','time'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
    np.savetxt(f, cols, fmt='%s', delimiter=',')
    while True:
        Time_key = 2
        Power_key= input('Power(uW):')
        if Power_key == 'q':
            break
        else:             
            spectra = lf6.acquire()
            data = np.concatenate((np.array([0,0,float(Power_key)/1.6,Time_key], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
            winsound.Beep(frequency=750, duration=300)  # frequency in Hz, duration in milliseconds
            np.savetxt(f, data, fmt='%.5e', delimiter=',') 

## Vtg-Vbg Mega Sweep

In [ ]:
# import numpy as np
# f_name = '$YZD72$~$PL730nm$~$7Kp11$~$2.5uW$~$1s$~$Megasweep$'
# vbgs = np.linspace(-1, 0, 11)
# vtgs = np.linspace(-1, 0, 11)
# flag = True
# for vbg in vbgs:
#     if flag == True:
#         vtgseq = vtgs
#     else:
#         vtgseq = np.flip(vtgs)
#     for vtg in vtgseq:
#         print(vbg,vtg)
#     flag = not flag
vbgs = np.linspace(-8, 8, 161)
vtgs = np.linspace(-8, 8, 161)
print(vbgs,vtgs)

In [ ]:
stage.move_to(3000)
meas_power =  c_double()
tlPM.setWavelength(730)
tlPM.measPower(byref(meas_power))
print(meas_power.value*1e6-0.005)#unit uW

In [ ]:
iv_gate3.x_goto('Vcontact',0,0.05,delay)

In [ ]:
iv.x_goto('Vbg', 0, 0.1, delay)
iv.x_goto('Vtg', 0, 0.1, delay)

In [ ]:
# tg=bg bias
import numpy as np
#  real 167mins 0.2s expos 0.05step
for center in ['695']:

	lf6.change_spectra_center(center)
	lf6.change_expose_time("500")
	f_name =f'$YZ261megasweep$~$4KREF_{center}nm_TG=BG$~$0.5sx3$~$9T$~$pb$'
	sample_name = f_name
	# vbgs=np.concatenate((np.arange(0,0.75,0.03),np.arange(0.75,1,0.01)))
	# vbgs = np.linspace(-4.5, 6, 211)
	vgs = np.linspace(-2, 0.5, 101)
	vbias = np.linspace(-1.5, -1.75, 251)

	flag = True
	step = 0.02 #V
	delay = 0.1 #s
	with open('{}.csv'.format(f_name), 'a') as f:
				wls = lf6.get_wavelength_calibration()
				cols = np.concatenate((np.array(['Vbias','Vbg','Vtg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
				np.savetxt(f, cols, fmt='%s', delimiter=',')
				for vbs in vbias:
					iv_gate3.x_goto('Vcontact',vbs,0.0005,delay)
					if flag == True:
						vgseq = vgs
					else:
						vgseq = np.flip(vgs)
					for vg in vgseq:
						iv.x_goto('Vbg', vg, step, delay)
						iv.x_goto('Vtg', vg, step, delay)
						spectra = lf6.acquire()
						data = np.concatenate((np.array([vbs, vg, vg, ], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
						np.savetxt(f, data, fmt='%.5e', delimiter=',') 
					flag = not flag
				iv.x_goto('Vbg', 0, step, delay)
				iv.x_goto('Vtg', 0, step, delay)
				print('vg:', 0)
winsound.Beep(frequency=750, duration=300)  # frequency in Hz, duration in milliseconds

In [ ]:
# tg=bg bias
import numpy as np
#  real 167mins 0.2s expos 0.05step
for center in ['730']:

	lf6.change_spectra_center(center)
	lf6.change_expose_time("1000")
	f_name =f'$YZ261megasweep$~$4KPL_{center}nm_TG=BG$~$1s$~$9T$~$pb$'
	sample_name = f_name
	# vbgs=np.concatenate((np.arange(0,0.75,0.03),np.arange(0.75,1,0.01)))
	# vbgs = np.linspace(-4.5, 6, 211)
	vgs = np.linspace(-2, 1, 31)
	vbias = np.linspace(-1.5, -1.75, 26)

	flag = True
	step = 0.05 #V
	delay = 0.1 #s
	with open('{}.csv'.format(f_name), 'a') as f:
				wls = lf6.get_wavelength_calibration()
				cols = np.concatenate((np.array(['Vbg','Vtg','Vbias'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
				np.savetxt(f, cols, fmt='%s', delimiter=',')
				for vg in vgs:
					iv.x_goto('Vbg', vg, step, delay)
					iv.x_goto('Vtg', vg, step, delay)
					if flag == True:
						vbiasseq = vbias
					else:
						vbiasseq = np.flip(vbias)
					for vbs in vbiasseq:
						iv_gate3.x_goto('Vcontact',vbs,0.006,delay)
						spectra = lf6.acquire()
						data = np.concatenate((np.array([vg, vg, vbs], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
						np.savetxt(f, data, fmt='%.5e', delimiter=',') 
					flag = not flag
				iv.x_goto('Vbg', 0, step, delay)
				iv.x_goto('Vtg', 0, step, delay)
				print('vg:', 0)
winsound.Beep(frequency=750, duration=300)  # frequency in Hz, duration in milliseconds

In [ ]:
import numpy as np
#  real 167mins 0.2s expos 0.05step
center = '860'
lf6.change_spectra_center(center)
lf6.change_expose_time("1000")
f_name ='$YZ302megasweep$~$4KPL_860nm$~$1sx1$~$0T$~$p1$'
sample_name = f_name
# vbgs=np.concatenate((np.arange(0,0.75,0.03),np.arange(0.75,1,0.01)))
# vbgs = np.linspace(-4.5, 6, 211)
vbgs = np.linspace(-6, 8, 141)
vtgs = np.linspace(-6, 8, 141)

flag = True
step = 0.05 #V
delay = 0.1 #s
with open('{}.csv'.format(f_name), 'a') as f:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['Vbg','Vtg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f, cols, fmt='%s', delimiter=',')
            for vbg in vbgs:
                iv.x_goto('Vbg', vbg, step, delay)
                if flag == True:
                    vtgseq = vtgs
                else:
                    vtgseq = np.flip(vtgs)
                for vtg in vtgseq:
                    iv.x_goto('Vtg', vtg, step, delay)
                    spectra = lf6.acquire()
                    data = np.concatenate((np.array([vbg, vtg], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                    np.savetxt(f, data, fmt='%.5e', delimiter=',') 
                flag = not flag
            iv.x_goto('Vbg', 0, step, delay)
            iv.x_goto('Vtg', 0, step, delay)
            print('vg:', 0)
winsound.Beep(frequency=750, duration=300)  # frequency in Hz, duration in milliseconds


In [ ]:
import numpy as np
#  real 167mins 0.2s expos 0.05step

# lf6.change_expose_time("200")
f_name ='$YZ237-5cmL$~$4KREF$~$p3$~$730nmC$~$0.5sX4$~$9T$~'
vbgs = np.linspace(-9, 9, 91)
vtgs = np.linspace(-9, 9, 91)
flag = True
step = 0.05 #V
delay = 0.1 #s
lf6.change_spectra_center('730')
with open('{}.csv'.format(f_name), 'a') as f:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['Vbg','Vtg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f, cols, fmt='%s', delimiter=',')
            for vbg in vbgs:
                iv.x_goto('Vbg', vbg, step, delay)
                if flag == True:
                    vtgseq = vtgs
                else:
                    vtgseq = np.flip(vtgs)
                for vtg in vtgseq:
                    iv.x_goto('Vtg', vtg, step, delay)
                    spectra = lf6.acquire()
                    data = np.concatenate((np.array([vbg, vtg], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                    np.savetxt(f, data, fmt='%.5e', delimiter=',') 
                flag = not flag
            iv.x_goto('Vbg', 0, step, delay)
            iv.x_goto('Vtg', 0, step, delay)
            print('vg:', 0)
winsound.Beep(frequency=750, duration=300)  # frequency in Hz, duration in milliseconds

f_name ='$YZ237-5cmL$~$4KREF$~$p3$~$690nmC$~$0.5sX4$~$9T$~'

vbgs = np.linspace(-9, 9, 91)
vtgs = np.linspace(-9, 9, 91)
flag = True
step = 0.05 #V
delay = 0.1 #s
lf6.change_spectra_center('690')
with open('{}.csv'.format(f_name), 'a') as f:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['Vbg','Vtg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f, cols, fmt='%s', delimiter=',')
            for vbg in vbgs:
                iv.x_goto('Vbg', vbg, step, delay)
                if flag == True:
                    vtgseq = vtgs
                else:
                    vtgseq = np.flip(vtgs)
                for vtg in vtgseq:
                    iv.x_goto('Vtg', vtg, step, delay)
                    spectra = lf6.acquire()
                    data = np.concatenate((np.array([vbg, vtg], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                    np.savetxt(f, data, fmt='%.5e', delimiter=',') 
                flag = not flag
            iv.x_goto('Vbg', 0, step, delay)
            iv.x_goto('Vtg', 0, step, delay)
            print('vg:', 0)
winsound.Beep(frequency=750, duration=300)  # frequency in Hz, duration in milliseconds

In [ ]:
ep300.set_position(1,38)
print(ep300.get_positions('1'))
meas_power =  c_double()
tlPM.setWavelength(730)
tlPM.measPower(byref(meas_power))
print(meas_power.value*1e6)#unit uW
print(round(meas_power.value*1e6,3)/10*7)#unit uW


In [ ]:
import numpy as np
#  real 167mins 0.2s expos 0.05step
power_angles = [70, 73, 79.8,100]
for angle in power_angles:
    ep300.set_position(1,angle)
    print(ep300.get_positions('1'))
    meas_power =  c_double()
    tlPM.setWavelength(730)
    tlPM.measPower(byref(meas_power))
    print(meas_power.value*1e6)#unit uW
    print(round(meas_power.value*1e6,3)/10*7)#unit uW  
    real_power = round(meas_power.value*1e6,3)/10*7
    if real_power < 1.2:
         expose_time = 1.0
         lf6.change_expose_time("1000")
    else:
         expose_time = 0.2
         lf6.change_expose_time('200')
    f_name ='$YZ194megasweep$~$PL730nm$~$4Kpn11$~${}s{}uW$~$0T$'.format(expose_time,real_power)
    sample_name = f_name
    # vbgs=np.concatenate((np.arange(0,0.75,0.03),np.arange(0.75,1,0.01)))
    # vbgs = np.linspace(-4.5, 6, 211)
    vbgs = np.linspace(-5, 5, 101)
    vtgs = np.linspace(-5, 5, 101)
    flag = True
    step = 0.05 #V
    delay = 0.05 #s
    with open('{}.csv'.format(f_name), 'a') as f:
                wls = lf6.get_wavelength_calibration()
                cols = np.concatenate((np.array(['Vbg','Vtg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
                np.savetxt(f, cols, fmt='%s', delimiter=',')
                for vbg in vbgs:
                    iv.x_goto('Vbg', vbg, step, delay)
                    if flag == True:
                        vtgseq = vtgs
                    else:
                        vtgseq = np.flip(vtgs)
                    for vtg in vtgseq:
                        iv.x_goto('Vtg', vtg, step, delay)
                        spectra = lf6.acquire()
                        data = np.concatenate((np.array([vbg, vtg], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                        np.savetxt(f, data, fmt='%.5e', delimiter=',') 
                    flag = not flag
                iv.x_goto('Vbg', 0, step, delay)
                iv.x_goto('Vtg', 0, step, delay)
                print('vg:', 0)
winsound.Beep(frequency=750, duration=300)  # frequency in Hz, duration in milliseconds


In [ ]:
# Efield/Doping megasweep
# sample_name = '$DX156$~$PL730nm300g900C$~$4Kp3r$~$0.2uW1s$~$0T$~'
f_name ='$DX156megasweepFixDoping[-2,2]$~$PL730nm0.2uW WLon delay0.5$~$4Kp4n1$~$1s$~$0T$'
Efields = np.linspace(-2,2,21)
Dopings = np.linspace(-2,2,21)
Flag = True
delay = 0.5
with open('{}.csv'.format(f_name), 'a') as f:
    wls = np.linspace(1,1024,1024)
    cols = np.concatenate((np.array(['Vbg','Vtg','Doping','Efield'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
    np.savetxt(f, cols, fmt='%s', delimiter=',')
    for Doping in Dopings:
        if Flag == True:
            Efields_seq = Efields
        else:
            # Efields_seq = np.flip(Efields) 
            Efields_seq = Efields
        for Efield in Efields_seq:
            vtg = np.around((Efield+Doping)/2,decimals=2)
            vbg = np.around((Doping-Efield)/2,decimals=2)
            iv.x_goto('Vbg', vbg, 0.075, delay)
            iv.x_goto('Vtg', vtg, 0.075, delay)
            spectra = lf6.acquire()
            data = np.concatenate((np.array([vbg, vtg, Doping, Efield], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
            np.savetxt(f, data, fmt='%.5e', delimiter=',') 
        Flag = not Flag

    iv.x_goto('Vbg', 0, 0.1, 0.1)
    iv.x_goto('Vtg', 0, 0.1, 0.1)
    print('vg:', 0)

#sweep doping
# f_name = '$DX156$~$Megasweep$~$PL730nm300g900C4K$~$4Kp3r$~$9uW1s$~'
# Efields = np.linspace(-5,5,101)
# Dopings = np.linspace(-2.5,2.5,51)
# Flag = True
# with open('{}.csv'.format(f_name), 'a') as f:
#     wls = np.linspace(1,1024,1024)
#     cols = np.concatenate((np.array(['Vbg','Vtg','Doping','Efield'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
#     np.savetxt(f, cols, fmt='%s', delimiter=',')
#     for Efield in Efields:
#         if Flag == True:
#             Doping_seq = Dopings
#         else:
#             Doping_seq = np.flip(Dopings) 
#         for Doping in Doping_seq:
#             vtg = np.around((Efield+Doping)/2,decimals=2)
#             vbg = np.around((Doping-Efield)/2,decimals=2)
#             iv.x_goto('Vbg', vbg, 0.03, 0.1)
#             iv.x_goto('Vtg', vtg, 0.03, 0.1)
#             spectra = lf6.acquire()
#             data = np.concatenate((np.array([vbg, vtg, Doping, Efield], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
#             np.savetxt(f, data, fmt='%.5e', delimiter=',') 
#         Flag = not Flag

#     iv.x_goto('Vbg', 0, 0.03, 0.1)
#     iv.x_goto('Vtg', 0, 0.03, 0.1)
#     print('vg:', 0)

In [ ]:
import numpy as np
#  real 167mins 0.2s expos 0.05step

f_name ='$DX156megasweepFixbg$~$PL730nm900uW delay0.3$~$4Kp3n2$~$1s$~$0T$'
sample_name = f_name
# vbgs=np.concatenate((np.arange(0,0.75,0.03),np.arange(0.75,1,0.01)))
# vbgs = np.linspace(-4.5, 6, 211)
vbgs = np.linspace(-2, 2, 21)
vtgs = np.linspace(-2, 2, 21)
flag = True
step = 0.075 #V
delay = 0.3 #s
with open('{}.csv'.format(f_name), 'a') as f:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['Vbg','Vtg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f, cols, fmt='%s', delimiter=',')
            for vbg in vbgs:
                iv.x_goto('Vbg', vbg, step, delay)
                if flag == True:
                    vtgseq = vtgs
                else:
                    vtgseq = vtgs
                for vtg in vtgseq:
                    iv.x_goto('Vtg', vtg, step, delay)
                    spectra = lf6.acquire()
                    data = np.concatenate((np.array([vbg, vtg], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                    np.savetxt(f, data, fmt='%.5e', delimiter=',') 
                flag = not flag
            iv.x_goto('Vbg', 0, step, delay)
            iv.x_goto('Vtg', 0, step, delay)
            print('vg:', 0)
winsound.Beep(frequency=750, duration=300)  # frequency in Hz, duration in milliseconds

# import numpy as np
# #  real 167mins 0.2s expos 0.05step

# f_name ='$DX156megasweepFixtg$~$PL730nm0.2uW WLon delay0.3$~$4Kp3n1$~$1s$~$0T$'
# sample_name = f_name
# # vbgs=np.concatenate((np.arange(0,0.75,0.03),np.arange(0.75,1,0.01)))
# # vbgs = np.linspace(-4.5, 6, 211)
# vbgs = np.linspace(-2, 2, 21)
# vtgs = np.linspace(-2, 2, 21)
# flag = True
# step = 0.075 #V
# delay = 0.3 #s
# with open('{}.csv'.format(f_name), 'a') as f:
#             wls = lf6.get_wavelength_calibration()
#             cols = np.concatenate((np.array(['Vbg','Vtg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
#             np.savetxt(f, cols, fmt='%s', delimiter=',')
#             for vbg in vbgs:
#                 iv.x_goto('Vtg', vbg, step, delay)
#                 if flag == True:
#                     vtgseq = vtgs
#                 else:
#                     vtgseq = vtgs
#                 for vtg in vtgseq:
#                     iv.x_goto('Vbg', vtg, step, delay)
#                     spectra = lf6.acquire()
#                     data = np.concatenate((np.array([vtg, vbg], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
#                     np.savetxt(f, data, fmt='%.5e', delimiter=',') 
#                 flag = not flag
#             iv.x_goto('Vbg', 0, step, delay)
#             iv.x_goto('Vtg', 0, step, delay) 
#             print('vg:', 0)
# winsound.Beep(frequency=750, duration=300)  # frequency in Hz, duration in milliseconds


In [ ]:
# # reflection megasweep with background 
# sample_name = '$YZD154$~$REF$~$10Kp51+1+1$~$720nmC600g$~$2s$~$REF_forBackground$'
# flag = True
# exps = spectral_experiments.SpectralExperimentsCollections()
# exps.add_dual_gate_spectra_sweep(sample_name, '$start$', iv, lf6, vbg_start=-5, vbg_stop=5,
#                                     vtg_start=-5, vtg_stop=5, frames=201, repeat=1, plot=False)
# exps.execute(repeat=1)
# with open('{}.csv'.format(f_name), 'a') as f:
#             wls = lf6.get_wavelength_calibration()
#             cols = np.concatenate((np.array(['Vbg','Vtg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
#             np.savetxt(f, cols, fmt='%s', delimiter=',')
#             for vbg in vbgs:
#                 iv.x_goto('Vbg', vbg, 0.03, 0.1)
#                 if flag == True:
#                     vtgseq = vtgs
#                 else:
#                     vtgseq = np.flip(vtgs)
#                 for vtg in vtgseq:
#                     iv.x_goto('Vtg', vtg, 0.03, 0.1)
#                     spectra = lf6.acquire()
#                     data = np.concatenate((np.array([vbg, vtg], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
#                     np.savetxt(f, data, fmt='%.5e', delimiter=',') 
#                 flag = not flag
#             iv.x_goto('Vbg', 0, 0.03, 0.1)
#             iv.x_goto('Vtg', 0, 0.03, 0.1)
#             print('vg:', 0)
# exps = spectral_experiments.SpectralExperimentsCollections()
# exps.add_dual_gate_spectra_sweep(sample_name, '$end$', iv, lf6, vbg_start=-5, vbg_stop=5,
#                                     vtg_start=-5, vtg_stop=5, frames=201, repeat=1, plot=False)
# exps.execute(repeat=1)

In [ ]:
import numpy as np
f_name = '$YZD73$~$PL730nm$~$7Kpb$~$355uW$~$1s$~$VPMegasweep$'
# f_name = 'test20231001'
vbgs = np.linspace(-5, 5, 101)
vtgs = np.linspace(-5, 5, 101)

In [ ]:
# VP megasweep 
# KK KKp
halfangles = [46,91]
flag = True
with open('{}.csv'.format(f_name), 'a') as f:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['Vbg','Vtg','halfangle'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f, cols, fmt='%s', delimiter=',')
            for vbg in vbgs:
                iv.x_goto('Vbg', vbg, 0.03, 0.1)
                if flag == True:
                    vtgseq = vtgs
                else:
                    vtgseq = np.flip(vtgs)
                for vtg in vtgseq:
                    iv.x_goto('Vtg', vtg, 0.03, 0.1)
                    for halfangle in halfangles:
                        ep300.set_position(1,halfangle)
                        spectra = lf6.acquire()
                        data = np.concatenate((np.array([vbg, vtg, halfangle], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                        np.savetxt(f, data, fmt='%.5e', delimiter=',') 
                flag = not flag
            iv.x_goto('Vbg', 0, 0.03, 0.1)
            iv.x_goto('Vtg', 0, 0.03, 0.1)
            print('vg:', 0)

### Check gate


In [ ]:
iv.x_goto('Vbg', 0, 0.05, 0.05)
iv.x_goto('Vtg', 0, 0.05, 0.05)
iv.report_status()


### New file writing

In [ ]:
#  f_name = input('Input a filename:')
#         with open('{}.csv'.format(f_name), 'a') as f:
#             wls = lf6.get_wavelength_calibration()
#             cols = np.concatenate((np.array(['X', 'Y'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
#             np.savetxt(f, cols, fmt='%s', delimiter=',')

#             for y in np.arange(y_min, y_max, y_step):
#                 for x in np.arange(x_min, x_max, x_step):
#                     daq.set_voltages([x, y])
#                     spectra = lf6.acquire()
#                     data = np.concatenate((np.array([x, y], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
#                     np.savetxt(f, data, fmt='%.5e', delimiter=',')

In [ ]:
# wls = lf6.get_wavelength_calibration()
# print(wls)
lf6.change_center_wavelength(880)

In [ ]:
# vbgs = [1,1.5,2]
# vtgs = [0.5,1,1.5]
# # for x,y in zip(vbgs,vtgs):
# #     print(x,y)
# iv.x_goto('Vbg', 0.25, 0.05, 0.1)
# ep300.set_position(2,51)
# iv.x_goto('Vbg', 0, 0.03, 0.1)
# spectra = lf6.acquire()
# print(spectra)

In [ ]:
import numpy as np
angles = np.linspace(7, 51, 45)
halfangles = [48,93]
print(angles)
# f_name = '$YZ95_VP$~$PL$~$PH$~$730nm90uw$~$1s$~$266degree$'
f_name = '$YZ95_VP$~$PL$~$PH$~$730nm$~$2s$~'
vgs = np.linspace(-4.5,5,191)
# vgs = np.linspace(-0.2,0.2,9)
print(vgs)

In [ ]:
def gate_sweep_vp(filename, vbgs, vbge, vtgs, vtge, step, halfangles):
    vbg = np.linspace(vbgs,vbge,step)
    vtg = np.linspace(vtgs,vtgs,step)
    with open('{}.csv'.format(filename), 'a') as f:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['Vbg','Vtg','halfangles'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f, cols, fmt='%s', delimiter=',')
            for step_vbg,step_vtg in zip(vbg,vtg):
                iv.x_goto('Vbg', step_vbg, 0.03, 0.1)
                iv.x_goto('Vtg', step_vtg, 0.03, 0.1)
                for halfangle in halfangles:
                    ep300.set_position(1,halfangle)
                    spectra = lf6.acquire()
                    data = np.concatenate((np.array([step_vbg, step_vtg, halfangle], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                    np.savetxt(f, data, fmt='%.5e', delimiter=',')               
            iv.x_goto('Vbg', 0, 0.03, 0.1)
            iv.x_goto('Vtg', 0, 0.03, 0.1)
    

In [ ]:
# with open('{}.csv'.format(f_name), 'a') as f:
#             wls = lf6.get_wavelength_calibration()
#             cols = np.concatenate((np.array(['Vbg','Vtg','halfangle','powerangle'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
#             np.savetxt(f, cols, fmt='%s', delimiter=',')
#             for vg in vgs:
#                 iv.x_goto('Vbg', vg, 0.03, 0.1)
#                 iv.x_goto('Vtg', vg, 0.03, 0.1)
#                 print(vg)
#                 for angle in angles:
#                         ep300.set_position(2,angle)
# #                         print('powerangle:',angle)
#                         for halfangle in halfangles:
#                             ep300.set_position(1,halfangle)
# #                             print('halfangle:', halfangle)
#                             spectra = lf6.acquire()
#                             data = np.concatenate((np.array([vg, vg,halfangle,angle], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
#                             np.savetxt(f, data, fmt='%.5e', delimiter=',')
                
#             iv.x_goto('Vbg', 0, 0.03, 0.1)
#             iv.x_goto('Vtg', 0, 0.03, 0.1)
#             print('vg:', 0)
#             ep300.set_position(2,7)
#             print('power', 0)

In [ ]:
# unit ms
lf6.change_expose_time('500')

In [ ]:
# unit nm
a = '870'
lf6.change_spectra_center(a)

In [ ]:
lf6.change_roi_FullSensor()
lf6.change_roi_LineSensor()

In [ ]:
lf6.change_roi_LineSensor()

In [ ]:
# A = lf6.acquire()
# print(lf6.get_wavelength_calibration())
# B = lf6.acquire()
# print(lf6.get_wavelength_calibration())
# C = lf6.acquire()
# print(lf6.get_wavelength_calibration())
# D = lf6.acquire()
# print(lf6.get_wavelength_calibration())

In [ ]:
print(A)
print(np.shape(A)[0])

In [ ]:
vbg = 0
vtg = 0
iv.x_goto('Vbg', vbg, 0.03, 0.1)
iv.x_goto('Vtg', vtg, 0.03, 0.1)
iv.report_status()

In [ ]:
iv.x_goto('Vbg',1.6, 0.03, 0.1)
iv.x_goto('Vtg', 1.6, 0.03, 0.1)
iv.report_status()

In [ ]:
x = np.array([1,2,3,4,5])
for i in [1,2,3,4,5]:
    y = np.array([1,2,3,4,5])*i
    plt.cla()
    plt.plot(x,y)
    
    plt.pause(2)
    

### SHG

In [ ]:
ep300.set_position(1,0)

In [ ]:
import numpy as np
f_name = '$DX156_20240117$~$SHG$~$RTSpot7$~$900nm65mW$~$2s$'
angles = np.linspace(0,360,361)
with open('{}.csv'.format(f_name), 'a') as f:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['SHG_angle','Vbg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f, cols, fmt='%s', delimiter=',')
            for angle in angles:
                ep300.set_position(1,angle)                    
                spectra = lf6.acquire()
                data = np.concatenate((np.array([angle,0], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                np.savetxt(f, data, fmt='%.5e', delimiter=',') 

## Print experiment

In [ ]:
import numpy as np

def generate_sweeps(dvs, coupling_ratio,
                    vtg_min, vtg_max, vbg_min, vbg_max,
                    step=0.05, sign=+1, sample_name="sample",
                    min_frames=2, ensure_n_when_tuple=True):
    """
    Generate sweeps for lines:
        + case: dv = c*VTG + VBG
        - case: dv = c*VTG - VBG

    dvs: float | list[float] | (dv_min, dv_max, n)
    Returns list of dicts and prints exps.add_dual_gate_spectra_sweep(...) lines.
    """

    c = float(coupling_ratio)

    def dv_envelope(sign):
        if sign == +1:
            # dv = c*vtg + vbg
            return c*vtg_min + vbg_min, c*vtg_max + vbg_max
        else:
            # dv = c*vtg - vbg
            # min at (vtg_min, vbg_max); max at (vtg_max, vbg_min)
            return c*vtg_min - vbg_max, c*vtg_max - vbg_min

    def vtg_bounds_for_dv(dv, sign):
        if sign == +1:
            # vbg = dv - c*vtg, require vbg_min ≤ ... ≤ vbg_max
            lo = (dv - vbg_max)/c
            hi = (dv - vbg_min)/c
        else:
            # vbg = c*vtg - dv, require vbg_min ≤ ... ≤ vbg_max
            lo = (dv + vbg_min)/c
            hi = (dv + vbg_max)/c
        # intersect with VTG limits
        return max(vtg_min, lo), min(vtg_max, hi)

    def sweep_for_dv(dv):
        vtg_lo, vtg_hi = vtg_bounds_for_dv(dv, sign)
        if vtg_hi <= vtg_lo:
            return None

        # frames (endpoints included; floor to grid)
        span = vtg_hi - vtg_lo
        frs = int(np.floor(span/step + 1e-12)) + 1
        if frs < min_frames:
            return None

        # Use the continuous endpoints (rounded just for printing)
        vtg_start = float(np.around(vtg_lo, 3))
        vtg_stop  = float(np.around(vtg_hi, 3))

        # Corresponding VBG along the line
        if sign == +1:
            # vbg = dv - c*vtg
            vbg_start = dv - c*vtg_start
            vbg_stop  = dv - c*vtg_stop
        else:
            # vbg = c*vtg - dv
            vbg_start = c*vtg_start - dv
            vbg_stop  = c*vtg_stop  - dv

        vbg_start = float(np.around(vbg_start, 3))
        vbg_stop  = float(np.around(vbg_stop, 3))

        label = '+' if sign == +1 else '-'
        exptitle = f'${c}TG{label}BG={np.around(dv, 2)}$'

        print(f'''exps.add_dual_gate_spectra_sweep(sample_name, '{exptitle}', iv, lf6,
                 vbg_start={vbg_start}, vbg_stop={vbg_stop},
                 vtg_start={vtg_start}, vtg_stop={vtg_stop},
                 frames={frs}, repeat=repeat_n, plot=False)''')

        return {
            "dv": float(np.around(dv, 3)),
            "vtg_start": vtg_start,
            "vtg_stop": vtg_stop,
            "vbg_start": vbg_start,
            "vbg_stop": vbg_stop,
            "frames": frs,
            "exptitle": exptitle
        }

    # --- normalize dvs input ---
    is_tuple = isinstance(dvs, tuple) and len(dvs) == 3
    results = []

    if np.isscalar(dvs):
        dv_list = [float(dvs)]

    elif is_tuple:
        raw_lo, raw_hi, n = float(dvs[0]), float(dvs[1]), int(dvs[2])

        # Feasible global envelope
        env_lo, env_hi = dv_envelope(sign)

        # Pad each end to guarantee >= min_frames:
        # Smallest vtg-span that yields (min_frames) points is (min_frames-1)*step.
        # Map that to dv by moving along vtg within bounds; a simple safe pad is:
        pad = c*step*(min_frames-1)

        dv_lo = max(raw_lo, env_lo + pad)
        dv_hi = min(raw_hi, env_hi - pad)

        if ensure_n_when_tuple:
            # First try: exactly n in padded window
            if dv_hi > dv_lo:
                candidates = np.linspace(dv_lo, dv_hi, n)
                for dv in candidates:
                    r = sweep_for_dv(float(dv))
                    if r: results.append(r)

            # If not enough, oversample original raw window and filter
            if len(results) < n:
                K = max(5*n, 50)
                overs = np.linspace(min(raw_lo, raw_hi), max(raw_lo, raw_hi), K)
                valid = []
                for dv in overs:
                    r = sweep_for_dv(float(dv))
                    if r: valid.append(r)
                if len(valid) >= n:
                    # pick n roughly evenly
                    idxs = np.round(np.linspace(0, len(valid)-1, n)).astype(int)
                    results = [valid[i] for i in idxs]
                else:
                    results = valid

            if len(results) < n:
                print(f"# Note: requested {n} sweeps, generated {len(results)} due to bounds/min_frames.")

            return results

        else:
            # Simple path: may return < n
            dv_list = np.linspace(dv_lo, dv_hi, n) if dv_hi > dv_lo else []

    else:
        dv_list = list(dvs)

    # Single/list path
    for dv in dv_list:
        r = sweep_for_dv(float(dv))
        if r: results.append(r)
    return results


In [ ]:
res1 = generate_sweeps((-35,35,15), coupling_ratio = 1.05, 
                       vtg_min = -20, vtg_max = 20, 
                       vbg_min = -21, vbg_max = 21,
                       step = 0.1, sign = -1, sample_name="$YZ191$",
                       min_frames = 2, ensure_n_when_tuple=True)

## Efield/Doping scan 

In [ ]:
tg_voltages = np.linspace(-4.5,1,111)
bg_voltages = np.linspace(1,-4.5,111)
# print(tg_voltages)
# print(bg_voltages)
# print(tg_voltages+ bg_voltages)
# print(tg_voltages- bg_voltages)
# tg_voltages = np.linspace(-0.1,0.1,3)
# bg_voltages = np.linspace(0.1,-0.1,3)
# print(tg_voltages+ bg_voltages)
# print(tg_voltages- bg_voltages)

In [ ]:
# Assume TG/BG has same efficency 
# Doping = -3.5
# tg_voltages = np.linspace()
# bg_voltages = np.linspace()
f_name = '$DX40$~$REF720nm$~$4Kp2a$~$1s$~$0T$~$Doping-3.5V$~$qt62deg$'
sample_name = f_name
#  BG if needed
exps = spectral_experiments.SpectralExperimentsCollections()
exps.add_dual_gate_spectra_sweep(sample_name, '$Highdoping_BG$', iv, lf6, vbg_start=-4.5, vbg_stop=-4.5,
                                         vtg_start=-4.45, vtg_stop=-4.45, frames=2, repeat=1, plot=False)
exps.execute(repeat=1)
#  
# with open('{}.csv'.format(f_name), 'a') as f:
#             wls = lf6.get_wavelength_calibration()
#             cols = np.concatenate((np.array(['Vbg','Vtg','Vtg+Vbg','Vtg-Vbg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
#             np.savetxt(f, cols, fmt='%s', delimiter=',')
#             for vtg,vbg in zip(tg_voltages,bg_voltages):
#                 Doping = vtg+vbg
#                 Efield = vtg-vbg
#                 iv.x_goto('Vbg', vbg, 0.03, 0.1)
#                 iv.x_goto('Vtg', vtg, 0.03, 0.1)                 
#                 spectra = lf6.acquire()
#                 data = np.concatenate((np.array([vbg,vtg,Doping,Efield], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
#                 np.savetxt(f, data, fmt='%.5e', delimiter=',') 
#             iv.x_goto('Vbg', 0, 0.03, 0.1)
#             iv.x_goto('Vtg', 0, 0.03, 0.1) 



In [ ]:
tg_voltages = np.linspace(0,0,201)
bg_voltages = np.linspace(0,5,201)
print(tg_voltages)
print(bg_voltages)
# print(tg_voltages+ bg_voltages)
# print(tg_voltages- bg_voltages)

In [ ]:
# Assume TG/BG has same efficency 
# Doping = -4.5
# tg_voltages = np.linspace()
# bg_voltages = np.linspace()
f_name = '$YZ181$~$PL730nm$~$4Kp1near3sdelay$~$1uW1s$~$0T$~$BGonly0to-5V$'
sample_name = f_name

with open('{}.csv'.format(f_name), 'a') as f:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['Vbg','Vtg','Vtg+Vbg','Vtg-Vbg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f, cols, fmt='%s', delimiter=',')
            for vtg,vbg in zip(tg_voltages,bg_voltages):
                Doping = vtg+vbg
                Efield = vtg-vbg
                iv.x_goto('Vbg', vbg, 0.03, 0.1)
                iv.x_goto('Vtg', vtg, 0.03, 0.1)       
                time.sleep(3)
                spectra = lf6.acquire()
                data = np.concatenate((np.array([vbg,vtg,Doping,Efield], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                np.savetxt(f, data, fmt='%.5e', delimiter=',') 
            iv.x_goto('Vbg', 0, 0.03, 0.1)
            iv.x_goto('Vtg', 0, 0.03, 0.1) 

In [ ]:
tg_voltages = np.linspace(8.4 ,-6,289)
bg_voltages = np.linspace(-8.4,6,289)
print(tg_voltages,bg_voltages)

In [ ]:
tg_voltages = np.linspace(8.4 ,-6,289)
bg_voltages = np.linspace(-8.4,6,289)
f1_name = '$YZD199$~$PL633nm$~$4Kp1r$~$5s$~$15uW0T$~$TG+BG=0$~$out41deg$'
f2_name = '$YZD199$~$PL633nm$~$4Kp1r$~$5s$~$15uW0T$~$TG+BG=0$~$out86deg$'
valleys = [41,86]
with open('{}.csv'.format(f1_name), 'a') as f1, open('{}.csv'.format(f2_name),'a') as f2:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['Vbg','Vtg','Vtg+Vbg','Angle'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f1, cols, fmt='%s', delimiter=',')
            np.savetxt(f2, cols, fmt='%s', delimiter=',')
            for vtg,vbg in zip(tg_voltages,bg_voltages):
                Doping = vtg+vbg
                Efield = vtg-vbg
                iv.x_goto('Vbg', vbg, 0.03, 0.1)
                iv.x_goto('Vtg', vtg, 0.03, 0.1) 
                for index, valley in enumerate(valleys):
                    ep300.set_position(2,valley)
                    if index == 0:   
                        spectra = lf6.acquire()
                        data = np.concatenate((np.array([vbg,vtg,Doping,valley], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                        np.savetxt(f1, data, fmt='%.5e', delimiter=',') 
                    else:
                        spectra = lf6.acquire()
                        data = np.concatenate((np.array([vbg,vtg,Doping,valley], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                        np.savetxt(f2, data, fmt='%.5e', delimiter=',') 
            iv.x_goto('Vbg', 0, 0.03, 0.1)
            iv.x_goto('Vtg', 0, 0.03, 0.1) 

time.sleep(20)

tg_voltages = np.linspace(-6,6,241)
bg_voltages = np.linspace(-6,6,241)
f1_name = '$YZD199$~$PL633nm$~$4Kp1r$~$5s$~$15uW0T$~$TG-BG=0$~$out41deg$'
f2_name = '$YZD199$~$PL633nm$~$4Kp1r$~$5s$~$15uW0T$~$TG-BG=0$~$out86deg$'
valleys = [41,86]
with open('{}.csv'.format(f1_name), 'a') as f1, open('{}.csv'.format(f2_name),'a') as f2:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['Vbg','Vtg','Vtg+Vbg','Angle'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f1, cols, fmt='%s', delimiter=',')
            np.savetxt(f2, cols, fmt='%s', delimiter=',')
            for vtg,vbg in zip(tg_voltages,bg_voltages):
                Doping = vtg+vbg
                Efield = vtg-vbg
                iv.x_goto('Vbg', vbg, 0.03, 0.1)
                iv.x_goto('Vtg', vtg, 0.03, 0.1) 
                for index, valley in enumerate(valleys):
                    ep300.set_position(2,valley)
                    if index == 0:   
                        spectra = lf6.acquire()
                        data = np.concatenate((np.array([vbg,vtg,Doping,valley], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                        np.savetxt(f1, data, fmt='%.5e', delimiter=',') 
                    else:
                        spectra = lf6.acquire()
                        data = np.concatenate((np.array([vbg,vtg,Doping,valley], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                        np.savetxt(f2, data, fmt='%.5e', delimiter=',') 
            iv.x_goto('Vbg', 0, 0.03, 0.1)
            iv.x_goto('Vtg', 0, 0.03, 0.1) 

In [ ]:
tg_voltages = np.linspace(-6,6,241)
bg_voltages = np.linspace(-6,6,241)
f1_name = '$YZD199$~$PL633nm$~$4Kp1r$~$1s$~$15uW0T$~$TG-BG=0$~$out41deg$'
f2_name = '$YZD199$~$PL633nm$~$4Kp1r$~$1s$~$15uW0T$~$TG-BG=0$~$out86deg$'
valleys = [41,86]
with open('{}.csv'.format(f1_name), 'a') as f1, open('{}.csv'.format(f2_name),'a') as f2:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['Vbg','Vtg','Vtg+Vbg','Vtg-Vbg'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f1, cols, fmt='%s', delimiter=',')
            np.savetxt(f2, cols, fmt='%s', delimiter=',')
            for vtg,vbg in zip(tg_voltages,bg_voltages):
                Doping = vtg+vbg
                Efield = vtg-vbg
                iv.x_goto('Vbg', vbg, 0.03, 0.1)
                iv.x_goto('Vtg', vtg, 0.03, 0.1) 
                for index, valley in enumerate(valleys):
                    ep300.set_position(2,valley)
                    if index == 0:   
                        spectra = lf6.acquire()
                        data = np.concatenate((np.array([vbg,vtg,Doping,Efield], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                        np.savetxt(f1, data, fmt='%.5e', delimiter=',') 
                        print(valley,index)
                    else:
                        spectra = lf6.acquire()
                        data = np.concatenate((np.array([vbg,vtg,Doping,Efield], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                        np.savetxt(f2, data, fmt='%.5e', delimiter=',') 
                        print(valley,index)
            iv.x_goto('Vbg', 0, 0.03, 0.1)
            iv.x_goto('Vtg', 0, 0.03, 0.1) 

## Spatial scan

In [ ]:
import instrControl as ic, pyvisa, numpy as np, matplotlib, time
%matplotlib qt

daq = ic.DaqControl('Dev1')
daq.add_ao_channel('ao0', 'x_axis')
daq.add_ao_channel('ao1', 'y_axis')
daq.check_status()

instrument_list = [daq]

sample_name = 'dev309_4K'
ex = ic.Experiment(instrument_list, sample_name)



In [ ]:
daq.close()


In [ ]:
ex.x_goto('x_axis', 0, 0.1, 0.1)
ex.x_goto('y_axis', 0, 0.1, 0.1)
ex.get_status()

In [ ]:
ex.x_goto('x_axis', 5, 0.1, 0.1)
ex.x_goto('y_axis', 5, 0.1, 0.1)


In [ ]:
x_axis_start = 1
y_axis_start = 1
x_axis_end = 9
y_axis_end = 9
x_step = 81
y_step = 81
x_steps = np.linspace(x_axis_start, x_axis_end, x_step)
y_steps = np.linspace(y_axis_start, y_axis_end, y_step)
scanner_steps = 0.1
print(x_steps,y_steps)

In [ ]:
f_name = '$YZD267 spatial scan$~$PL900nm$~$633@5uW~$2sx1$~$TG=BG=0V$'
lf6.change_spectra_center('900')
x_start, x_stop, x_points = 1, 9, 41   # from 0 → 10 with 21 steps
y_start, y_stop, y_points = 1, 9, 41   # from -5 → 5 with 11 steps

x_steps = np.linspace(x_start, x_stop, x_points)
y_steps = np.linspace(y_start, y_stop, y_points)
scanner_steps=0.05
flag = True
iv.x_goto('Vbg', 0, 0.1, 0.05)
iv.x_goto('Vtg', 0, 0.1, 0.05)
iv.report_status()
with open('{}.csv'.format(f_name), 'a') as f:
            wls = lf6.get_wavelength_calibration()
            cols = np.concatenate((np.array(['x_axis','y_axis'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
            np.savetxt(f, cols, fmt='%s', delimiter=',')
            for x_step in x_steps:
                ex.x_goto('x_axis', x_step, scanner_steps, 0.1)
                if flag == True:
                    y_steps_seq = y_steps
                else:
                    y_steps_seq = np.flip(y_steps)
                for y_step in y_steps_seq:
                    ex.x_goto('y_axis', y_step, scanner_steps, 0.1)
                    spectra = lf6.acquire()
                    data = np.concatenate((np.array([x_step, y_step], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
                    np.savetxt(f, data, fmt='%.5e', delimiter=',') 
                flag = not flag
# iv.x_goto('Vbg', 0, 0.1, 0.05)
# iv.x_goto('Vtg', 0, 0.1, 0.05)
iv.report_status()
# ex.x_goto('x_axis', 5, 0.1, 0.1)
# ex.x_goto('y_axis', 5, 0.1, 0.1)

# f_name = '$YZD144$~$PL633nm$~$0.2uw2s$~$TG=BG=0V$'
# flag = True
# with open('{}.csv'.format(f_name), 'a') as f:
#             wls = lf6.get_wavelength_calibration()
#             cols = np.concatenate((np.array(['x_axis','y_axis'], ndmin=1, dtype='U'), wls.astype('U'))).reshape([1, -1])
#             np.savetxt(f, cols, fmt='%s', delimiter=',')
#             for x_step in x_steps:
#                 ex.x_goto('x_axis', x_step, scanner_steps, 0.1)
#                 if flag == True:
#                     y_steps_seq = y_steps
#                 else:
#                     y_steps_seq = np.flip(y_steps)
#                 for y_step in y_steps_seq:
#                     ex.x_goto('y_axis', y_step, scanner_steps, 0.1)
#                     spectra = lf6.acquire()
#                     data = np.concatenate((np.array([x_step, y_step], ndmin=1, dtype=np.float64), spectra)).reshape([1, -1])
#                     np.savetxt(f, data, fmt='%.5e', delimiter=',') 
#                 flag = not flag
# ex.x_goto('x_axis', 5, 0.1, 0.1)
# ex.x_goto('y_axis', 5, 0.1, 0.1)

In [ ]:
ex.x_goto('x_axis', 0, 0.1, 0.1)
ex.x_goto('y_axis', 0, 0.1, 0.1)

In [ ]:
step = 0.1
x = ex.x_channels['x_axis'].collect_x()
y = ex.x_channels['y_axis'].collect_x()
number_of_w = 0
number_of_a = 0
while True:
    key = input()
    print(key)
    if key == "q":
        break
    elif key == "a":
        if  y+step<10:
            y += step
            ex.x_goto('y_axis', y, 0, 0)
            number_of_a += 1 
    elif key == "d":
        if y-step>0:
            y -= step 
            ex.x_goto('y_axis', y, 0, 0)
            number_of_a -= 1 
    elif key == "w":
        if x+step<10:
            x += step
            ex.x_goto('x_axis', x, 0, 0)
            number_of_w += 1
    elif key == "s":
        if x-step>0:
            x -= step
            ex.x_goto('x_axis', x, 0, 0)
            number_of_w -= 1
print('a:',number_of_a,'w:',number_of_w)

In [ ]:
A = lf6.multi_frame_acquire(True,x_pixels_num=512,y_pixels_num=512)

In [ ]:
Wavelength = lf6.get_wavelength_calibration()

In [ ]:
%matplotlib inline
frame = 5 
plt.imshow(A[frame,:].reshape(512,512))
plt.colorbar()
plt.title(frame)
plt.show()

In [ ]:
%matplotlib inline
for frame in range(20):
    data = A[frame,:].reshape(512,512)
    sum_along_y = np.sum(data,axis=0)
    plt.plot(sum_along_y,label=frame)
    plt.legend(loc="lower left")
    plt.ylim(250000,400000)
plt.show()

In [ ]:
for frame in range(10):
    data1 = A[frame*2,:].reshape(512,512)
    data2 = A[frame*2+1,:].reshape(512,512)
    sum_along_y_data1 = np.sum(data1,axis=0)
    sum_along_y_data2 = np.sum(data2,axis=0)
    diff = sum_along_y_data1-sum_along_y_data2
    x_norm = (diff-np.min(diff))/(np.max(diff)-np.min(diff))
    plt.plot(x_norm,label=frame)
    plt.legend(loc="lower left")
    # plt.ylim(250000,400000)
    plt.ylim(0, 0.1)
    plt.xlim(330,380)
plt.show()

In [ ]:
np.savetxt('EMCCD_20frames_exposeduringtrigger_raw',A)

In [ ]:
B = lf6.multi_frame_acquire(image_mode=False,x_pixels_num=512,y_pixels_num=512)

In [ ]:
%matplotlib inline
for frame in range(20):
    data = B[frame,:]
    plt.plot(data,label=frame)
    plt.legend(loc="lower left")
plt.show()

In [ ]:
for frame in range(10):
    data1 = B[frame*2,:]
    data2 = B[frame*2+1,:]
    plt.plot((data1-data2),label=frame)
    plt.legend(loc="lower left")
    # plt.ylim(250000,400000)
plt.show()

### Transport & Photocurrent

In [ ]:
amp_rate = 5E8
n_sample = 5
b_gates = np.linspace(0, -7, 141)
plot_y = np.empty_like(b_gates)
plot_y[:] = np.nan
f_name = 'YZDev207_40K_BG0to-7V_CG-11V_2Bias1.9V_amp2nA3Hz_withoutLight'
with open('{}.csv'.format(f_name), 'a') as f:
    cols = np.array(['vds','ids'], ndmin=1).reshape([1, -1])
    np.savetxt(f, cols, fmt='%s', delimiter=',')
    for index, b_gate in enumerate(b_gates):
        iv.x_goto('Vbg',b_gate,0.05,0.1)
        time.sleep(1)
        raw = 0
        for j_sample in range(n_sample):
            daq.read_y()
            raw += daq.y_values[2]
        raw = raw/n_sample
        ids = raw / amp_rate
        plot_y[index] = ids           
        plt.cla()
        plt.plot(b_gates,plot_y)
        plt.pause(0.01)
        data = np.array([b_gate,ids ], ndmin=1).reshape([1, -1])
        np.savetxt(f, data, fmt='%.5e', delimiter=',')
         

iv.x_goto('Vbg',0 ,0.05,0.1)
plt.title(f_name)
plt.savefig(f_name+'.png')